In [ ]:
# ===================================================================
# === CELL 1: SETUP, LIBRARIES & CONFIGURATION
# This is the single source of truth for the entire notebook.
# Run this cell once at the beginning of every session.
# ===================================================================

# --- [1] Import Core Libraries ---
import pandas as pd
import sqlite3
import os
import gspread
import warnings
from google.colab import drive, auth
from google.auth import default
from tqdm import tqdm

print("✅ All libraries imported.")

warnings.filterwarnings('ignore', category=FutureWarning)

# --- [2] Mount Google Drive ---
# This gives the Colab environment access to your Drive files.
try:
    drive.mount('/content/drive', force_remount=True)
    print("✅ Google Drive mounted successfully.")
except Exception as e:
    print(f"🛑 ERROR: Failed to mount Google Drive. Reason: {e}")


# --- [3] Define Global Paths (SINGLE SOURCE OF TRUTH) ---
# All paths are defined here. No other cell should define a path.
project_root = '/content/drive/My Drive/_Pienza' # Main project folder

db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
schema_file_path = os.path.join(project_root, 'Opus1_101001/06_architecture/Opus_1_V251105.sql')
staging_area_path = os.path.join(project_root, 'Assets/Database/02_Staging_Data')

# Define source GSheet names
RAW_OCR_SHEET_NAME = "raw_Offers"
GTS_SHEET_NAME = "GTS-4"

# --- [4] Path Validation ---
# This block verifies that our paths are correct before we proceed.
print("\n--- Verifying critical paths ---")
paths_to_check = {
    "Database file": db_file_path,
    "Schema file": schema_file_path,
    "Staging area": staging_area_path
}

# NOTE: We only check the schema and staging paths for existence,
# as the db file will be created by our script.
if not os.path.exists(os.path.dirname(db_file_path)):
    print(f"🛑 CRITICAL ERROR: The directory for the database does not exist. Check your project_root.")
elif not os.path.exists(schema_file_path):
    print(f"🛑 CRITICAL ERROR: The schema file was not found at the specified path.")
else:
    print(f"✅ All paths appear valid. Database will be built at: {db_file_path}")



# --- [5] Google Authentication for GSheets ---
try:
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    print("\n✅ Google authentication successful. Ready to access GSheets.")
except Exception as e:
    print(f"🛑 ERROR: Google authentication failed. Reason: {e}")

print("\n--- SETUP COMPLETE ---")

✅ All libraries imported.
Mounted at /content/drive
✅ Google Drive mounted successfully.

--- Verifying critical paths ---
✅ All paths appear valid. Database will be built at: /content/drive/My Drive/_Pienza/Assets/Database/opus.db

✅ Google authentication successful. Ready to access GSheets.

--- SETUP COMPLETE ---


In [ ]:
# =========================================================
# === CELL 2: PHASE 1 - BUILD & POPULATE DATABASE ===
# =========================================================

print(f"--- PHASE 1 STARTED: Building database '{db_file_path}' ---")

# --- Step 1: Clean slate ---
# Remove the old database file if it exists to ensure a fresh build
if os.path.exists(db_file_path):
    os.remove(db_file_path)
    print(f"Removed existing database file.")

# --- Step 2: Build Schema from .sql file ---
try:
    conn = sqlite3.connect(db_file_path)
    with open(schema_file_path, 'r') as f:
        schema_script = f.read()
    conn.executescript(schema_script)
    conn.commit()
    print("✅ Schema created successfully from .sql file.")
except FileNotFoundError:
    print(f"❌ ERROR: The schema file was not found at '{schema_file_path}'. Halting execution.")
    # Stop the cell execution if the schema file isn't found
    assert False, "Schema file not found. Please check the path in Cell 1."
except Exception as e:
    print(f"❌ An error occurred during schema creation: {e}")
    assert False, "Failed to build schema."
finally:
    if conn:
        conn.close()

# --- Step 3: Populate All Lookup Tables ---
print(f"\n--- Populating all lookup tables... ---")

# --- Data Definitions ---
product_category_data = [(1, 'uberx'), (2, 'comfort'), (3, 'business_comfort'), (4, 'black'), (5, 'uber_planet'), (6, 'uber_pet'), (7, 'envíos_uber')]
offer_action_data = [(1, 'accepted'), (2, 'reject')]
reason_primary_data = [(1, 'dropoff_non_operational'), (2, 'dropoff_proxy'), (3, 'low_profitability'), (4, 'long_pickup_time'), (5, 'dropoff_strategic_mismatch'), (6, 'expected_value_gamble'), (7, 'system_logic_failure')]
heuristic_flag_data = [(1, 'deadhead_risk'), (2, 'long_ride_traffic_risk'), (3, 'obj_end_session'), (4, 'market_anomaly'), (5, 'friday_traffic_risk'), (6, 'dropoff_uncertain'), (7, 'trap'), (8, 'protest_anomaly')]
post_offer_status_data = [(1, 'continue_game'), (2, 'disconnect')]
driver_state_at_request_data = [(1, 'looking_for_rides'), (2, 'trip_in_progress')]
outcome_data = [(1, 'completed'), (2, 'rider_canceled'), (3, 'driver_canceled'), (4, 'unfulfilled'), (5, 'system_failure')]
interpolation_quality_data = [(1, 'unanchored'), (2, 'interpolated'), (3, 'extrapolated_end'), (4, 'extrapolated_start'), (5, 'interpolated_stationary')]
record_status_data = [(1, 'valid'), (2, 'invalid_non_offer')]

# --- Connect and Insert ---
try:
    conn = sqlite3.connect(db_file_path)
    cur = conn.cursor()

    cur.executemany("INSERT INTO product_category (product_category_id, category_name) VALUES (?, ?)", product_category_data)
    cur.executemany("INSERT INTO offer_action (offer_action_id, offer_action_description) VALUES (?, ?)", offer_action_data)
    cur.executemany("INSERT INTO reason_primary (reason_primary_id, reason_primary_description) VALUES (?, ?)", reason_primary_data)
    cur.executemany("INSERT INTO heuristic_flag (heuristic_flag_id, heuristic_flag_description) VALUES (?, ?)", heuristic_flag_data)
    cur.executemany("INSERT INTO post_offer_status (post_offer_status_id, post_offer_status_description) VALUES (?, ?)", post_offer_status_data)
    cur.executemany("INSERT INTO driver_state_at_request (driver_state_at_request_id, driver_state_at_request_description) VALUES (?, ?)", driver_state_at_request_data)
    cur.executemany("INSERT INTO outcome (outcome_id, outcome_description) VALUES (?, ?)", outcome_data)
    cur.executemany("INSERT INTO interpolation_quality (interpolation_quality_id, interpolation_quality_description) VALUES (?, ?)", interpolation_quality_data)
    cur.executemany("INSERT INTO record_status (record_status_id, record_status_description) VALUES (?, ?)", record_status_data)

    conn.commit()
    print("✅ Success: All lookup tables have been populated.")

except sqlite3.Error as e:
    print(f"\n❌ An SQL error occurred during lookup population: {e}")
    conn.rollback()
finally:
    if conn:
        conn.close()

print("\n--- PHASE 1 COMPLETE ---")

--- PHASE 1 STARTED: Building database '/content/drive/My Drive/_Pienza/Assets/Database/opus.db' ---
Removed existing database file.
✅ Schema created successfully from .sql file.

--- Populating all lookup tables... ---
✅ Success: All lookup tables have been populated.

--- PHASE 1 COMPLETE ---


In [ ]:
import sqlite3

# Connect to the database created in the previous step
conn = sqlite3.connect(db_file_path)
cursor = conn.cursor()

# --- Audit 'offers' table ---
print("--- Auditing 'offers' table schema ---")
offers_query = "PRAGMA table_info(offers);"
cursor.execute(offers_query)
offers_schema = cursor.fetchall()
for column in offers_schema:
    print(column)

# --- Audit 'offer_action' table ---
print("\n--- Auditing 'offer_action' table schema ---")
action_query = "PRAGMA table_info(offer_action);"
cursor.execute(action_query)
action_schema = cursor.fetchall()
for column in action_schema:
    print(column)

# Close the connection
conn.close()

--- Auditing 'offers' table schema ---
(0, 'offer_id', 'varchar(64)', 1, None, 1)
(1, 'session_id', 'INTEGER', 1, None, 2)
(2, 'ocr_id', 'INTEGER', 0, None, 0)
(3, 'image_content_hash', 'varchar(64)', 0, None, 0)
(4, 'offer_timestamp', 'datetime', 1, None, 0)
(5, 'upfront_fare', 'decimal(10,2)', 1, None, 0)
(6, 'time_to_pickup_sec', 'INTEGER', 1, None, 0)
(7, 'dist_to_pickup_km', 'decimal(5,2)', 1, None, 0)
(8, 'est_trip_time_sec', 'INTEGER', 1, None, 0)
(9, 'est_trip_dist_km', 'decimal(5,2)', 1, None, 0)
(10, 'pickup_address', 'TEXT', 1, None, 0)
(11, 'dropoff_address', 'TEXT', 1, None, 0)
(12, 'pickup_lat', 'decimal(9,6)', 0, None, 0)
(13, 'pickup_lon', 'decimal(9,6)', 0, None, 0)
(14, 'dropoff_lat', 'decimal(9,6)', 0, None, 0)
(15, 'dropoff_lon', 'decimal(9,6)', 0, None, 0)
(16, 'is_surge', 'boolean', 1, None, 0)
(17, 'surge_amount', 'decimal(10,2)', 0, None, 0)
(18, 'is_turbo_plus', 'boolean', 1, None, 0)
(19, 'turbo_plus_amount', 'decimal(10,2)', 0, None, 0)
(20, 'is_reserve', 'bo

In [ ]:
# ====================================================================================
# === CELL 3 (v3.6 - Correct Architecture): PHASE 2 - STAGE RAW OCR DATA ===
# ====================================================================================
print(f"--- PHASE 2 STARTED: Staging Raw OCR data from GSheet (Mirroring Source) ---")

# This script assumes that CELL 1 has been executed and all global variables
# (gc, RAW_OCR_SHEET_NAME, staging_area_path) are available.
import pandas as pd
import os
import gspread
from google.colab import auth
from google.auth import default

SHEET_TAB_NAME = "raw_requests_messy"
staged_ocr_file = os.path.join(staging_area_path, "staged_raw_ocr.parquet")

try:
    workbook = gc.open(RAW_OCR_SHEET_NAME)
    sheet = workbook.worksheet(SHEET_TAB_NAME)
    print(f"Successfully opened worksheet '{SHEET_TAB_NAME}'.")

    # --- [1] ROBUST EXTRACTION ---
    all_values = sheet.get_all_values()
    data_rows = all_values[1:]

    df = pd.DataFrame(data_rows)
    # Slice the DataFrame to keep only the first 11 columns (A-K)
    df = df.iloc[:, 0:11]

    # Apply our canonical headers to the sliced data
    canonical_headers = [
        'ocr_id', 'image_filename', 'time_taken', 'ride_type', 'upfront_fare',
        'pickup_details', 'pickup_address', 'trip_details', 'dropoff_address',
        'rider_rating', 'special_note'
    ]
    df.columns = canonical_headers
    print("✅ Data extracted and sliced to columns A-K using canonical headers.")

    # --- [2] DATA CLEANSING (Minimal) ---
    df.dropna(subset=['ocr_id', 'image_filename'], how='any', inplace=True)
    df = df[(df['ocr_id'] != '') & (df['image_filename'] != '')].copy()
    print(f"Extracted and cleaned {len(df)} valid rows from the sheet.")

    # --- [3] NO TRANSFORMATION ---
    df_staged = df
    print("✅ Skipping transformation. Staging raw text data to preserve source integrity.")

    # --- [4] LOAD to Staging File ---
    df_staged.to_parquet(staged_ocr_file, index=False)
    if os.path.exists(staged_ocr_file):
        print(f"✅ SUCCESS! Staged {len(df_staged)} raw records to: {staged_ocr_file}")
    else:
        print(f"❌ FAILED: Staging file was not created.")

except Exception as e:
    print(f"❌ An error occurred during OCR data staging: {e}")

print("\n--- PHASE 2 COMPLETE ---")

--- PHASE 2 STARTED: Staging Raw OCR data from GSheet (Mirroring Source) ---
Successfully opened worksheet 'raw_requests_messy'.
✅ Data extracted and sliced to columns A-K using canonical headers.
Extracted and cleaned 4765 valid rows from the sheet.
✅ Skipping transformation. Staging raw text data to preserve source integrity.
✅ SUCCESS! Staged 4765 raw records to: /content/drive/My Drive/_Pienza/Assets/Database/02_Staging_Data/staged_raw_ocr.parquet

--- PHASE 2 COMPLETE ---


In [ ]:
# =================================================================================
# === CELL 4 (v2.4 - with Nullification): LOAD STAGED OCR DATA TO DB ===
# =================================================================================
print("--- PHASE 3 STARTED: Loading staged OCR data into 'raw_offers_ocr' table ---")

import pandas as pd
import sqlite3
import os
import numpy as np # Import numpy for np.nan

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    staging_area_path = os.path.join(project_root, 'Assets/Database/02_Staging_Data')
    print("INFO: 'db_file_path' not found in global scope. Using default path.")

source_parquet_file = os.path.join(staging_area_path, "staged_raw_ocr.parquet")
table_name = 'raw_offers_ocr'

try:
    # --- [1] EXTRACT from staged file ---
    df_staged = pd.read_parquet(source_parquet_file)
    print(f"Extracted {len(df_staged)} raw records from staged Parquet file.")

    # --- [2] TRANSFORM (NEW Nullification Step) ---
    print("Transforming string 'NULL's into true NULL values...")
    # We replace the string 'NULL' with a representation pandas understands as null (np.nan)
    # pandas' to_sql will then convert this to a database NULL.
    df_to_load = df_staged.replace('NULL', np.nan)
    print("✅ Nullification complete.")

    # --- [3] LOAD ---
    with sqlite3.connect(db_file_path) as conn:
        df_to_load.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"✅ Success: Loaded {len(df_to_load)} cleaned records into '{table_name}'.")

    # --- [4] VALIDATE ---
    with sqlite3.connect(db_file_path) as conn:
        count_df = pd.read_sql_query(f"SELECT COUNT(*) as count FROM {table_name}", conn)
        total_rows = count_df['count'][0]

    if total_rows == len(df_to_load):
        print(f"✅ Validation successful: DB table contains {total_rows} records.")
    else:
        print(f"❌ VALIDATION FAILED: Row count mismatch.")

except Exception as e:
    print(f"❌ An error occurred during the loading process: {e}")

print("\n--- PHASE 3 COMPLETE ---")

--- PHASE 3 STARTED: Loading staged OCR data into 'raw_offers_ocr' table ---
Extracted 4765 raw records from staged Parquet file.
Transforming string 'NULL's into true NULL values...
✅ Nullification complete.
✅ Success: Loaded 4765 cleaned records into 'raw_offers_ocr'.
✅ Validation successful: DB table contains 4765 records.

--- PHASE 3 COMPLETE ---


In [ ]:
# =========================================================================================
# === CELL 5 (v6.0 - M:M Capable): ETL FOR THE POLISHED 'offers' TABLE ===
# === BLOCK 1 of 3: Extraction & 1-to-1 Enrichment ===
# =========================================================================================
print("--- PHASE 4 STARTED: Polishing and Loading data for the 'offers' table ---")

import pandas as pd
import sqlite3
import os
import gspread
from google.colab import auth
from google.auth import default
from tqdm import tqdm
import re

# --- Self-Contained Configuration ---
try: gc
except NameError:
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

DIAMOND_SHEET_NAME = "raw_Offers"
DIAMOND_TAB_NAME = "diamond_offers"
TARGET_OFFER_TABLE = "offers"
TARGET_JOIN_TABLE = "heuristic_flag_offers" # New target table

# ==============================================================================
# --- 1. EXTRACT ---
# ==============================================================================
print("\n--- [E] EXTRACTING all necessary data sources ---")
try:
    workbook = gc.open(DIAMOND_SHEET_NAME)
    sheet = workbook.worksheet(DIAMOND_TAB_NAME)
    df_polished_raw = pd.DataFrame(sheet.get_all_records())
    print(f"✅ Extracted {len(df_polished_raw)} polished rows from GSheet.")

    with sqlite3.connect(db_file_path) as conn:
        df_ocr_link = pd.read_sql_query("SELECT ocr_id FROM raw_offers_ocr", conn)
    print(f"✅ Extracted {len(df_ocr_link)} rows from 'raw_offers_ocr' for linking.")

    with sqlite3.connect(db_file_path) as conn:
        lookup_table_names = [
            "product_category", "offer_action", "reason_primary", "heuristic_flag",
            "post_offer_status", "driver_state_at_request", "outcome",
            "interpolation_quality", "record_status"
        ]
        lookup_dfs = {}
        for table in tqdm(lookup_table_names, desc="Extracting lookups"):
            lookup_dfs[table] = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    print(f"✅ Extracted all {len(lookup_dfs)} lookup tables.")

except Exception as e:
    print(f"❌ ERROR during extraction: {e}")
    assert False, "Extraction failed, halting execution."

# ==============================================================================
# --- 2. TRANSFORM (PART 1) ---
# ==============================================================================
print("\n--- [T] TRANSFORMING and enriching 'offer' data ---")

# --- A. DEFENSIVE DROP & INTELLIGENT KEY JOIN ---
if 'ocr_fk' in df_polished_raw.columns:
    df_polished_raw.drop(columns=['ocr_fk'], inplace=True)
df_polished_raw['join_key'] = pd.to_numeric(df_polished_raw['offer_id'].str.replace('OF', ''), errors='coerce')
df_ocr_link['join_key'] = pd.to_numeric(df_ocr_link['ocr_id'].str.replace('OCR_', ''), errors='coerce')
df_ocr_link.rename(columns={'ocr_id': 'ocr_fk'}, inplace=True)
df_transformed = pd.merge(df_polished_raw, df_ocr_link[['join_key', 'ocr_fk']], on='join_key', how='left')
df_transformed.drop(columns=['join_key'], inplace=True)
print("✅ ocr_fk link established.")

# --- B. FULL DATA TYPE & CONTENT CLEANING ---
df_transformed.replace({'': None, 'N/A': None, 'nan': None, 'NULL': None}, inplace=True)

# --- Numeric Columns ---
numeric_cols = [
    'upfront_fare', 'time_to_pickup_sec', 'dist_to_pickup_km', 'est_trip_time_sec',
    'est_trip_dist_km', 'pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon',
    'surge_amount', 'turbo_plus_amount', 'reservation_amount', 'priority_amount',
    'rider_star_rating', 'rider_trip_count', 'time_in_session_sec', 'session_progress_ratio',
    'inferred_agent_lat', 'inferred_agent_lon', 'inferred_agent_bearing', 'inferred_agent_speed_mps'
]
for col in numeric_cols:
    if col in df_transformed.columns:
        df_transformed[col] = pd.to_numeric(df_transformed[col], errors='coerce')
        # Explicitly cast to float64 to handle NaNs and ensure correct DB type
        df_transformed[col] = df_transformed[col].astype('float64')

# --- Timestamp Column ---
df_transformed['offer_timestamp'] = pd.to_datetime(
    df_transformed['offer_timestamp'], format='%Y:%m:%d %H:%M:%S', errors='coerce'
).dt.strftime('%Y-%m-%d %H:%M:%S')

# --- Boolean Columns (The Definitive "Three-State" Fix) ---
print("Applying three-state boolean conversion (True/False/NULL)...")
boolean_cols = [col for col in df_transformed.columns if col.startswith('is_')]

def three_state_bool_converter(value):
    if isinstance(value, str):
        val_lower = value.lower().strip()
        if val_lower == 'true':
            return True
        elif val_lower == 'false':
            return False
        else: # Handles '', 'N/A', etc.
            return None
    elif isinstance(value, bool):
        return value
    else: # Handles np.nan, etc.
        return None

for col in boolean_cols:
    if col in df_transformed.columns:
        df_transformed[col] = df_transformed[col].apply(three_state_bool_converter)
        # Use pandas' nullable boolean type. This is critical.
        df_transformed[col] = df_transformed[col].astype('boolean')

print("✅ Data types cleaned and standardized with robust three-state boolean logic.")

# --- C. ENRICH WITH ONE-TO-ONE FOREIGN KEYS ---
# NOTE: 'heuristic_flag' is deliberately excluded from this map.
print("Enriching with ONE-TO-ONE foreign key IDs...")
merge_map_1_to_1 = {
    'product_category': ('product_category_fk', 'category_name'),
    'offer_action': ('offer_action_fk', 'offer_action_description'),
    'reason_primary': ('reason_primary_fk', 'reason_primary_description'),
    'post_offer_status': ('post_offer_status_fk', 'post_offer_status_description'),
    'driver_state_at_request': ('driver_state_at_request_fk', 'driver_state_at_request_description'),
    'outcome': ('outcome_fk', 'outcome_description'),
    'interpolation_quality': ('interpolation_quality_fk', 'interpolation_quality_description'),
    'record_status': ('record_status_fk', 'record_status_description')
}
for lookup_table, (source_fk_col, desc_col) in merge_map_1_to_1.items():
    if source_fk_col in df_transformed.columns:
        df_transformed[source_fk_col] = df_transformed[source_fk_col].astype(str).str.strip().str.lower().str.replace(' ', '_')
        if source_fk_col == 'product_category_fk':
            df_transformed[source_fk_col] = df_transformed[source_fk_col].str.replace('artículo', 'envíos_uber', regex=False)
        df_lookup = lookup_dfs[lookup_table]
        id_col = f"{lookup_table}_id"
        df_transformed = pd.merge(df_transformed, df_lookup[[id_col, desc_col]], left_on=source_fk_col, right_on=desc_col, how='left')
        df_transformed.drop(columns=[source_fk_col, desc_col], inplace=True)
        df_transformed.rename(columns={id_col: source_fk_col}, inplace=True)
print("✅ All ONE-TO-ONE foreign key IDs enriched.")


# ==============================================================================
# === BLOCK 2 of 3: The 'Heuristic Flag' Processor (Many-to-Many) ===
# ==============================================================================
print("\n--- Processing MANY-TO-MANY relationship for 'heuristic_flag' ---")

# Isolate the 'heuristic_flag' lookup table for clarity
df_heuristic_lookup = lookup_dfs['heuristic_flag']

# Create a clean DataFrame to build our join table records
# It only needs the offer_id and the raw heuristic_flag string
df_flags_raw = df_transformed[['offer_id', 'heuristic_flag']].dropna(subset=['heuristic_flag'])
print(f"Found {len(df_flags_raw)} offers with heuristic flags to process.")

# --- The "Explode and Map" Logic ---
# 1. Split the comma-separated strings into lists of individual flags
df_flags_raw['flag_list'] = df_flags_raw['heuristic_flag'].str.split(', ')

# 2. Use .explode() to transform each item in the list into its own row
# This creates the long-format data needed for the join table
df_exploded = df_flags_raw.explode('flag_list')
df_exploded['flag_list'] = df_exploded['flag_list'].str.strip()

# 3. Join with the lookup table to get the integer ID for each flag
df_join_table_prep = pd.merge(
    df_exploded,
    df_heuristic_lookup,
    left_on='flag_list',
    right_on='heuristic_flag_description',
    how='inner' # Use inner join to drop any flags that don't have a valid lookup
)

# --- Final Schema Alignment for the Join Table ---
# Select and rename the columns to EXACTLY match the 'heuristic_flag_offers' schema
df_heuristic_flag_offers_to_load = df_join_table_prep[['offer_id', 'heuristic_flag_id']].copy()
df_heuristic_flag_offers_to_load.rename(columns={
    'offer_id': 'offers_offer_id',
    'heuristic_flag_id': 'heuristic_flag_heuristic_flag_id'
}, inplace=True)

print(f"✅ Successfully prepared {len(df_heuristic_flag_offers_to_load)} records for the 'heuristic_flag_offers' join table.")



# ==============================================================================
# === BLOCK 3 of 3: Final Schema Alignment & Loading ===
# ==============================================================================
print("\n--- Finalizing schema for 'offers' table and preparing for load ---")

# --- D. FINAL SCHEMA ALIGNMENT for 'offers' table ---
# This list is now definitive. It includes all columns EXCEPT the raw 'heuristic_flag' text.
final_offer_columns = [
    'offer_id', 'session_fk', 'ocr_fk', 'image_content_hash', 'offer_timestamp', 'upfront_fare',
    'time_to_pickup_sec', 'dist_to_pickup_km', 'est_trip_time_sec', 'est_trip_dist_km',
    'pickup_address', 'dropoff_address', 'pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon',
    'is_surge', 'surge_amount', 'is_turbo_plus', 'turbo_plus_amount', 'is_reservation',
    'reservation_amount', 'is_priority', 'priority_amount', 'is_exclusive', 'is_vip',
    'is_identity_verified', 'is_long_trip', 'is_multiple_destinations', 'is_teens',
    'rider_star_rating', 'rider_trip_count', 'time_in_session_sec', 'session_progress_ratio',
    'inferred_agent_lat', 'inferred_agent_lon', 'inferred_agent_bearing',
    'inferred_agent_speed_mps', 'is_imputed', 'special_note_raw', 'comment_1', 'comment_2',
    'product_category_fk', 'offer_action_fk', 'reason_primary_fk', 'post_offer_status_fk',
    'driver_state_at_request_fk', 'outcome_fk', 'interpolation_quality_fk', 'record_status_fk'
]

# The 'df_transformed' DataFrame still contains the original 'heuristic_flag' text column.
# By using this final_offer_columns list, we are effectively DROPPING it from the final table.
existing_final_columns = [col for col in final_offer_columns if col in df_transformed.columns]
df_offers_to_load = df_transformed[existing_final_columns]
print("✅ Final schema alignment complete for 'offers' table.")

# ==============================================================================
# --- 3. LOAD (Dual Table Operation) ---
# ==============================================================================
print(f"\n--- [L] LOADING data into database '{db_file_path}' ---")
try:
    with sqlite3.connect(db_file_path) as conn:
        # Load the main 'offers' table
        df_offers_to_load.to_sql(TARGET_OFFER_TABLE, conn, if_exists='replace', index=False)
        print(f"✅ Success: Loaded {len(df_offers_to_load)} records into '{TARGET_OFFER_TABLE}'.")

        # Load the new 'heuristic_flag_offers' join table
        df_heuristic_flag_offers_to_load.to_sql(TARGET_JOIN_TABLE, conn, if_exists='replace', index=False)
        print(f"✅ Success: Loaded {len(df_heuristic_flag_offers_to_load)} records into '{TARGET_JOIN_TABLE}'.")

except Exception as e:
    print(f"❌ ERROR during load operation: {e}")

print("\n--- PHASE 4 COMPLETE ---")



--- PHASE 4 STARTED: Polishing and Loading data for the 'offers' table ---

--- [E] EXTRACTING all necessary data sources ---
✅ Extracted 4765 polished rows from GSheet.
✅ Extracted 4765 rows from 'raw_offers_ocr' for linking.


Extracting lookups: 100%|██████████| 9/9 [00:00<00:00, 804.74it/s]

✅ Extracted all 9 lookup tables.

--- [T] TRANSFORMING and enriching 'offer' data ---
✅ ocr_fk link established.
Applying three-state boolean conversion (True/False/NULL)...
✅ Data types cleaned and standardized with robust three-state boolean logic.
Enriching with ONE-TO-ONE foreign key IDs...


✅ All ONE-TO-ONE foreign key IDs enriched.

--- Processing MANY-TO-MANY relationship for 'heuristic_flag' ---
Found 831 offers with heuristic flags to process.
✅ Successfully prepared 831 records for the 'heuristic_flag_offers' join table.

--- Finalizing schema for 'offers' table and preparing for load ---
✅ Final schema alignment complete for 'offers' table.

--- [L] LOADING data into database '/content/drive/My Drive/_Pienza/Assets/Database/opus.db' ---
✅ Success: Loaded 4765 records into 'offers'.
✅ Success: Loaded 831 records into 'heuristic_flag_offers'.

--- PHASE 4 COMPLETE ---


In [ ]:
# =================================================================================
# === CELL 5b (NEW): The "Three-State Boolean" Post-Processing Patch ===
# =================================================================================
print("--- Starting the 'Three-State Boolean' Post-Processing Patch ---")

import pandas as pd
import sqlite3
import os
import gspread
from google.colab import auth
from google.auth import default

# --- Self-Contained Configuration ---
try: gc
except NameError: auth.authenticate_user(); creds, _ = default(); gc = gspread.authorize(creds)
if 'db_file_path' not in locals(): # and other paths...

  DIAMOND_SHEET_NAME = "raw_Offers"
  DIAMOND_TAB_NAME = "diamond_offers"

try:
    # --- [1] Get the ground truth for NULL booleans from the GSheet ---
    print("Extracting source data to identify 'Unknown' booleans...")
    workbook = gc.open(DIAMOND_SHEET_NAME)
    sheet = workbook.worksheet(DIAMOND_TAB_NAME)
    df_source = pd.DataFrame(sheet.get_all_records())
    df_source.columns = df_source.columns.str.lower().str.replace(' ', '_')

    boolean_cols = [col for col in df_source.columns if col.startswith('is_')]

    # Create a dictionary to hold the lists of offer_ids that need nulling for each column
    ids_to_null = {col: [] for col in boolean_cols}

    for col in boolean_cols:
        # Find where the original value is not 'TRUE' or 'FALSE' (i.e., it's '', 'NULL', etc.)
        mask = ~df_source[col].isin(['TRUE', 'FALSE', True, False])
        ids_to_null[col] = df_source[mask]['offer_id'].tolist()

    print("✅ Successfully identified offers with 'Unknown' boolean states.")

    # --- [2] Execute the Surgical UPDATEs on the Database ---
    print("\nConnecting to opus.db to perform surgical UPDATEs...")
    total_updates = 0
    with sqlite3.connect(db_file_path) as conn:
        cursor = conn.cursor()
        for col, ids in ids_to_null.items():
            if ids: # Only run the query if there are IDs to update
                # Create a string of placeholders for the IN clause, e.g., (?, ?, ?)
                placeholders = ', '.join(['?'] * len(ids))

                # The SQL statement to set the column to NULL for the specific offer_ids
                update_sql = f"UPDATE offers SET {col} = NULL WHERE offer_id IN ({placeholders});"

                cursor.execute(update_sql, ids)
                total_updates += cursor.rowcount
                print(f"  - Set {cursor.rowcount} records to NULL for column '{col}'.")

    print(f"\n✅ Patch complete. A total of {total_updates} fields were updated to NULL.")
    print("The 'offers' table now correctly represents the three-state boolean logic.")

except Exception as e:
    print(f"❌ ERROR during post-processing patch: {e}")

--- Starting the 'Three-State Boolean' Post-Processing Patch ---
Extracting source data to identify 'Unknown' booleans...
✅ Successfully identified offers with 'Unknown' boolean states.

Connecting to opus.db to perform surgical UPDATEs...

✅ Patch complete. A total of 0 fields were updated to NULL.
The 'offers' table now correctly represents the three-state boolean logic.


In [ ]:
# ==============================================================================
# === DIAGNOSTIC: Audit for Boolean Flags (Count of TRUEs) ===
# ==============================================================================
print("--- Auditing the count of TRUE values for all boolean flags in the 'offers' table ---")

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("INFO: 'db_file_path' not found in global scope. Using default path.")

# This query uses UNION ALL to build a clean, multi-row report.
# In SQLite, booleans are stored as 1 (True) and 0 (False), so SUM() is a fast way to count TRUEs.
query_sql = """
SELECT 'is_surge' AS flag_name, SUM(is_surge) AS count_of_trues FROM offers
UNION ALL
SELECT 'is_turbo_plus', SUM(is_turbo_plus) FROM offers
UNION ALL
SELECT 'is_reservation', SUM(is_reservation) FROM offers
UNION ALL
SELECT 'is_priority', SUM(is_priority) FROM offers
UNION ALL
SELECT 'is_exclusive', SUM(is_exclusive) FROM offers
UNION ALL
SELECT 'is_vip', SUM(is_vip) FROM offers
UNION ALL
SELECT 'is_identity_verified', SUM(is_identity_verified) FROM offers
UNION ALL
SELECT 'is_long_trip', SUM(is_long_trip) FROM offers
UNION ALL
SELECT 'is_multiple_destinations', SUM(is_multiple_destinations) FROM offers
UNION ALL
SELECT 'is_teens', SUM(is_teens) FROM offers
UNION ALL
SELECT 'is_imputed', SUM(is_imputed) FROM offers;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_audit = pd.read_sql_query(query_sql, conn)

    print("\n✅ Audit query executed successfully.")
    display(df_audit)

except Exception as e:
    print(f"❌ ERROR during audit: {e}")

--- Auditing the count of TRUE values for all boolean flags in the 'offers' table ---

✅ Audit query executed successfully.


,flag_name,count_of_trues
0,is_surge,1274
1,is_turbo_plus,1552
2,is_reservation,177
3,is_priority,293
4,is_exclusive,2642
5,is_vip,1485
6,is_identity_verified,1983
7,is_long_trip,304
8,is_multiple_destinations,62
9,is_teens,18


In [ ]:
# ==============================================================================
# === DIAGNOSTIC: Finding the "Dirty" Heuristic Flags ===
# ==============================================================================
print("--- Hunting for the two missing heuristic flag records ---")

import pandas as pd

# We can re-use the DataFrames that are already in memory from CELL 5

# Get the set of all unique, VALID flag descriptions from our lookup table
valid_flags = set(lookup_dfs['heuristic_flag']['heuristic_flag_description'])

# Get the raw data from the GSheet that has flags
df_flags_raw = df_transformed[['offer_id', 'heuristic_flag']].dropna(subset=['heuristic_flag'])
# Explode it just like we did in the ETL
df_flags_raw['flag_list'] = df_flags_raw['heuristic_flag'].str.split(', ')
df_exploded = df_flags_raw.explode('flag_list')
df_exploded['flag_list'] = df_exploded['flag_list'].str.strip()

# THE ANTI-JOIN: Find all rows in our exploded data where the flag is NOT in the set of valid flags
df_anomalies = df_exploded[~df_exploded['flag_list'].isin(valid_flags)]

print(f"\n✅ Anomaly hunt complete. Found {len(df_anomalies)} invalid flag entries.")
if not df_anomalies.empty:
    print("These are the specific offers and flag strings that failed to match:")
    display(df_anomalies[['offer_id', 'heuristic_flag', 'flag_list']])

--- Hunting for the two missing heuristic flag records ---

✅ Anomaly hunt complete. Found 0 invalid flag entries.


In [ ]:
# =================================================================================
# === CELL 6 (Self-Contained): CREATE THE MASTER ANALYTICAL VIEW (v_reconciled_offer) ===
# =================================================================================
print("--- Creating the master analytical view: v_reconciled_offer ---")

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("INFO: 'db_file_path' not found in global scope. Using default path.")


# This SQL query joins every single lookup table to the main 'offers' table.
view_sql = """
CREATE VIEW v_reconciled_offer AS
SELECT
    o.*, -- Select all columns from the main offers table

    -- Alias each description column to a clean, unique name
    pc.category_name,
    oa.offer_action_description AS offer_action,
    rp.reason_primary_description AS reason_primary,
    pos.post_offer_status_description AS post_offer_status,
    ds.driver_state_at_request_description AS driver_state_at_request,
    ot.outcome_description AS outcome,
    iq.interpolation_quality_description AS interpolation_quality,
    rs.record_status_description AS record_status
FROM
    offers o
-- Each JOIN now uses the TRUE foreign key from 'offers' and the TRUE primary key from the lookup table
LEFT JOIN product_category pc        ON o.product_category_fk = pc.product_category_id
LEFT JOIN offer_action oa             ON o.offer_action_fk = oa.offer_action_id
LEFT JOIN reason_primary rp           ON o.reason_primary_fk = rp.reason_primary_id
LEFT JOIN post_offer_status pos       ON o.post_offer_status_fk = pos.post_offer_status_id
LEFT JOIN driver_state_at_request ds  ON o.driver_state_at_request_fk = ds.driver_state_at_request_id
LEFT JOIN outcome ot                  ON o.outcome_fk = ot.outcome_id
LEFT JOIN interpolation_quality iq    ON o.interpolation_quality_fk = iq.interpolation_quality_id
LEFT JOIN record_status rs            ON o.record_status_fk = rs.record_status_id;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_reconciled_offer;")
        conn.execute(view_sql)
    print("✅ Success: 'v_reconciled_offer' VIEW created successfully.")

    # --- Verification Step ---
    print("\n--- Final Verification: Querying the VIEW ---")
    with sqlite3.connect(db_file_path) as conn:
        df_view = pd.read_sql_query("""
            SELECT
                offer_id,
                offer_action,
                reason_primary,
                record_status
            FROM
                v_reconciled_offer
            LIMIT 10;
        """, conn)
        print("Successfully queried the view. Displaying results:")
        display(df_view)

except Exception as e:
    print(f"❌ ERROR: {e}")

--- Creating the master analytical view: v_reconciled_offer ---
✅ Success: 'v_reconciled_offer' VIEW created successfully.

--- Final Verification: Querying the VIEW ---
Successfully queried the view. Displaying results:


,offer_id,offer_action,reason_primary,record_status
0,OF00001,reject,dropoff_non_operational,valid
1,OF00002,reject,dropoff_non_operational,valid
2,OF00003,accepted,None,invalid_non_offer
3,OF00004,reject,low_profitability,valid
4,OF00005,reject,dropoff_non_operational,valid
5,OF00006,reject,dropoff_non_operational,valid
6,OF00007,reject,low_profitability,valid
7,OF00008,reject,dropoff_non_operational,valid
8,OF00009,reject,dropoff_non_operational,valid
9,OF00010,reject,low_profitability,valid


In [ ]:
# =================================================================================
# === CELL 6 (Self-Contained): CREATE THE MASTER ANALYTICAL VIEW (v_reconciled_offer) ===
# =================================================================================
print("--- Creating the master analytical view: v_reconciled_offer ---")

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("INFO: 'db_file_path' not found in global scope. Using default path.")

# This SQL query joins every single lookup table to the main 'offers' table.
view_sql = """
CREATE VIEW v_reconciled_offer AS
SELECT
    o.*,
    pc.category_name,
    oa.offer_action_description AS offer_action,
    rp.reason_primary_description AS reason_primary,
    pos.post_offer_status_description AS post_offer_status,
    ds.driver_state_at_request_description AS driver_state_at_request,
    ot.outcome_description AS outcome,
    iq.interpolation_quality_description AS interpolation_quality,
    rs.record_status_description AS record_status
FROM
    offers o
LEFT JOIN product_category pc        ON o.product_category_fk = pc.product_category_id
LEFT JOIN offer_action oa             ON o.offer_action_fk = oa.offer_action_id
LEFT JOIN reason_primary rp           ON o.reason_primary_fk = rp.reason_primary_id
LEFT JOIN post_offer_status pos       ON o.post_offer_status_fk = pos.post_offer_status_id
LEFT JOIN driver_state_at_request ds  ON o.driver_state_at_request_fk = ds.driver_state_at_request_id
LEFT JOIN outcome ot                  ON o.outcome_fk = ot.outcome_id
LEFT JOIN interpolation_quality iq    ON o.interpolation_quality_fk = iq.interpolation_quality_id
LEFT JOIN record_status rs            ON o.record_status_fk = rs.record_status_id;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_reconciled_offer;")
        conn.execute(view_sql)
    print("✅ Success: 'v_reconciled_offer' VIEW created successfully.")

except Exception as e:
    print(f"❌ ERROR creating view: {e}")

--- Creating the master analytical view: v_reconciled_offer ---
✅ Success: 'v_reconciled_offer' VIEW created successfully.


In [ ]:
# ==============================================================================
# === QUERY 1 (DIAGNOSTIC MODE): The 10,000-Foot View ===
# ==============================================================================
import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("INFO: 'db_file_path' not found in global scope. Using default path.")

print("\n--- Querying the master view for 20 sample records (DIAGNOSTIC MODE) ---")

query = """
SELECT
    offer_id,
    offer_timestamp,
    upfront_fare,
    category_name,
    offer_action,
    reason_primary
FROM
    v_reconciled_offer
ORDER BY
    offer_timestamp DESC
LIMIT 20;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_sample = pd.read_sql_query(query, conn)

    print("\n✅ SUCCESS: Diagnostic query executed.")
    print("Displaying raw data from the top 20 most recent offers in the VIEW:")
    display(df_sample)

except Exception as e:
    print(f"❌ ERROR: Query failed. This indicates a deeper problem with the VIEW itself. Reason: {e}")


--- Querying the master view for 20 sample records (DIAGNOSTIC MODE) ---

✅ SUCCESS: Diagnostic query executed.
Displaying raw data from the top 20 most recent offers in the VIEW:


,offer_id,offer_timestamp,upfront_fare,category_name,offer_action,reason_primary
0,OF04765,2025-10-01 09:56:26,130.33,comfort,reject,dropoff_proxy
1,OF04764,2025-10-01 09:54:25,79.66,uber_planet,reject,dropoff_proxy
2,OF04763,2025-10-01 09:51:29,44.23,comfort,reject,dropoff_non_operational
3,OF04762,2025-10-01 09:51:00,76.27,uberx,reject,dropoff_non_operational
4,OF04761,2025-10-01 09:06:38,135.37,comfort,accepted,None
5,OF04760,2025-10-01 08:35:48,165.68,business_comfort,accepted,None
6,OF04759,2025-10-01 08:35:27,233.71,business_comfort,reject,dropoff_non_operational
7,OF04758,2025-10-01 08:35:03,256.12,uberx,reject,dropoff_non_operational
8,OF04757,2025-10-01 08:34:58,149.25,uberx,reject,dropoff_non_operational
9,OF04756,2025-10-01 08:33:16,123.83,uberx,reject,expected_value_gamble


In [ ]:
# ==============================================================================
# === CELL 10 (Self-Contained): DATABASE DIAGNOSTIC - LIST ROWS WITH NULLS ===
# ==============================================================================
print("--- Retrieving list of records where 'category_name' is NULL ---")

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("INFO: 'db_file_path' not found in global scope. Using default path.")


# We select the original foreign key to see the problematic raw data
query_sql = """
SELECT
    offer_id,
    offer_timestamp,
    product_category_fk  -- This is the key column to inspect
FROM
    v_reconciled_offer
WHERE
    category_name IS NULL;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_nulls = pd.read_sql_query(query_sql, conn)

        print(f"\n✅ DIAGNOSTIC COMPLETE: Found {len(df_nulls)} records with a NULL category name.")

        if not df_nulls.empty:
            print("Displaying the raw 'product_category_fk' values from the source data for these records:")
            # Display the full list
            display(df_nulls)

            # Also, let's print the unique problematic values to get a summary
            unique_problem_values = df_nulls['product_category_fk'].unique()
            print("\n--- Summary of Unique Problematic Values ---")
            print("The following values in the 'product_category_fk' column could not be matched:")
            print(unique_problem_values)
        else:
            print("✅ No records found with a NULL category name. The link is 100% complete.")

except Exception as e:
    print(f"❌ ERROR auditing the view: {e}")

--- Retrieving list of records where 'category_name' is NULL ---

✅ DIAGNOSTIC COMPLETE: Found 0 records with a NULL category name.
✅ No records found with a NULL category name. The link is 100% complete.


In [ ]:
# ==============================================================================
# === DIAGNOSTIC: Validating Foreign Key Values for 'product_category' ===
# ==============================================================================
print("--- Validating the distinct values in 'offers.product_category_fk' against the lookup table ---")

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("INFO: 'db_file_path' not found in global scope. Using default path.")

try:
    with sqlite3.connect(db_file_path) as conn:
        # Query 1: What distinct, non-null integer IDs are in the 'offers' table?
        print("\n--- [1] Distinct values found in 'offers.product_category_fk' ---")
        df_keys = pd.read_sql_query("SELECT DISTINCT product_category_fk FROM offers WHERE product_category_fk IS NOT NULL;", conn)
        display(df_keys)

        # Query 2: What are the valid, possible IDs in the 'product_category' table?
        print("\n--- [2] All values in the 'product_category' Lookup Table ---")
        df_lookup = pd.read_sql_query("SELECT * FROM product_category;", conn)
        display(df_lookup)

    print("\n--- [3] VERIFICATION ---")
    # A simple check for validation
    key_set = set(df_keys['product_category_fk'])
    lookup_set = set(df_lookup['product_category_id'])

    if key_set.issubset(lookup_set):
        print("✅ SUCCESS: All foreign key values in 'offers' are valid and exist in the 'product_category' lookup table.")
    else:
        print(f"🛑 WARNING: Found invalid foreign key values: {key_set - lookup_set}")

except Exception as e:
    print(f"❌ ERROR: {e}")

--- Validating the distinct values in 'offers.product_category_fk' against the lookup table ---

--- [1] Distinct values found in 'offers.product_category_fk' ---


,product_category_fk
0,1
1,6
2,2
3,5
4,3
5,7
6,4



--- [2] All values in the 'product_category' Lookup Table ---


,product_category_id,category_name
0,1,uberx
1,2,comfort
2,3,business_comfort
3,4,black
4,5,uber_planet
5,6,uber_pet
6,7,envíos_uber



--- [3] VERIFICATION ---
✅ SUCCESS: All foreign key values in 'offers' are valid and exist in the 'product_category' lookup table.


In [ ]:
# ==============================================================================
# === CELL 11 (Self-Contained): Timestamp Column Audit ('offer_timestamp') ===
# ==============================================================================
print("--- Performing a deep-dive audit of the 'offer_timestamp' column ---")

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("INFO: 'db_file_path' not found in global scope. Using default path.")

try:
    with sqlite3.connect(db_file_path) as conn:
        # --- [1] Schema Audit ---
        print("\n--- [1] Schema Audit ---")
        df_schema = pd.read_sql_query("PRAGMA table_info(offers);", conn)
        timestamp_schema = df_schema[df_schema['name'] == 'offer_timestamp']
        display(timestamp_schema)

        # --- [2] Null Value Audit ---
        print("\n--- [2] Null Value Audit ---")
        null_count_query = "SELECT COUNT(*) AS null_count FROM offers WHERE offer_timestamp IS NULL;"
        df_null_count = pd.read_sql_query(null_count_query, conn)
        null_count = df_null_count['null_count'].iloc[0]
        print(f"Number of rows with a NULL 'offer_timestamp': {null_count}")

        # --- [3] Format & Range Audit ---
        print("\n--- [3] Format & Range Audit (Sample of distinct values) ---")
        format_query = """
        SELECT
            MIN(offer_timestamp) AS earliest_timestamp,
            MAX(offer_timestamp) AS latest_timestamp
        FROM offers
        WHERE offer_timestamp IS NOT NULL;
        """
        df_format = pd.read_sql_query(format_query, conn)
        display(df_format)

        total_rows_query = "SELECT COUNT(*) as total FROM offers;"
        total_rows = pd.read_sql_query(total_rows_query, conn)['total'].iloc[0]

        # --- [4] Final Diagnosis ---
        print("\n--- [4] Final Diagnosis ---")
        if null_count == total_rows:
            print("🛑 CRITICAL FINDING: All rows have a NULL 'offer_timestamp'. The ETL conversion is failing.")
        elif null_count > 0:
            print(f"⚠️ WARNING: {null_count} out of {total_rows} rows have a NULL timestamp. Verify if those are the imputed rows")
        else:
            print("✅ SUCCESS: The 'offer_timestamp' column is fully populated with no NULL values.")
            # We can also check the format of the first valid timestamp
            first_ts = pd.read_sql_query("SELECT offer_timestamp FROM offers WHERE offer_timestamp IS NOT NULL LIMIT 1;", conn).iloc[0,0]
            print(f"Sample format check: '{first_ts}'. This appears to be in the correct 'YYYY-MM-DD HH:MM:SS' format for SQLite.")

except Exception as e:
    print(f"❌ ERROR during audit: {e}")

--- Performing a deep-dive audit of the 'offer_timestamp' column ---

--- [1] Schema Audit ---


,cid,name,type,notnull,dflt_value,pk
4,4,offer_timestamp,TEXT,0,None,0



--- [2] Null Value Audit ---
Number of rows with a NULL 'offer_timestamp': 0

--- [3] Format & Range Audit (Sample of distinct values) ---


,earliest_timestamp,latest_timestamp
0,2025-08-22 06:44:33,2025-10-01 09:56:26



--- [4] Final Diagnosis ---
✅ SUCCESS: The 'offer_timestamp' column is fully populated with no NULL values.
Sample format check: '2025-08-22 06:44:33'. This appears to be in the correct 'YYYY-MM-DD HH:MM:SS' format for SQLite.


In [ ]:
# =========================================================
# === QUERY 4: Tarifa Promedio por Día de la Semana ===
# =========================================================
print("\n--- Calculando la tarifa promedio (upfront_fare) por día de la semana ---")

# Esta consulta utiliza strftime('%w', ...) para agrupar por el día de la semana.
# Una declaración CASE se utiliza para convertir el número (0-6) en un nombre legible.
query_sql = """
SELECT
    CASE strftime('%w', offer_timestamp)
        WHEN '0' THEN '0 - Domingo'
        WHEN '1' THEN '1 - Lunes'
        WHEN '2' THEN '2 - Martes'
        WHEN '3' THEN '3 - Miércoles'
        WHEN '4' THEN '4 - Jueves'
        WHEN '5' THEN '5 - Viernes'
        WHEN '6' THEN '6 - Sábado'
    END AS day_of_week,
    AVG(upfront_fare) AS average_fare
FROM
    v_reconciled_offer
WHERE
    upfront_fare IS NOT NULL  -- Asegurarnos de no incluir ofertas sin tarifa en el promedio
GROUP BY
    day_of_week
ORDER BY
    strftime('%w', offer_timestamp); -- Ordenar por el número del día para mantener el orden de la semana
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_avg_fare = pd.read_sql_query(query_sql, conn)

    print("✅ SUCCESS: La consulta se ejecutó correctamente.")
    print("Mostrando la tarifa promedio por día de la semana:")

    # Usamos .style.format para presentar el número de una manera más limpia
    display(df_avg_fare.style.format({'average_fare': '{:,.2f}'}))

except Exception as e:
    print(f"❌ ERROR: La consulta falló. Razón: {e}")


--- Calculando la tarifa promedio (upfront_fare) por día de la semana ---
✅ SUCCESS: La consulta se ejecutó correctamente.
Mostrando la tarifa promedio por día de la semana:


,day_of_week,average_fare
0,0 - Domingo,106.12
1,1 - Lunes,123.02
2,2 - Martes,124.36
3,3 - Miércoles,133.91
4,4 - Jueves,138.77
5,5 - Viernes,141.80
6,6 - Sábado,124.83


In [ ]:
# =========================================================
# === QUERY 5: Average Fare by Hour of the Day ===
# =========================================================
print("\n--- Calculating average upfront_fare by hour of the day ---")

# This query uses strftime('%H', ...) to extract the hour.
query = """
SELECT
    strftime('%H', offer_timestamp) as hour_of_day,
    COUNT(offer_id) as number_of_offers,
    AVG(upfront_fare) as average_fare
FROM
    v_reconciled_offer -- Using our view is easiest!
WHERE
    upfront_fare IS NOT NULL
GROUP BY
    hour_of_day
ORDER BY
    hour_of_day;
"""

conn = sqlite3.connect(db_file_path)
df_hod_agg = pd.read_sql_query(query, conn)
conn.close()

from IPython.display import display
print("Average Fare by Hour of Day:")
display(df_hod_agg)

# --- Bonus Visualization ---
# A bar chart is the perfect way to visualize this result.
# We need to make sure the hour column is treated correctly for plotting.
df_hod_agg['hour_of_day'] = pd.to_numeric(df_hod_agg['hour_of_day'])
df_hod_agg = df_hod_agg.set_index('hour_of_day')



--- Calculating average upfront_fare by hour of the day ---
Average Fare by Hour of Day:


,hour_of_day,number_of_offers,average_fare
0,05,72,119.513611
1,06,797,133.209297
2,07,595,129.524924
3,08,414,122.646739
4,09,157,91.632229
5,10,68,92.844706
6,11,106,102.010755
7,12,112,104.132946
8,13,511,103.427397
9,14,496,145.397923


In [ ]:
# ==============================================================================
# === QUERY 5: La Hipótesis Central de Opus ===
# ==============================================================================
print("\n--- Calculando la tarifa promedio para ofertas Aceptadas vs. Rechazadas ---")

# Esta consulta agrupa todas las ofertas por su estado (aceptada/rechazada)
# y calcula la tarifa promedio para cada grupo.
query_sql = """
SELECT
    offer_action,
    AVG(upfront_fare) AS average_fare,
    COUNT(*) AS total_offers
FROM
    v_reconciled_offer
WHERE
    -- Nos aseguramos de incluir solo los dos grupos que nos interesan
    offer_action IN ('accepted', 'reject')
    AND upfront_fare IS NOT NULL
GROUP BY
    offer_action
ORDER BY
    average_fare DESC; -- Ordenamos para ver inmediatamente qué grupo tiene la tarifa más alta
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_hypothesis = pd.read_sql_query(query_sql, conn)

    print("✅ SUCCESS: La consulta de la hipótesis central se ejecutó correctamente.")
    print("Mostrando los resultados:")

    # Aplicamos un formato elegante para la moneda y los números enteros
    display(df_hypothesis.style.format({
        'average_fare': 'MXN {:,.2f}',
        'total_offers': '{:,}'
    }).hide(axis="index")) # Ocultamos el índice para una presentación más limpia

except Exception as e:
    print(f"❌ ERROR: La consulta falló. Razón: {e}")


--- Calculando la tarifa promedio para ofertas Aceptadas vs. Rechazadas ---
✅ SUCCESS: La consulta de la hipótesis central se ejecutó correctamente.
Mostrando los resultados:


offer_action,average_fare,total_offers
accepted,MXN 147.59,346
reject,MXN 129.88,"4,419"


In [ ]:
import gspread
import pandas as pd
from google.colab import auth

# Authenticate with your Google Account
auth.authenticate_user()

# Authorize the gspread client
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

# --- This is the core of your request ---
# Open the spreadsheet by its exact name
spreadsheet = gc.open("GTS-4")

# Select the worksheet by its exact name
worksheet = spreadsheet.worksheet("trip_events")


# ===================================================================
# THIS REPLACES THE ORIGINAL PLACEHOLDER
# ===================================================================

# Get all records from the selected worksheet and load into a DataFrame
df_raw_gts = pd.DataFrame(worksheet.get_all_records())

# ===================================================================
# THE REST OF YOUR PHASE 1 BLUEPRINT CONTINUES FROM HERE
# ===================================================================

# Rename the legacy trip identifier for clarity and precision
df_raw_gts.rename(columns={'event_id_legacy': 'trip_id_legacy'}, inplace=True)

# Create the new, unique, and non-negotiable Primary Key for this table
df_raw_gts.reset_index(inplace=True)
df_raw_gts.rename(columns={'index': 'event_id'}, inplace=True)
df_raw_gts['event_id'] = df_raw_gts['event_id'] + 1 # Start PKs at 1

# Ensure timestamp is in the correct format
df_raw_gts['event_timestamp'] = pd.to_datetime(df_raw_gts['event_timestamp'], errors='coerce')

# --- We now have a staged DataFrame that is ready for Phase 2 ---
df_staged = df_raw_gts

# ===================================================================
# BLOCK 1.5: DATA PURITY & SCHEMA ENFORCEMENT
# ===================================================================

# 1. Handle Missing Timestamps (CRITICAL)
# An event without a timestamp is not a valid record. Drop it.
df_staged.dropna(subset=['event_timestamp'], inplace=True)

# 2. Coerce Numeric Types
# Define all columns that MUST be numeric
numeric_cols = ['event_lat', 'event_lon', 'upfront_fare', 'realized_fare']
for col in numeric_cols:
    df_staged[col] = pd.to_numeric(df_staged[col], errors='coerce')

# 3. Coerce Boolean Type
# Use a robust mapping to handle different possible string values
boolean_map = {'TRUE': True, 'FALSE': False, True: True, False: False}
df_staged['is_imputed'] = df_staged['is_imputed'].map(boolean_map)
# Fill any remaining NaNs (from empty cells in the source) with a safe default: False
df_staged['is_imputed'] = df_staged['is_imputed'].fillna(False)

# 4. Final Health Check
# This is our quality gate. The output of this MUST be perfect.
print("\n--- After Data Purity Enforcement ---")
df_staged.info()

# Display the first few rows to verify
print(df_staged.head())
print(df_staged.info())


--- After Data Purity Enforcement ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1031 entries, 0 to 1030
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   event_id           1031 non-null   int64         
 1   trip_id_legacy     1031 non-null   object        
 2   event_timestamp    1031 non-null   datetime64[ns]
 3   event_types_id_fk  1031 non-null   object        
 4   event_lat          966 non-null    float64       
 5   event_lon          966 non-null    float64       
 6   event_address      1031 non-null   object        
 7   upfront_fare       259 non-null    float64       
 8   realized_fare      249 non-null    float64       
 9   is_imputed         1031 non-null   bool          
 10  comment            1031 non-null   object        
 11  source             1031 non-null   object        
dtypes: bool(1), datetime64[ns](1), float64(4), int64(1), object(5)
memory usage: 89

In [ ]:
# ===================================================================
# PHASE 2: TRANSFORM & NORMALIZE (event_types) v2.0
# ===================================================================

print("\n--- PHASE 2: TRANSFORMING & NORMALIZING event_types ---")

# ... (Steps 1-3 remain the same) ...

# 1. Isolate unique event descriptions from the source data
# The 'event_type' column in your sheet is actually the description.
unique_event_descriptions = df_staged['event_types_id_fk'].unique()

# 2. Create the base lookup DataFrame
df_event_types = pd.DataFrame({'description': unique_event_descriptions})

# 3. Engineer the 'event_code' from the description
# As per our ERD, we need a short code (e.g., 'T1', 'T3'). We'll parse this from the string.
# Example: "T1: Ride Accepted, Driving to Pickup" -> "T1"
df_event_types['event_code'] = df_event_types['description'].str.split(':').str[0]



# 4. REORDER THE TABLE as per your directive
# Create a custom order based on the logical flow of a trip
event_order = ['T0', 'T1', 'T2', 'T3', 'T4']
df_event_types['event_code'] = pd.Categorical(df_event_types['event_code'], categories=event_order, ordered=True)
df_event_types = df_event_types.sort_values('event_code')

# 5. Create the Primary Key AFTER sorting
df_event_types.reset_index(drop=True, inplace=True) # drop=True is important here
df_event_types.reset_index(inplace=True)
df_event_types.rename(columns={'index': 'event_type_id'}, inplace=True)
df_event_types['event_type_id'] = df_event_types['event_type_id'] + 1 # Start PKs at 1

# 6. Final Schema Alignment
final_columns = ['event_type_id', 'event_code', 'description']
df_event_types = df_event_types[final_columns]

# 7. Load into SQLite using a CONTEXT MANAGER (The Robust Fix)
import sqlite3

try:
    # This block ensures the connection is opened, used, and then automatically closed.
    with sqlite3.connect(db_file_path) as conn:
        df_event_types.to_sql('event_types', conn, if_exists='replace', index=False)
    print("SUCCESS: 'event_types' table has been created and populated.")
    print(f"Total event types loaded: {len(df_event_types)}")
except Exception as e:
    print(f"ERROR: Failed to load 'event_types' table. Reason: {e}")

# --- Display the final lookup table for verification ---
print("\n--- Final event_types Lookup Table ---")
print(df_event_types)


--- PHASE 2: TRANSFORMING & NORMALIZING event_types ---
SUCCESS: 'event_types' table has been created and populated.
Total event types loaded: 5

--- Final event_types Lookup Table ---
   event_type_id event_code                           description
0              1         T0                 T0: Looking for rides
1              2         T1  T1: Ride Accepted, Driving to Pickup
2              3         T2             T2: Waiting for passenger
3              4         T3                      T3: Ride Started
4              5         T4                    T4: Ride completed


In [ ]:
# ===================================================================
# PHASE 3: ENRICH & LOAD (trip_events) - CORRECTED
# ===================================================================

print("\n--- PHASE 3: ENRICHING & LOADING trip_events ---")

# 1. Enrich the main DataFrame by joining with the lookup table
df_trip_events_prep = pd.merge(
    df_staged,
    df_event_types[['description', 'event_type_id']], # df_event_types fue creado en la celda anterior
    left_on='event_types_id_fk', # El nombre de la columna de texto en el DataFrame de origen
    right_on='description',
    how='left'
)

# === CORRECCIÓN ARQUITECTÓNICA ===
# 2. Descartar las columnas de texto redundantes y renombrar el ID numérico
# Nos deshacemos de la columna de texto original y de la columna de descripción del join.
df_trip_events_prep.drop(columns=['event_types_id_fk', 'description'], inplace=True)
# Renombramos la columna de ID numérico al nombre de clave foránea canónico de nuestro esquema.
df_trip_events_prep.rename(columns={'event_type_id': 'event_types_id_fk'}, inplace=True)
# ================================

# 3. Final Schema Alignment for the 'trip_events' table
# La lista de columnas ahora se alinea con las columnas que realmente tenemos
final_columns = [
    'event_id',
    'event_timestamp',
    'event_lat',
    'event_lon',
    'event_address',
    'upfront_fare',
    'realized_fare',
    'source',
    'is_imputed',
    'comment',
    'trip_id_legacy',
    'event_types_id_fk' # ¡CORREGIDO! Esta es ahora la columna de ID numérico renombrada
]

# Aseguramos que solo seleccionamos columnas que existen para evitar errores
existing_final_columns = [col for col in final_columns if col in df_trip_events_prep.columns]
df_trip_events = df_trip_events_prep[existing_final_columns]

# 4. Load the final, enriched DataFrame into the SQLite database
try:
    with sqlite3.connect(db_file_path) as conn:
        df_trip_events.to_sql('trip_events', conn, if_exists='replace', index=False)
    print("SUCCESS: 'trip_events' table has been created and populated.")
    print(f"Total events loaded: {len(df_trip_events)}")
except Exception as e:
    print(f"ERROR: Failed to load 'trip_events' table. Reason: {e}")

# --- Final verification ---
print("\n--- Verifying the first 5 rows of the loaded trip_events data ---")
display(df_trip_events.head())
print("\n--- Verifying the schema of the loaded trip_events data ---")
df_trip_events.info()


--- PHASE 3: ENRICHING & LOADING trip_events ---
SUCCESS: 'trip_events' table has been created and populated.
Total events loaded: 1031

--- Verifying the first 5 rows of the loaded trip_events data ---


,event_id,event_timestamp,event_lat,event_lon,event_address,upfront_fare,realized_fare,source,is_imputed,comment,trip_id_legacy,event_types_id_fk
0,1,2025-08-22 06:48:00,NaN,NaN,N/A,136.53,NaN,GTS-1,False,N/A,250822-01,2
1,2,2025-08-22 07:00:00,19.469418,-99.164297,"Colonia Ampliación Del Gas, Mexico City, Azcap...",NaN,NaN,GTS-1,False,N/A,250822-01,4
2,3,2025-08-22 07:22:05,19.435358,-99.182785,"Avenida Homero, Polanco 1ª Sección, Mexico Cit...",NaN,114.49,GTS-1,False,N/A,250822-01,5
3,4,2025-08-22 07:28:00,NaN,NaN,N/A,162.00,NaN,GTS-1,False,N/A,250822-02,2
4,5,2025-08-22 07:38:00,19.453696,-99.208148,"35, Callejón San Joaquín, Argentina Antigua, M...",NaN,NaN,GTS-1,False,N/A,250822-02,4



--- Verifying the schema of the loaded trip_events data ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1031 entries, 0 to 1030
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   event_id           1031 non-null   int64         
 1   event_timestamp    1031 non-null   datetime64[ns]
 2   event_lat          966 non-null    float64       
 3   event_lon          966 non-null    float64       
 4   event_address      1031 non-null   object        
 5   upfront_fare       259 non-null    float64       
 6   realized_fare      249 non-null    float64       
 7   source             1031 non-null   object        
 8   is_imputed         1031 non-null   bool          
 9   comment            1031 non-null   object        
 10  trip_id_legacy     1031 non-null   object        
 11  event_types_id_fk  1031 non-null   int64         
dtypes: bool(1), datetime64[ns](1), float64(4), int64(2), obje

In [ ]:
# ===================================================================
# QUERY 1: VERIFY DATA PRESENCE IN 'trip_events'
# ===================================================================
import pandas as pd
import sqlite3

db_file_path = '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'
query_1 = "SELECT * FROM trip_events LIMIT 20;"

print("--- Running Query 1: 'Can I see the data?' ---")
try:
    with sqlite3.connect(db_file_path) as conn:
        df_query_1_results = pd.read_sql_query(query_1, conn)

    print("SUCCESS: Query executed. Displaying first 20 rows from the database:")
    display(df_query_1_results)

except Exception as e:
    print(f"ERROR: Query 1 failed. Reason: {e}")

--- Running Query 1: 'Can I see the data?' ---
SUCCESS: Query executed. Displaying first 20 rows from the database:


,event_id,event_timestamp,event_lat,event_lon,event_address,upfront_fare,realized_fare,source,is_imputed,comment,trip_id_legacy,event_types_id_fk
0,1,2025-08-22 06:48:00,NaN,NaN,N/A,136.53,NaN,GTS-1,0,N/A,250822-01,2
1,2,2025-08-22 07:00:00,19.469418,-99.164297,"Colonia Ampliación Del Gas, Mexico City, Azcap...",NaN,NaN,GTS-1,0,N/A,250822-01,4
2,3,2025-08-22 07:22:05,19.435358,-99.182785,"Avenida Homero, Polanco 1ª Sección, Mexico Cit...",NaN,114.49,GTS-1,0,N/A,250822-01,5
3,4,2025-08-22 07:28:00,NaN,NaN,N/A,162.00,NaN,GTS-1,0,N/A,250822-02,2
4,5,2025-08-22 07:38:00,19.453696,-99.208148,"35, Callejón San Joaquín, Argentina Antigua, M...",NaN,NaN,GTS-1,0,N/A,250822-02,4
5,6,2025-08-22 08:10:43,19.386737,-99.253680,"Calle Paseo de los Tamarindos, Bosques de las ...",NaN,135.72,GTS-1,0,N/A,250822-02,5
6,7,2025-08-22 08:12:00,NaN,NaN,N/A,64.31,NaN,GTS-1,0,N/A,250822-03,2
7,8,2025-08-22 08:17:00,19.373797,-99.259800,"Calle Roberto Medellín, Santa Fe Peña Blanca, ...",NaN,NaN,GTS-1,0,N/A,250822-03,4
8,9,2025-08-22 08:28:17,19.404960,-99.241322,"Calle Bosque de Duraznos, Bosques de las Lomas...",NaN,57.53,GTS-1,0,N/A,250822-03,5
9,10,2025-08-22 08:38:00,NaN,NaN,N/A,146.00,NaN,GTS-1,0,N/A,250822-04,2


In [ ]:
# ===================================================================
# QUERY: Get the LAST 20 records from 'trip_events'
# ===================================================================
import pandas as pd
import sqlite3

# Using the canonical db_file_path variable from our config block
# db_file_path = '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'

# The SQL query to get the last 20 records
query_last_20 = """
SELECT *
FROM trip_events
ORDER BY event_id DESC -- Sort by the primary key in descending order
LIMIT 20;
"""

print("--- Running Query: 'Can I see the LAST 20 records?' ---")
try:
    with sqlite3.connect(db_file_path) as conn:
        df_query_last_20_results = pd.read_sql_query(query_last_20, conn)

    print("SUCCESS: Query executed. Displaying the last 20 rows from the database:")
    # We might want to re-sort it back to ascending for display purposes
    display(df_query_last_20_results.sort_values('event_id'))

except Exception as e:
    print(f"ERROR: Query failed. Reason: {e}")

--- Running Query: 'Can I see the LAST 20 records?' ---
SUCCESS: Query executed. Displaying the last 20 rows from the database:


,event_id,event_timestamp,event_lat,event_lon,event_address,upfront_fare,realized_fare,source,is_imputed,comment,trip_id_legacy,event_types_id_fk
19,1012,2025-10-01 07:10:17,19.444644,-99.200335,"Torre Río 436, 436, Avenida Río San Joaquín, N...",NaN,NaN,GTS-4,0,N/A,251001-03,3
18,1013,2025-10-01 07:11:27,19.444644,-99.200335,"Torre Río 436, 436, Avenida Río San Joaquín, N...",NaN,NaN,GTS-4,0,N/A,251001-03,4
17,1014,2025-10-01 07:43:01,19.388093,-99.251259,"Sambalca Enterprise, Calle Paseo de los Tamari...",NaN,156.07,GTS-4,0,N/A,251001-03,5
16,1015,2025-10-01 07:43:53,19.390382,-99.249264,"14, Paseo de los Laureles, Bosques de las Loma...",133.47,NaN,GTS-4,0,N/A,251001-04,2
15,1016,2025-10-01 07:45:39,19.391369,-99.252085,"Calle Bosque de los Tabachines, Bosques de las...",NaN,NaN,GTS-4,0,N/A,251001-04,3
14,1017,2025-10-01 07:47:21,19.391383,-99.252093,"Calle Bosque de los Tabachines, Bosques de las...",NaN,NaN,GTS-4,0,N/A,251001-04,4
13,1018,2025-10-01 07:53:08,19.381855,-99.267378,"Colonia La Puntada, Lomas de Vista Hermosa, Me...",NaN,120.01,GTS-4,0,N/A,251001-04,5
12,1019,2025-10-01 07:53:16,19.381855,-99.267379,"Colonia La Puntada, Lomas de Vista Hermosa, Me...",NaN,NaN,GTS-4,0,N/A,251001-05,1
11,1020,2025-10-01 07:57:49,19.392293,-99.251349,"Calle Bosque de los Tabachines, Bosques de las...",120.57,NaN,GTS-4,0,N/A,251001-05,2
10,1021,2025-10-01 08:04:21,19.389307,-99.259022,"Privada Calle Ballonetas, Colonia Lomas del Ch...",NaN,NaN,GTS-4,0,N/A,251001-05,3


In [ ]:
# ==============================================================================
# === DIAGNOSTIC AUDIT: Investigating the Failing JOIN in the "Money Table" Query ===
# ==============================================================================
import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("WARNING: 'db_file_path' not found in global scope. Using default path.")

print("--- Auditing the JOIN condition between 'trip_events' and 'event_types' ---")
print("The failing condition is: te.event_types_id_fk = et.event_type_id")
print("We will now inspect the actual data on both sides of this equation.")

try:
    with sqlite3.connect(db_file_path) as conn:
        # --- Audit 1: What values are we trying to join FROM? ---
        print("\n--- [1] Unique values found in the 'trip_events.event_types_id_fk' column ---")
        query1 = "SELECT DISTINCT event_types_id_fk FROM trip_events;"
        df_from = pd.read_sql_query(query1, conn)
        display(df_from)

        # --- Audit 2: What values are we trying to join TO? ---
        print("\n--- [2] All values in the 'event_types' lookup table ---")
        query2 = "SELECT event_type_id, event_code, description FROM event_types;"
        df_to = pd.read_sql_query(query2, conn)
        display(df_to)

except Exception as e:
    print(f"❌ ERROR during diagnostic audit: {e}")

--- Auditing the JOIN condition between 'trip_events' and 'event_types' ---
The failing condition is: te.event_types_id_fk = et.event_type_id
We will now inspect the actual data on both sides of this equation.

--- [1] Unique values found in the 'trip_events.event_types_id_fk' column ---


,event_types_id_fk
0,2
1,4
2,5
3,1
4,3



--- [2] All values in the 'event_types' lookup table ---


,event_type_id,event_code,description
0,1,T0,T0: Looking for rides
1,2,T1,"T1: Ride Accepted, Driving to Pickup"
2,3,T2,T2: Waiting for passenger
3,4,T3,T3: Ride Started
4,5,T4,T4: Ride completed


In [ ]:
# ===================================================================
# === FINAL ANALYTICAL QUERY (v3.0 - Production) ===
# This cell uses the canonical 'db_file_path' defined in CELL 1.
# ===================================================================
import pandas as pd
import sqlite3

print("--- INITIALIZING ANALYTICAL QUERY ---")

# We use the 'db_file_path' variable defined in your main setup cell.
# This ensures we are always connecting to the correct, canonical database.
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    print("🛑 CRITICAL FAILURE: The 'db_file_path' variable is not defined.")
    print("Please ensure you have run the main setup cell at the top of the notebook.")
else:
    print(f"✅ Connecting to database at: {db_file_path}")

    # The SQL query using 'trip_id_legacy'
    sql_query_v2 = """
    WITH trip_financials AS (
        SELECT
            trip_id_legacy,
            MAX(CASE WHEN et.event_code = 'T1' THEN te.upfront_fare ELSE NULL END) AS trip_upfront_fare,
            MAX(CASE WHEN et.event_code = 'T4' THEN te.realized_fare ELSE NULL END) AS trip_real_fare
        FROM
            trip_events te
        JOIN
            event_types et ON te.event_types_id_fk = et.event_type_id
        WHERE
            te.trip_id_legacy IS NOT NULL
        GROUP BY
            te.trip_id_legacy
    )
    SELECT * FROM trip_financials;
    """

    try:
        with sqlite3.connect(db_file_path) as conn:
            df_results_v2 = pd.read_sql_query(sql_query_v2, conn)

        print("\n✅ SUCCESS: CTE query executed.")
        print("Displaying the aggregated 'trip_financials' data:")
        display(df_results_v2.head(20)) # Display first 20 for brevity

    except Exception as e:
        print(f"\n🛑 ERROR: Query failed. This is now likely a SQL syntax error. Reason: {e}")

--- INITIALIZING ANALYTICAL QUERY ---
✅ Connecting to database at: /content/drive/MyDrive/_Pienza/Assets/Database/opus.db

✅ SUCCESS: CTE query executed.
Displaying the aggregated 'trip_financials' data:


,trip_id_legacy,trip_upfront_fare,trip_real_fare
0,250822-01,136.53,114.49
1,250822-02,162.00,135.72
2,250822-03,64.31,57.53
3,250822-04,146.00,130.60
4,250822-05,60.00,49.93
5,250822-06,108.67,86.91
6,250822-07,170.00,129.02
7,250822-08,173.44,131.25
8,250822-09,89.00,76.50
9,250822-10,202.00,155.24


In [ ]:
# ===================================================================
# === FINAL ANALYTICAL CELL (v4.1 - Schema Reconciled) ===
# This cell contains EVERYTHING it needs to run. Zero external dependencies.
# ===================================================================
import pandas as pd
import sqlite3
import os

# --- [1] DEFINE ENVIRONMENT & QUERY ---
# All definitions are contained within this single cell.
db_file_path = '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'

# THE QUERY IS NOW CORRECTED TO MATCH THE NEW SCHEMA.
query_the_money = """
WITH trip_summary_raw AS (
    SELECT
        trip_id_legacy,
        MAX(CASE WHEN et.event_code = 'T0' THEN te.event_timestamp ELSE NULL END) AS t0_timestamp,
        MAX(CASE WHEN et.event_code = 'T1' THEN te.event_timestamp ELSE NULL END) AS t1_timestamp,
        MAX(CASE WHEN et.event_code = 'T2' THEN te.event_timestamp ELSE NULL END) AS t2_timestamp,
        MAX(CASE WHEN et.event_code = 'T3' THEN te.event_timestamp ELSE NULL END) AS t3_timestamp,
        MAX(CASE WHEN et.event_code = 'T4' THEN te.event_timestamp ELSE NULL END) AS t4_timestamp,
        MAX(CASE WHEN et.event_code = 'T1' THEN te.upfront_fare ELSE NULL END) AS t1_upfront_fare,
        MAX(CASE WHEN et.event_code = 'T4' THEN te.realized_fare ELSE NULL END) AS t4_realized_fare
    FROM
        trip_events te
    JOIN
        event_types et ON te.event_types_id_fk = et.event_type_id -- CORRECTED JOIN CONDITION
    GROUP BY
        te.trip_id_legacy
)
SELECT * FROM trip_summary_raw;
"""

print(f"--- INITIALIZING: Targeting database at '{db_file_path}' ---")


# --- [2] EXECUTE & VALIDATE ---
if not os.path.exists(db_file_path):
    print("🛑 CRITICAL FAILURE: The database file does NOT exist.")
else:
    print("✅ Pre-flight check PASSED. Database file found.")
    try:
        with sqlite3.connect(db_file_path) as conn:
            # The query variable is now guaranteed to exist and be correct.
            df_money_table = pd.read_sql_query(query_the_money, conn)

        print("\n✅ SUCCESS: Query executed. The money table has been shown.")

        # Data Type Conversion
        for col in ['t0_timestamp', 't1_timestamp', 't2_timestamp', 't3_timestamp', 't4_timestamp']:
            df_money_table[col] = pd.to_datetime(df_money_table[col], errors='coerce')

        display(df_money_table)

    except Exception as e:
        print(f"\n🛑 ERROR: Query failed. This is a SQL syntax error. Reason: {e}")

--- INITIALIZING: Targeting database at '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db' ---
✅ Pre-flight check PASSED. Database file found.

✅ SUCCESS: Query executed. The money table has been shown.


,trip_id_legacy,t0_timestamp,t1_timestamp,t2_timestamp,t3_timestamp,t4_timestamp,t1_upfront_fare,t4_realized_fare
0,250822-01,NaT,2025-08-22 06:48:00,NaT,2025-08-22 07:00:00,2025-08-22 07:22:05,136.53,114.49
1,250822-02,NaT,2025-08-22 07:28:00,NaT,2025-08-22 07:38:00,2025-08-22 08:10:43,162.00,135.72
2,250822-03,NaT,2025-08-22 08:12:00,NaT,2025-08-22 08:17:00,2025-08-22 08:28:17,64.31,57.53
3,250822-04,NaT,2025-08-22 08:38:00,NaT,2025-08-22 08:49:00,2025-08-22 09:17:10,146.00,130.60
4,250822-05,NaT,2025-08-22 09:16:00,NaT,2025-08-22 09:20:00,2025-08-22 09:29:01,60.00,49.93
...,...,...,...,...,...,...,...,...
254,251001-03,NaT,2025-10-01 07:00:04,2025-10-01 07:10:17,2025-10-01 07:11:27,2025-10-01 07:43:01,182.17,156.07
255,251001-04,NaT,2025-10-01 07:43:53,2025-10-01 07:45:39,2025-10-01 07:47:21,2025-10-01 07:53:08,133.47,120.01
256,251001-05,2025-10-01 07:53:16,2025-10-01 07:57:49,2025-10-01 08:04:21,2025-10-01 08:04:43,2025-10-01 08:34:35,120.57,106.95
257,251001-06,2025-10-01 08:34:44,2025-10-01 08:36:35,NaT,2025-10-01 08:44:11,2025-10-01 09:13:29,165.68,148.14


In [ ]:
# ==============================================================================
# === AUDITORÍA DE ESQUEMA: Obteniendo la Verdad Fundamental para 'trip_events' ===
# ==============================================================================
import pandas as pd
import sqlite3

print("--- Auditando el Esquema Canónico de la tabla 'trip_events' ---")

# La consulta PRAGMA es la única fuente de verdad para la estructura de una tabla
query_sql = "PRAGMA table_info(trip_events);"

try:
    with sqlite3.connect(db_file_path) as conn:
        # Usamos pandas para una salida limpia y tabular
        df_schema = pd.read_sql_query(query_sql, conn)

    print("\n✅ ÉXITO: Auditoría de esquema completada.")
    print("Esta es la lista definitiva y canónica de columnas tal como existen en la base de datos:")
    display(df_schema)

except Exception as e:
    print(f"❌ ERROR: La consulta de auditoría falló. Esto podría indicar un problema con la conexión a la base de datos o el nombre de la tabla. Razón: {e}")

--- Auditando el Esquema Canónico de la tabla 'trip_events' ---

✅ ÉXITO: Auditoría de esquema completada.
Esta es la lista definitiva y canónica de columnas tal como existen en la base de datos:


,cid,name,type,notnull,dflt_value,pk
0,0,event_id,INTEGER,0,None,0
1,1,event_timestamp,TIMESTAMP,0,None,0
2,2,event_lat,REAL,0,None,0
3,3,event_lon,REAL,0,None,0
4,4,event_address,TEXT,0,None,0
5,5,upfront_fare,REAL,0,None,0
6,6,realized_fare,REAL,0,None,0
7,7,source,TEXT,0,None,0
8,8,is_imputed,INTEGER,0,None,0
9,9,comment,TEXT,0,None,0


In [ ]:
# ===================================================================
# QUERY: Get the LAST 20 records from 'trip_events' (schema-aligned)
# ===================================================================
import pandas as pd
import sqlite3

# La consulta SQL, utilizando los nombres de columna canónicos y validados 'event_lat' y 'event_lon'
query_last_20 = """
SELECT
    event_id,
    event_timestamp,
    event_address,
    event_lat,       -- Validado
    event_lon,       -- Validado
    upfront_fare,
    realized_fare,
    source,
    is_imputed,
    comment,
    trip_id_legacy,
    event_types_id_fk
FROM
    trip_events
ORDER BY
    event_id DESC -- Ordenar por la clave primaria en orden descendente para obtener los últimos 20
LIMIT 20;
"""

print("--- Ejecutando Consulta Final: 'Ver los ÚLTIMOS 20 registros con geo-coordenadas' ---")
try:
    with sqlite3.connect(db_file_path) as conn:
        df_query_last_20_results = pd.read_sql_query(query_last_20, conn)

    print("✅ ÉXITO: Consulta ejecutada. Mostrando los últimos 20 registros de la base de datos:")
    # Reordenar de nuevo a ascendente para una visualización cronológica
    display(df_query_last_20_results.sort_values('event_id'))

except Exception as e:
    print(f"❌ ERROR: La consulta falló. Razón: {e}")

--- Ejecutando Consulta Final: 'Ver los ÚLTIMOS 20 registros con geo-coordenadas' ---
✅ ÉXITO: Consulta ejecutada. Mostrando los últimos 20 registros de la base de datos:


,event_id,event_timestamp,event_address,event_lat,event_lon,upfront_fare,realized_fare,source,is_imputed,comment,trip_id_legacy,event_types_id_fk
19,1012,2025-10-01 07:10:17,"Torre Río 436, 436, Avenida Río San Joaquín, N...",19.444644,-99.200335,NaN,NaN,GTS-4,0,N/A,251001-03,3
18,1013,2025-10-01 07:11:27,"Torre Río 436, 436, Avenida Río San Joaquín, N...",19.444644,-99.200335,NaN,NaN,GTS-4,0,N/A,251001-03,4
17,1014,2025-10-01 07:43:01,"Sambalca Enterprise, Calle Paseo de los Tamari...",19.388093,-99.251259,NaN,156.07,GTS-4,0,N/A,251001-03,5
16,1015,2025-10-01 07:43:53,"14, Paseo de los Laureles, Bosques de las Loma...",19.390382,-99.249264,133.47,NaN,GTS-4,0,N/A,251001-04,2
15,1016,2025-10-01 07:45:39,"Calle Bosque de los Tabachines, Bosques de las...",19.391369,-99.252085,NaN,NaN,GTS-4,0,N/A,251001-04,3
14,1017,2025-10-01 07:47:21,"Calle Bosque de los Tabachines, Bosques de las...",19.391383,-99.252093,NaN,NaN,GTS-4,0,N/A,251001-04,4
13,1018,2025-10-01 07:53:08,"Colonia La Puntada, Lomas de Vista Hermosa, Me...",19.381855,-99.267378,NaN,120.01,GTS-4,0,N/A,251001-04,5
12,1019,2025-10-01 07:53:16,"Colonia La Puntada, Lomas de Vista Hermosa, Me...",19.381855,-99.267379,NaN,NaN,GTS-4,0,N/A,251001-05,1
11,1020,2025-10-01 07:57:49,"Calle Bosque de los Tabachines, Bosques de las...",19.392293,-99.251349,120.57,NaN,GTS-4,0,N/A,251001-05,2
10,1021,2025-10-01 08:04:21,"Privada Calle Ballonetas, Colonia Lomas del Ch...",19.389307,-99.259022,NaN,NaN,GTS-4,0,N/A,251001-05,3


In [ ]:
# ==============================================================================
# === TASK 1: CREATE VIEW v_trip_funnel_metrics ===
# ==============================================================================
print("--- Architecting the 'v_trip_funnel_metrics' analytical view ---")

view_sql = """
CREATE VIEW v_trip_funnel_metrics AS
WITH PivotedEvents AS (
    -- This CTE pivots the event log, creating one row per trip_id_legacy
    -- with columns for each key event's timestamp and fare.
    SELECT
        trip_id_legacy,
        MAX(CASE WHEN event_types_id_fk = 1 THEN event_timestamp END) AS t0_looking,
        MAX(CASE WHEN event_types_id_fk = 2 THEN event_timestamp END) AS t1_accepted,
        MAX(CASE WHEN event_types_id_fk = 3 THEN event_timestamp END) AS t2_arrived,
        MAX(CASE WHEN event_types_id_fk = 4 THEN event_timestamp END) AS t3_started,
        MAX(CASE WHEN event_types_id_fk = 5 THEN event_timestamp END) AS t4_completed,
        MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END) AS upfront_fare,
        MAX(CASE WHEN event_types_id_fk = 5 THEN realized_fare END) AS realized_fare
    FROM
        trip_events
    GROUP BY
        trip_id_legacy
)
-- The final SELECT statement calculates the duration KPIs in seconds.
SELECT
    p.trip_id_legacy,
    p.upfront_fare,
    p.realized_fare,
    (julianday(p.t2_arrived) - julianday(p.t1_accepted)) * 86400.0 AS duration_to_pickup_sec,
    (julianday(p.t3_started) - julianday(p.t2_arrived)) * 86400.0 AS duration_waiting_sec,
    (julianday(p.t4_completed) - julianday(p.t3_started)) * 86400.0 AS duration_trip_sec,
    (julianday(p.t4_completed) - julianday(p.t1_accepted)) * 86400.0 AS duration_total_engagement_sec,
    p.t0_looking,
    p.t1_accepted,
    p.t2_arrived,
    p.t3_started,
    p.t4_completed
FROM
    PivotedEvents p;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_trip_funnel_metrics;")
        conn.execute(view_sql)
    print("✅ SUCCESS: View 'v_trip_funnel_metrics' created successfully.")

    # --- Verification Step ---
    print("\n--- Displaying first 10 rows of the new view for verification ---")
    with sqlite3.connect(db_file_path) as conn:
        df_verify = pd.read_sql_query("SELECT * FROM v_trip_funnel_metrics LIMIT 10;", conn)
        display(df_verify)

except Exception as e:
    print(f"❌ ERROR creating view: {e}")

--- Architecting the 'v_trip_funnel_metrics' analytical view ---
✅ SUCCESS: View 'v_trip_funnel_metrics' created successfully.

--- Displaying first 10 rows of the new view for verification ---


,trip_id_legacy,upfront_fare,realized_fare,duration_to_pickup_sec,duration_waiting_sec,duration_trip_sec,duration_total_engagement_sec,t0_looking,t1_accepted,t2_arrived,t3_started,t4_completed
0,250822-01,136.53,114.49,None,None,1325.000015,2045.000012,None,2025-08-22 06:48:00,None,2025-08-22 07:00:00,2025-08-22 07:22:05
1,250822-02,162.00,135.72,None,None,1962.999974,2562.999979,None,2025-08-22 07:28:00,None,2025-08-22 07:38:00,2025-08-22 08:10:43
2,250822-03,64.31,57.53,None,None,676.999989,976.999971,None,2025-08-22 08:12:00,None,2025-08-22 08:17:00,2025-08-22 08:28:17
3,250822-04,146.00,130.60,None,None,1690.000007,2350.000007,None,2025-08-22 08:38:00,None,2025-08-22 08:49:00,2025-08-22 09:17:10
4,250822-05,60.00,49.93,None,None,541.000003,781.000029,None,2025-08-22 09:16:00,None,2025-08-22 09:20:00,2025-08-22 09:29:01
5,250822-06,108.67,86.91,None,None,1079.999976,1319.999962,None,2025-08-22 15:06:00,None,2025-08-22 15:10:00,2025-08-22 15:28:00
6,250822-07,170.00,129.02,None,None,1817.000002,2056.999987,None,2025-08-22 15:30:00,None,2025-08-22 15:34:00,2025-08-22 16:04:17
7,250822-08,173.44,131.25,None,None,1293.000028,2013.000025,None,2025-08-22 16:09:00,None,2025-08-22 16:21:00,2025-08-22 16:42:33
8,250822-09,89.00,76.50,None,None,1623.000008,1862.999994,None,2025-08-22 16:51:20,None,2025-08-22 16:55:20,2025-08-22 17:22:23
9,250822-10,202.00,155.24,None,None,2235.999982,2978.999975,None,2025-08-22 17:10:00,None,2025-08-22 17:22:23,2025-08-22 17:59:39


In [ ]:
# ==============================================================================
# === TASK 1 (Self-Contained): CREATE VIEW v_trip_funnel_metrics ===
# ==============================================================================

# This cell is self-contained with all necessary imports per the new SOP.
import pandas as pd
import sqlite3
import os

# --- Configuration: Ensure db_file_path is defined ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("WARNING: 'db_file_path' not found in global scope. Using default path.")

print("--- Architecting the 'v_trip_funnel_metrics' analytical view ---")

view_sql = """
CREATE VIEW v_trip_funnel_metrics AS
WITH PivotedEvents AS (
    SELECT
        trip_id_legacy,
        MAX(CASE WHEN event_types_id_fk = 1 THEN event_timestamp END) AS t0_looking,
        MAX(CASE WHEN event_types_id_fk = 2 THEN event_timestamp END) AS t1_accepted,
        MAX(CASE WHEN event_types_id_fk = 3 THEN event_timestamp END) AS t2_arrived,
        MAX(CASE WHEN event_types_id_fk = 4 THEN event_timestamp END) AS t3_started,
        MAX(CASE WHEN event_types_id_fk = 5 THEN event_timestamp END) AS t4_completed,
        MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END) AS upfront_fare,
        MAX(CASE WHEN event_types_id_fk = 5 THEN realized_fare END) AS realized_fare
    FROM
        trip_events
    GROUP BY
        trip_id_legacy
)
SELECT
    p.trip_id_legacy,
    p.upfront_fare,
    p.realized_fare,
    (julianday(p.t2_arrived) - julianday(p.t1_accepted)) * 86400.0 AS duration_to_pickup_sec,
    (julianday(p.t3_started) - julianday(p.t2_arrived)) * 86400.0 AS duration_waiting_sec,
    (julianday(p.t4_completed) - julianday(p.t3_started)) * 86400.0 AS duration_trip_sec,
    (julianday(p.t4_completed) - julianday(p.t1_accepted)) * 86400.0 AS duration_total_engagement_sec,
    p.t0_looking,
    p.t1_accepted,
    p.t2_arrived,
    p.t3_started,
    p.t4_completed
FROM
    PivotedEvents p;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_trip_funnel_metrics;")
        conn.execute(view_sql)
    print("✅ SUCCESS: View 'v_trip_funnel_metrics' created successfully.")

    # --- Verification Step (CORRECTED) ---
    print("\n--- Displaying LAST 20 rows of the new view for verification ---")
    with sqlite3.connect(db_file_path) as conn:
        # CORRECTED: The query now orders by a timestamp DESC to get the most recent trips.
        verification_query = """
        SELECT *
        FROM v_trip_funnel_metrics
        ORDER BY t1_accepted DESC
        LIMIT 20;
        """
        df_verify = pd.read_sql_query(verification_query, conn)
        display(df_verify)

except Exception as e:
    print(f"❌ ERROR creating view: {e}")

--- Architecting the 'v_trip_funnel_metrics' analytical view ---
✅ SUCCESS: View 'v_trip_funnel_metrics' created successfully.

--- Displaying LAST 20 rows of the new view for verification ---


,trip_id_legacy,upfront_fare,realized_fare,duration_to_pickup_sec,duration_waiting_sec,duration_trip_sec,duration_total_engagement_sec,t0_looking,t1_accepted,t2_arrived,t3_started,t4_completed
0,251001-07,135.37,125.34,259.999998,48.000021,1992.000018,2300.000037,None,2025-10-01 09:13:54,2025-10-01 09:18:14,2025-10-01 09:19:02,2025-10-01 09:52:14
1,251001-06,165.68,148.14,NaN,NaN,1758.000000,2213.999981,2025-10-01 08:34:44,2025-10-01 08:36:35,None,2025-10-01 08:44:11,2025-10-01 09:13:29
2,251001-05,120.57,106.95,392.000006,22.000001,1791.999976,2205.999984,2025-10-01 07:53:16,2025-10-01 07:57:49,2025-10-01 08:04:21,2025-10-01 08:04:43,2025-10-01 08:34:35
3,251001-04,133.47,120.01,106.000029,101.999970,347.000009,555.000007,None,2025-10-01 07:43:53,2025-10-01 07:45:39,2025-10-01 07:47:21,2025-10-01 07:53:08
4,251001-03,182.17,156.07,613.000014,69.999982,1893.999986,2576.999983,None,2025-10-01 07:00:04,2025-10-01 07:10:17,2025-10-01 07:11:27,2025-10-01 07:43:01
5,251001-02,114.69,97.40,395.999984,70.000023,1053.999996,1520.000003,2025-10-01 06:30:56,2025-10-01 06:33:52,2025-10-01 06:40:28,2025-10-01 06:41:38,2025-10-01 06:59:12
6,251001-01,151.30,142.47,390.000017,168.999968,1119.000006,1677.999991,2025-10-01 05:59:11,2025-10-01 06:02:43,2025-10-01 06:09:13,2025-10-01 06:12:02,2025-10-01 06:30:41
7,250930-10,194.44,163.10,NaN,NaN,1545.000029,1802.000003,2025-09-30 15:40:18,2025-09-30 15:44:44,None,2025-09-30 15:49:01,2025-09-30 16:14:46
8,250930-09,102.68,84.81,NaN,NaN,1512.000006,1660.000008,None,2025-09-30 15:12:14,None,2025-09-30 15:14:42,2025-09-30 15:39:54
9,250930-08,108.91,102.68,501.000018,85.999976,818.000029,1405.000024,2025-09-30 14:41:41,2025-09-30 14:47:33,2025-09-30 14:55:54,2025-09-30 14:57:20,2025-09-30 15:10:58


In [ ]:
# =========================================================================================
# === TASK 1, STEP 1/2 (v1.2): CREATE VIEW v_trip_funnel_wide (Final Schema) ===
# =========================================================================================

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("WARNING: 'db_file_path' not found in global scope. Using default path.")

print("--- Architecting the foundational 'v_trip_funnel_wide' view (v1.2) ---")

# RE-CORRECTED: Column order for fares is now upfront_fare, then realized_fare.
view_sql = """
CREATE VIEW v_trip_funnel_wide AS
SELECT
    trip_id_legacy,
    MAX(CASE WHEN event_types_id_fk = 1 THEN event_timestamp END) AS t0_timestamp,
    MAX(CASE WHEN event_types_id_fk = 1 THEN event_address END)   AS t0_address,
    MAX(CASE WHEN event_types_id_fk = 2 THEN event_timestamp END) AS t1_timestamp,
    MAX(CASE WHEN event_types_id_fk = 2 THEN event_address END)   AS t1_address,
    MAX(CASE WHEN event_types_id_fk = 3 THEN event_timestamp END) AS t2_timestamp,
    MAX(CASE WHEN event_types_id_fk = 3 THEN event_address END)   AS t2_address,
    MAX(CASE WHEN event_types_id_fk = 4 THEN event_timestamp END) AS t3_timestamp,
    MAX(CASE WHEN event_types_id_fk = 4 THEN event_address END)   AS t3_address,
    MAX(CASE WHEN event_types_id_fk = 5 THEN event_timestamp END) AS t4_timestamp,
    MAX(CASE WHEN event_types_id_fk = 5 THEN event_address END)   AS t4_address,
    MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END)    AS upfront_fare,
    MAX(CASE WHEN event_types_id_fk = 5 THEN realized_fare END)   AS realized_fare
FROM
    trip_events
GROUP BY
    trip_id_legacy;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_trip_funnel_wide;")
        conn.execute(view_sql)
    print("✅ SUCCESS: Foundational view 'v_trip_funnel_wide' (v1.2) created successfully.")

    # --- Verification Step (per new SOP) ---
    print("\n--- Displaying LAST 10 rows of the new foundational view for verification ---")
    with sqlite3.connect(db_file_path) as conn:
        verification_query = """
        SELECT *
        FROM v_trip_funnel_wide
        ORDER BY t1_timestamp DESC
        LIMIT 10;
        """
        df_verify = pd.read_sql_query(verification_query, conn)
        display(df_verify)

except Exception as e:
    print(f"❌ ERROR creating view: {e}")

--- Architecting the foundational 'v_trip_funnel_wide' view (v1.2) ---
✅ SUCCESS: Foundational view 'v_trip_funnel_wide' (v1.2) created successfully.

--- Displaying LAST 10 rows of the new foundational view for verification ---


,trip_id_legacy,t0_timestamp,t0_address,t1_timestamp,t1_address,t2_timestamp,t2_address,t3_timestamp,t3_address,t4_timestamp,t4_address,upfront_fare,realized_fare
0,251001-07,None,None,2025-10-01 09:13:54,"Marriott Mexico City Reforma Hotel, 276, Aveni...",2025-10-01 09:18:14,"Calle Nápoles, Zona Rosa, Mexico City, Cuauhté...",2025-10-01 09:19:02,"Calle Hamburgo, Zona Rosa, Mexico City, Cuauht...",2025-10-01 09:52:14,"Avenida Paseo de las Palmas, Lomas de Chapulte...",135.37,125.34
1,251001-06,2025-10-01 08:34:44,"360, Calle Montes Urales Norte, Lomas de Chapu...",2025-10-01 08:36:35,"220, Calle Montes Urales Norte, Lomas de Chapu...",None,None,2025-10-01 08:44:11,Tecamachalco 11650,2025-10-01 09:13:29,"Marriott Mexico City Reforma Hotel, 276, Aveni...",165.68,148.14
2,251001-05,2025-10-01 07:53:16,"Colonia La Puntada, Lomas de Vista Hermosa, Me...",2025-10-01 07:57:49,"Calle Bosque de los Tabachines, Bosques de las...",2025-10-01 08:04:21,"Privada Calle Ballonetas, Colonia Lomas del Ch...",2025-10-01 08:04:43,"Privada Calle Ballonetas, Colonia Lomas del Ch...",2025-10-01 08:34:35,"360, Calle Montes Urales Norte, Lomas de Chapu...",120.57,106.95
3,251001-04,None,None,2025-10-01 07:43:53,"14, Paseo de los Laureles, Bosques de las Loma...",2025-10-01 07:45:39,"Calle Bosque de los Tabachines, Bosques de las...",2025-10-01 07:47:21,"Calle Bosque de los Tabachines, Bosques de las...",2025-10-01 07:53:08,"Colonia La Puntada, Lomas de Vista Hermosa, Me...",133.47,120.01
4,251001-03,None,None,2025-10-01 07:00:04,"Avenida Ejército Nacional Mexicano, Granada, M...",2025-10-01 07:10:17,"Torre Río 436, 436, Avenida Río San Joaquín, N...",2025-10-01 07:11:27,"Torre Río 436, 436, Avenida Río San Joaquín, N...",2025-10-01 07:43:01,"Sambalca Enterprise, Calle Paseo de los Tamari...",182.17,156.07
5,251001-02,2025-10-01 06:30:56,"18, Calle Alicama, Lomas de Chapultepec 4ª Sec...",2025-10-01 06:33:52,"Avenida Lomas, Colonia Lomas de Virreyes, Loma...",2025-10-01 06:40:28,"Calle Poniente 81, Colonia Cove, Mexico City, ...",2025-10-01 06:41:38,"Calle Poniente 81, Colonia Cove, Mexico City, ...",2025-10-01 06:59:12,"Avenida Ejército Nacional Mexicano, Granada, M...",114.69,97.40
6,251001-01,2025-10-01 05:59:11,"Calle Comte, Colonia Nueva Anzures, Anzures, M...",2025-10-01 06:02:43,"Avenida Río Mississippi, Cuauhtémoc, Mexico Ci...",2025-10-01 06:09:13,"Calle Eligio Ancona, Santa María la Ribera, Me...",2025-10-01 06:12:02,"228, Calle Nogal, Santa María la Ribera, Mexic...",2025-10-01 06:30:41,"Calle Pedregal, Lomas de Chapultepec 4ª Secció...",151.30,142.47
7,250930-10,2025-09-30 15:40:18,"Corporativo Lomas Cantabria, 22, Cerrada de la...",2025-09-30 15:44:44,"Extra Periférico, 168, Boulevard Manuel Ávila ...",None,None,2025-09-30 15:49:01,FC Cuernavaca,2025-09-30 16:14:46,"364, Avenida Paseo de la Reforma, Little Seoul...",194.44,163.10
8,250930-09,None,None,2025-09-30 15:12:14,"Avenida Vasco de Quiroga, Santa Fe Cuajimalpa,...",None,None,2025-09-30 15:14:42,Geocoding failed,2025-09-30 15:39:54,"Corporativo Lomas Cantabria, 22, Cerrada de la...",102.68,84.81
9,250930-08,2025-09-30 14:41:41,"Calle Bosque de Pirules, Colonia Bosques de Re...",2025-09-30 14:47:33,"Avenida Paseo de los Ahuehuetes Sur, Colonia B...",2025-09-30 14:55:54,"Roche, 9, Calle Molino Bezares, Lomas de Bezar...",2025-09-30 14:57:20,"Avenida Constituyentes - Puente CONAFRUT, Boul...",2025-09-30 15:10:58,"Calle José Villagrán, Santa Fe Cuajimalpa, Mex...",108.91,102.68


In [ ]:
# =========================================================================================
# === TASK 1, STEP 1/2 (v1.3): CREATE VIEW v_trip_funnel_wide (Geo-Enriched) ===
# =========================================================================================

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("WARNING: 'db_file_path' not found in global scope. Using default path.")

print("--- Architecting the foundational 'v_trip_funnel_wide' view (v1.3 - Geo-Enriched) ---")

# FINAL VERSION: Now includes lat/lon for each event.
view_sql = """
CREATE VIEW v_trip_funnel_wide AS
SELECT
    trip_id_legacy,
    -- T0 Event Group
    MAX(CASE WHEN event_types_id_fk = 1 THEN event_timestamp END) AS t0_timestamp,
    MAX(CASE WHEN event_types_id_fk = 1 THEN event_lat END)       AS t0_lat,
    MAX(CASE WHEN event_types_id_fk = 1 THEN event_lon END)       AS t0_lon,
    -- T1 Event Group
    MAX(CASE WHEN event_types_id_fk = 2 THEN event_timestamp END) AS t1_timestamp,
    MAX(CASE WHEN event_types_id_fk = 2 THEN event_lat END)       AS t1_lat,
    MAX(CASE WHEN event_types_id_fk = 2 THEN event_lon END)       AS t1_lon,
    -- T2 Event Group
    MAX(CASE WHEN event_types_id_fk = 3 THEN event_timestamp END) AS t2_timestamp,
    MAX(CASE WHEN event_types_id_fk = 3 THEN event_lat END)       AS t2_lat,
    MAX(CASE WHEN event_types_id_fk = 3 THEN event_lon END)       AS t2_lon,
    -- T3 Event Group
    MAX(CASE WHEN event_types_id_fk = 4 THEN event_timestamp END) AS t3_timestamp,
    MAX(CASE WHEN event_types_id_fk = 4 THEN event_lat END)       AS t3_lat,
    MAX(CASE WHEN event_types_id_fk = 4 THEN event_lon END)       AS t3_lon,
    -- T4 Event Group
    MAX(CASE WHEN event_types_id_fk = 5 THEN event_timestamp END) AS t4_timestamp,
    MAX(CASE WHEN event_types_id_fk = 5 THEN event_lat END)       AS t4_lat,
    MAX(CASE WHEN event_types_id_fk = 5 THEN event_lon END)       AS t4_lon,
    -- Financials
    MAX(CASE WHEN event_types_id_fk = 2 THEN upfront_fare END)    AS upfront_fare,
    MAX(CASE WHEN event_types_id_fk = 5 THEN realized_fare END)   AS realized_fare
FROM
    trip_events
GROUP BY
    trip_id_legacy;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_trip_funnel_wide;")
        conn.execute(view_sql)
    print("✅ SUCCESS: Foundational view 'v_trip_funnel_wide' (v1.3) created successfully.")

    # --- Verification Step (per new SOP) ---
    print("\n--- Displaying LAST 10 rows of the new geo-enriched view for verification ---")
    with sqlite3.connect(db_file_path) as conn:
        verification_query = """
        SELECT *
        FROM v_trip_funnel_wide
        ORDER BY t1_timestamp DESC
        LIMIT 10;
        """
        df_verify = pd.read_sql_query(verification_query, conn)
        display(df_verify)

except Exception as e:
    print(f"❌ ERROR creating view: {e}")

--- Architecting the foundational 'v_trip_funnel_wide' view (v1.3 - Geo-Enriched) ---
✅ SUCCESS: Foundational view 'v_trip_funnel_wide' (v1.3) created successfully.

--- Displaying LAST 10 rows of the new geo-enriched view for verification ---


,trip_id_legacy,t0_timestamp,t0_lat,t0_lon,t1_timestamp,t1_lat,t1_lon,t2_timestamp,t2_lat,t2_lon,t3_timestamp,t3_lat,t3_lon,t4_timestamp,t4_lat,t4_lon,upfront_fare,realized_fare
0,251001-07,None,NaN,NaN,2025-10-01 09:13:54,19.428175,-99.164291,2025-10-01 09:18:14,19.429299,-99.161305,2025-10-01 09:19:02,19.427816,-99.161410,2025-10-01 09:52:14,19.429272,-99.215838,135.37,125.34
1,251001-06,2025-10-01 08:34:44,19.429017,-99.207317,2025-10-01 08:36:35,19.429866,-99.210289,None,NaN,NaN,2025-10-01 08:44:11,NaN,NaN,2025-10-01 09:13:29,19.428157,-99.164338,165.68,148.14
2,251001-05,2025-10-01 07:53:16,19.381855,-99.267379,2025-10-01 07:57:49,19.392293,-99.251349,2025-10-01 08:04:21,19.389307,-99.259022,2025-10-01 08:04:43,19.389395,-99.258921,2025-10-01 08:34:35,19.429016,-99.207312,120.57,106.95
3,251001-04,None,NaN,NaN,2025-10-01 07:43:53,19.390382,-99.249264,2025-10-01 07:45:39,19.391369,-99.252085,2025-10-01 07:47:21,19.391383,-99.252093,2025-10-01 07:53:08,19.381855,-99.267378,133.47,120.01
4,251001-03,None,NaN,NaN,2025-10-01 07:00:04,19.438687,-99.205490,2025-10-01 07:10:17,19.444644,-99.200335,2025-10-01 07:11:27,19.444644,-99.200335,2025-10-01 07:43:01,19.388093,-99.251259,182.17,156.07
5,251001-02,2025-10-01 06:30:56,19.422493,-99.204313,2025-10-01 06:33:52,19.418508,-99.205599,2025-10-01 06:40:28,19.401792,-99.196382,2025-10-01 06:41:38,19.401792,-99.196382,2025-10-01 06:59:12,19.438657,-99.205160,114.69,97.40
6,251001-01,2025-10-01 05:59:11,19.428923,-99.174633,2025-10-01 06:02:43,19.428491,-99.173390,2025-10-01 06:09:13,19.453481,-99.162617,2025-10-01 06:12:02,19.453011,-99.163214,2025-10-01 06:30:41,19.422494,-99.204347,151.30,142.47
7,250930-10,2025-09-30 15:40:18,19.433761,-99.212895,2025-09-30 15:44:44,19.434579,-99.211605,None,NaN,NaN,2025-09-30 15:49:01,NaN,NaN,2025-09-30 16:14:46,19.426131,-99.168587,194.44,163.10
8,250930-09,None,NaN,NaN,2025-09-30 15:12:14,19.364700,-99.269883,None,NaN,NaN,2025-09-30 15:14:42,19.367542,-99.262921,2025-09-30 15:39:54,19.433774,-99.212901,102.68,84.81
9,250930-08,2025-09-30 14:41:41,19.393637,-99.260057,2025-09-30 14:47:33,19.394674,-99.254599,2025-09-30 14:55:54,19.387984,-99.243675,2025-09-30 14:57:20,19.387250,-99.243269,2025-09-30 15:10:58,19.363002,-99.273047,108.91,102.68


In [ ]:
# =========================================================================================
# === TASK 1, STEP 2/2: CREATE VIEW v_trip_final_kpis (The Polished Money Table) ===
# =========================================================================================

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("WARNING: 'db_file_path' not found in global scope. Using default path.")

print("--- Architecting the final 'v_trip_final_kpis' analytical view ---")

# This query is built ON TOP of our v_trip_funnel_wide view.
# It handles all final KPI calculations.
view_sql = """
CREATE VIEW v_trip_final_kpis AS
-- A CTE is used to cleanly aggregate the is_imputed flag for each trip.
WITH ImputedFlag AS (
    SELECT
        trip_id_legacy,
        MAX(is_imputed) AS is_imputed -- MAX() on a boolean (0/1) correctly gets the flag
    FROM
        trip_events
    GROUP BY
        trip_id_legacy
)
SELECT
    -- Core Identifiers & Financials from the foundational view
    v.trip_id_legacy,
    DATE(v.t1_timestamp) AS trip_date,
    v.upfront_fare,
    v.realized_fare,

    -- Join the imputed flag from our CTE
    i.is_imputed,

    -- KPI: Spread Percentage (robust against division by zero)
    CASE
        WHEN v.upfront_fare > 0 THEN v.realized_fare / v.upfront_fare
        ELSE NULL
    END AS spread_percentage,

    -- KPI: Phase Durations in Seconds
    (julianday(v.t1_timestamp) - julianday(v.t0_timestamp)) * 86400.0 AS duration_lfr_sec,      -- Looking For Ride
    (julianday(v.t2_timestamp) - julianday(v.t1_timestamp)) * 86400.0 AS duration_dtp_sec,      -- Driving To Pickup
    (julianday(v.t3_timestamp) - julianday(v.t2_timestamp)) * 86400.0 AS duration_wfp_sec,      -- Waiting For Passenger
    (julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) * 86400.0 AS duration_on_ride_sec,  -- On Ride

    -- KPI: Earnings Per Hour (EPH), robust against zero or null duration
    CASE
        WHEN (julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) > 0
        THEN v.upfront_fare / ((julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) * 24.0)
        ELSE NULL
    END AS eph_upfront,

    CASE
        WHEN (julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) > 0
        THEN v.realized_fare / ((julianday(v.t4_timestamp) - julianday(v.t3_timestamp)) * 24.0)
        ELSE NULL
    END AS eph_realized,

    -- Raw Timestamps for reference and further analysis
    v.t0_timestamp, v.t1_timestamp, v.t2_timestamp, v.t3_timestamp, v.t4_timestamp

FROM
    v_trip_funnel_wide v
LEFT JOIN
    ImputedFlag i ON v.trip_id_legacy = i.trip_id_legacy;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_trip_final_kpis;")
        conn.execute(view_sql)
    print("✅ SUCCESS: Final KPI view 'v_trip_final_kpis' created successfully.")

    # --- Verification Step (per new SOP) ---
    print("\n--- Displaying LAST 10 rows of the new KPI view for verification ---")
    with sqlite3.connect(db_file_path) as conn:
        verification_query = "SELECT * FROM v_trip_final_kpis ORDER BY t1_timestamp DESC LIMIT 10;"
        df_verify = pd.read_sql_query(verification_query, conn)
        display(df_verify)

except Exception as e:
    print(f"❌ ERROR creating view: {e}")

--- Architecting the final 'v_trip_final_kpis' analytical view ---
✅ SUCCESS: Final KPI view 'v_trip_final_kpis' created successfully.

--- Displaying LAST 10 rows of the new KPI view for verification ---


,trip_id_legacy,trip_date,upfront_fare,realized_fare,is_imputed,spread_percentage,duration_lfr_sec,duration_dtp_sec,duration_wfp_sec,duration_on_ride_sec,eph_upfront,eph_realized,t0_timestamp,t1_timestamp,t2_timestamp,t3_timestamp,t4_timestamp
0,251001-07,2025-10-01,135.37,125.34,0,0.925907,NaN,259.999998,48.000021,1992.000018,244.644576,226.518070,None,2025-10-01 09:13:54,2025-10-01 09:18:14,2025-10-01 09:19:02,2025-10-01 09:52:14
1,251001-06,2025-10-01,165.68,148.14,1,0.894133,111.000001,NaN,NaN,1758.000000,339.276451,303.358362,2025-10-01 08:34:44,2025-10-01 08:36:35,None,2025-10-01 08:44:11,2025-10-01 09:13:29
2,251001-05,2025-10-01,120.57,106.95,0,0.887037,273.000008,392.000006,22.000001,1791.999976,242.216521,214.854914,2025-10-01 07:53:16,2025-10-01 07:57:49,2025-10-01 08:04:21,2025-10-01 08:04:43,2025-10-01 08:34:35
3,251001-04,2025-10-01,133.47,120.01,0,0.899153,NaN,106.000029,101.999970,347.000009,1384.703135,1245.060487,None,2025-10-01 07:43:53,2025-10-01 07:45:39,2025-10-01 07:47:21,2025-10-01 07:53:08
4,251001-03,2025-10-01,182.17,156.07,0,0.856727,NaN,613.000014,69.999982,1893.999986,346.257658,296.648365,None,2025-10-01 07:00:04,2025-10-01 07:10:17,2025-10-01 07:11:27,2025-10-01 07:43:01
5,251001-02,2025-10-01,114.69,97.40,1,0.849246,176.000011,395.999984,70.000023,1053.999996,391.730552,332.675523,2025-10-01 06:30:56,2025-10-01 06:33:52,2025-10-01 06:40:28,2025-10-01 06:41:38,2025-10-01 06:59:12
6,251001-01,2025-10-01,151.30,142.47,0,0.941639,212.000017,390.000017,168.999968,1119.000006,486.756030,458.348523,2025-10-01 05:59:11,2025-10-01 06:02:43,2025-10-01 06:09:13,2025-10-01 06:12:02,2025-10-01 06:30:41
7,250930-10,2025-09-30,194.44,163.10,1,0.838819,266.000006,NaN,NaN,1545.000029,453.064069,380.038828,2025-09-30 15:40:18,2025-09-30 15:44:44,None,2025-09-30 15:49:01,2025-09-30 16:14:46
8,250930-09,2025-09-30,102.68,84.81,0,0.825964,NaN,NaN,NaN,1512.000006,244.476189,201.928571,None,2025-09-30 15:12:14,None,2025-09-30 15:14:42,2025-09-30 15:39:54
9,250930-08,2025-09-30,108.91,102.68,0,0.942797,351.999982,501.000018,85.999976,818.000029,479.310496,451.892405,2025-09-30 14:41:41,2025-09-30 14:47:33,2025-09-30 14:55:54,2025-09-30 14:57:20,2025-09-30 15:10:58


In [ ]:
# =================================================================================================
# === SCRIPT 1: CREATE THE DEFINITIVE "MONEY TABLE" VIEW (v2.2 - Corrected EPH Logic) ===
# =================================================================================================

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("WARNING: 'db_file_path' not found in global scope. Using default path.")

print("--- Architecting the definitive 'v_trip_final_kpis' view (v2.2) ---")

# This query contains the corrected, robust logic and will be saved in the database.
# All durations are stored as raw SECONDS for analytical integrity.
view_sql = """
CREATE VIEW v_trip_final_kpis AS
WITH
ImputedFlag AS (
    SELECT
        trip_id_legacy,
        MAX(is_imputed) AS is_imputed
    FROM
        trip_events
    GROUP BY
        trip_id_legacy
),
Durations AS (
    SELECT
        trip_id_legacy,
        (julianday(t1_timestamp) - julianday(t0_timestamp)) * 86400.0 AS duration_lfr_sec,
        (julianday(COALESCE(t2_timestamp, t3_timestamp, t4_timestamp)) - julianday(t1_timestamp)) * 86400.0 AS duration_dtp_sec,
        (julianday(t3_timestamp) - julianday(t2_timestamp)) * 86400.0 AS duration_wfp_sec,
        (julianday(t4_timestamp) - julianday(t3_timestamp)) * 86400.0 AS duration_on_ride_sec,
        (julianday(MAX(
            IFNULL(t0_timestamp, '0000-01-01'), IFNULL(t1_timestamp, '0000-01-01'),
            IFNULL(t2_timestamp, '0000-01-01'), IFNULL(t3_timestamp, '0000-01-01'),
            IFNULL(t4_timestamp, '0000-01-01')
        )) -
         julianday(MIN(
            IFNULL(t0_timestamp, '9999-12-31'), IFNULL(t1_timestamp, '9999-12-31'),
            IFNULL(t2_timestamp, '9999-12-31'), IFNULL(t3_timestamp, '9999-12-31'),
            IFNULL(t4_timestamp, '9999-12-31')
        ))) * 86400.0 AS total_engagement_duration_sec
    FROM
        v_trip_funnel_wide
    GROUP BY
        trip_id_legacy
)
SELECT
    v.trip_id_legacy,
    DATE(v.t1_timestamp) AS trip_date,
    d.duration_lfr_sec,
    d.duration_dtp_sec,
    d.duration_wfp_sec,
    d.duration_on_ride_sec,
    (COALESCE(d.duration_lfr_sec, 0) +
     COALESCE(d.duration_dtp_sec, 0) +
     COALESCE(d.duration_wfp_sec, 0) +
     COALESCE(d.duration_on_ride_sec, 0)) AS total_duration_sec,
    v.upfront_fare,
    v.realized_fare,
    CASE WHEN v.upfront_fare > 0 THEN v.realized_fare / v.upfront_fare ELSE NULL END AS spread_percentage,
    CASE WHEN d.duration_on_ride_sec > 0 THEN v.realized_fare / (d.duration_on_ride_sec / 3600.0) ELSE NULL END AS eph_on_ride,
    CASE WHEN d.total_engagement_duration_sec > 0 THEN v.realized_fare / (d.total_engagement_duration_sec / 3600.0) ELSE NULL END AS eph_total_time,
    i.is_imputed
FROM
    v_trip_funnel_wide v
LEFT JOIN
    Durations d ON v.trip_id_legacy = d.trip_id_legacy
LEFT JOIN
    ImputedFlag i ON v.trip_id_legacy = i.trip_id_legacy;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        conn.execute("DROP VIEW IF EXISTS v_trip_final_kpis;")
        conn.execute(view_sql)
    print("✅ SUCCESS: Final KPI view 'v_trip_final_kpis' (v2.2) created successfully.")

    # --- Simple Technical Verification ---
    print("\n--- Displaying raw data from the last 10 trips for technical verification ---")
    with sqlite3.connect(db_file_path) as conn:
        verification_query = "SELECT * FROM v_trip_final_kpis ORDER BY trip_date DESC, trip_id_legacy DESC LIMIT 10;"
        df_verify = pd.read_sql_query(verification_query, conn)
        display(df_verify)

except Exception as e:
    print(f"❌ ERROR creating view: {e}")

--- Architecting the definitive 'v_trip_final_kpis' view (v2.2) ---
✅ SUCCESS: Final KPI view 'v_trip_final_kpis' (v2.2) created successfully.

--- Displaying raw data from the last 10 trips for technical verification ---


,trip_id_legacy,trip_date,duration_lfr_sec,duration_dtp_sec,duration_wfp_sec,duration_on_ride_sec,total_duration_sec,upfront_fare,realized_fare,spread_percentage,eph_on_ride,eph_total_time,is_imputed
0,251001-07,2025-10-01,NaN,259.999998,48.000021,1992.000018,2300.000037,135.37,125.34,0.925907,226.518070,196.184345,0
1,251001-06,2025-10-01,111.000001,455.999981,NaN,1758.000000,2324.999982,165.68,148.14,0.894133,303.358362,229.378066,1
2,251001-05,2025-10-01,273.000008,392.000006,22.000001,1791.999976,2478.999992,120.57,106.95,0.887037,214.854914,155.312627,0
3,251001-04,2025-10-01,NaN,106.000029,101.999970,347.000009,555.000007,133.47,120.01,0.899153,1245.060487,778.443233,0
4,251001-03,2025-10-01,NaN,613.000014,69.999982,1893.999986,2576.999983,182.17,156.07,0.856727,296.648365,218.025613,0
5,251001-02,2025-10-01,176.000011,395.999984,70.000023,1053.999996,1696.000014,114.69,97.40,0.849246,332.675523,206.745281,1
6,251001-01,2025-10-01,212.000017,390.000017,168.999968,1119.000006,1890.000008,151.30,142.47,0.941639,458.348523,271.371427,0
7,250930-10,2025-09-30,266.000006,256.999974,NaN,1545.000029,2068.000008,194.44,163.10,0.838819,380.038828,283.926498,1
8,250930-09,2025-09-30,NaN,148.000002,NaN,1512.000006,1660.000008,102.68,84.81,0.825964,201.928571,183.925300,0
9,250930-08,2025-09-30,351.999982,501.000018,85.999976,818.000029,1757.000005,108.91,102.68,0.942797,451.892405,210.385884,0


In [ ]:
# ==============================================================================
# === SCRIPT 2: FINAL AUDIT CELL (Formatted "Money Table" Inspection) ===
# ==============================================================================

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("WARNING: 'db_file_path' not found in global scope. Using default path.")

print("--- Running a formatted audit of the 'v_trip_final_kpis' view ---")

# --- [1] Data Extraction ---
try:
    with sqlite3.connect(db_file_path) as conn:
        query = "SELECT * FROM v_trip_final_kpis ORDER BY trip_date DESC, trip_id_legacy DESC LIMIT 10;"
        df_audit = pd.read_sql_query(query, conn)
    print("✅ Data successfully extracted from the view.")

    # --- [2] Data Transformation & Formatting (in Pandas) ---
    print("Applying presentation-layer formatting...")

    def format_seconds_to_hms(seconds):
        if pd.isna(seconds) or seconds < 0: return None
        seconds = int(seconds)
        hours, remainder = divmod(seconds, 3600)
        minutes, sec = divmod(remainder, 60)
        return f"{hours:02d}:{minutes:02d}:{sec:02d}"

    df_formatted = df_audit.copy()
    duration_cols = ['duration_lfr_sec', 'duration_dtp_sec', 'duration_wfp_sec', 'duration_on_ride_sec', 'total_duration_sec']
    for col in duration_cols:
        if col in df_formatted.columns:
            df_formatted[col] = df_formatted[col].apply(format_seconds_to_hms)

    # --- [3] Final Presentation using Pandas Styler ---
    styler_format = {
        'eph_on_ride': '$ {:,.2f}', 'eph_total_time': '$ {:,.2f}',
        'spread_percentage': '{:.2%}', 'upfront_fare': '{:,.2f}', 'realized_fare': '{:,.2f}'
    }

    print("\n--- Displaying the LAST 10 formatted rows from the 'Money Table' ---")
    display(df_formatted.style.format(styler_format, na_rep='-').hide(axis="index"))

except Exception as e:
    print(f"❌ ERROR during formatted audit: {e}")

--- Running a formatted audit of the 'v_trip_final_kpis' view ---
✅ Data successfully extracted from the view.
Applying presentation-layer formatting...

--- Displaying the LAST 10 formatted rows from the 'Money Table' ---


trip_id_legacy,trip_date,duration_lfr_sec,duration_dtp_sec,duration_wfp_sec,duration_on_ride_sec,total_duration_sec,upfront_fare,realized_fare,spread_percentage,eph_on_ride,eph_total_time,is_imputed
251001-07,2025-10-01,-,00:04:19,00:00:48,00:33:12,00:38:20,135.37,125.34,92.59%,$ 226.52,$ 196.18,0
251001-06,2025-10-01,00:01:51,00:07:35,-,00:29:17,00:38:44,165.68,148.14,89.41%,$ 303.36,$ 229.38,1
251001-05,2025-10-01,00:04:33,00:06:32,00:00:22,00:29:51,00:41:18,120.57,106.95,88.70%,$ 214.85,$ 155.31,0
251001-04,2025-10-01,-,00:01:46,00:01:41,00:05:47,00:09:15,133.47,120.01,89.92%,"$ 1,245.06",$ 778.44,0
251001-03,2025-10-01,-,00:10:13,00:01:09,00:31:33,00:42:56,182.17,156.07,85.67%,$ 296.65,$ 218.03,0
251001-02,2025-10-01,00:02:56,00:06:35,00:01:10,00:17:33,00:28:16,114.69,97.40,84.92%,$ 332.68,$ 206.75,1
251001-01,2025-10-01,00:03:32,00:06:30,00:02:48,00:18:39,00:31:30,151.30,142.47,94.16%,$ 458.35,$ 271.37,0
250930-10,2025-09-30,00:04:26,00:04:16,-,00:25:45,00:34:28,194.44,163.10,83.88%,$ 380.04,$ 283.93,1
250930-09,2025-09-30,-,00:02:28,-,00:25:12,00:27:40,102.68,84.81,82.60%,$ 201.93,$ 183.93,0
250930-08,2025-09-30,00:05:51,00:08:21,00:01:25,00:13:38,00:29:17,108.91,102.68,94.28%,$ 451.89,$ 210.39,0


In [ ]:
# ================================================================================================
# === DIAGNOSTIC AUDIT (v2): Identify "Completed but Unstarted" Data Integrity Violations ===
# ================================================================================================

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("WARNING: 'db_file_path' not found in global scope. Using default path.")

print("--- Auditing for trips that have a completion time (T4) but are missing a start time (T3) ---")

# CORRECTED: The query now targets the 'v_trip_funnel_wide' view, which contains the raw timestamps.
# The 'is_imputed' column is removed as it does not exist in this foundational view.
query_sql = """
SELECT
    trip_id_legacy,
    t1_timestamp,
    t2_timestamp,
    t3_timestamp, -- Will be NULL
    t4_timestamp, -- Will have a value
    realized_fare
FROM
    v_trip_funnel_wide -- CORRECTED SOURCE
WHERE
    t4_timestamp IS NOT NULL  -- The trip MUST have been completed
    AND t3_timestamp IS NULL; -- But the trip start time is MISSING
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_anomaly_trips = pd.read_sql_query(query_sql, conn)

    print(f"\n✅ AUDIT COMPLETE: Found {len(df_anomaly_trips)} trips that are COMPLETED but are missing a START time.")

    if not df_anomaly_trips.empty:
        print("Displaying the list of these trips requiring manual imputation review:")
        display(df_anomaly_trips)
    else:
        print("No such data integrity violations were found. All completed trips have a start time.")

except Exception as e:
    print(f"❌ ERROR during diagnostic audit: {e}")

--- Auditing for trips that have a completion time (T4) but are missing a start time (T3) ---

✅ AUDIT COMPLETE: Found 0 trips that are COMPLETED but are missing a START time.
No such data integrity violations were found. All completed trips have a start time.


In [ ]:
# ===================================================================================================
# === CELL 11 (v7.2 - The Final Cascade): ETL FOR 'trip_events' TABLE ===
# ===================================================================================================
print("\n--- PHASE 3: ENRICHING & LOADING trip_events ---")

import pandas as pd
import sqlite3
import os
import gspread
from google.colab import auth
from google.auth import default

# ==============================================================================
# --- 1. EXTRACT & PREPARE ---
# ==============================================================================
print("Executing self-contained extraction of all sources...")
try:
    # Authenticate and initialize GSpread client
    try: gc
    except NameError:
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)

    # Source 1: 'trip_events' data from GTS-4 GSheet
    GTS_SHEET_NAME = "GTS-4"
    TRIP_EVENTS_TAB_NAME = "trip_events"
    workbook_gts = gc.open(GTS_SHEET_NAME)
    sheet_gts = workbook_gts.worksheet(TRIP_EVENTS_TAB_NAME)
    df_raw_events = pd.DataFrame(sheet_gts.get_all_records())
    df_raw_events.columns = df_raw_events.columns.str.lower().str.replace(' ', '_')
    df_raw_events.reset_index(inplace=True); df_raw_events.rename(columns={'index': 'event_id'}, inplace=True); df_raw_events['event_id'] += 1
    print(f"✅ Extracted and prepared {len(df_raw_events)} raw events from GSheet.")

    # Source 2 & 3: 'offers' and 'event_types' tables from opus.db
    with sqlite3.connect(db_file_path) as conn:
        offers_query = "SELECT offer_id, session_fk, offer_timestamp, upfront_fare FROM offers WHERE offer_action_fk = 1;"
        df_offers = pd.read_sql_query(offers_query, conn)
        df_offers['offer_timestamp'] = pd.to_datetime(df_offers['offer_timestamp'], format='%Y:%m:%d %H:%M:%S', errors='coerce')
        df_event_types = pd.read_sql_query("SELECT * FROM event_types;", conn)
    print(f"✅ Extracted {len(df_offers)} accepted offers and {len(df_event_types)} event types from database.")
except Exception as e:
    print(f"❌ ERROR: Failed to extract source data. Reason: {e}")
    assert False, "Halting execution."

# ==============================================================================
# --- 2. TRANSFORM ---
# ==============================================================================
print("\n--- [T] TRANSFORMING and enriching 'trip_events' data ---")

# --- A. PRE-CLEANING & FK ENRICHMENT (event_types) ---
df_staged_events = pd.merge(df_raw_events, df_event_types, left_on='event_types_id_fk', right_on='description', how='left')
df_staged_events.drop(columns=['event_types_id_fk', 'description', 'event_code', 'event_name'], inplace=True, errors='ignore')
df_staged_events.rename(columns={'event_type_id': 'event_types_id_fk'}, inplace=True)
df_staged_events.replace({'': None, 'N/A': None}, inplace=True)
df_staged_events['event_timestamp'] = pd.to_datetime(df_staged_events['event_timestamp'], errors='coerce')
df_staged_events['upfront_fare'] = pd.to_numeric(df_staged_events['upfront_fare'].astype(str).str.replace(r'[MX$,]', '', regex=True), errors='coerce')
df_staged_events.dropna(subset=['event_id', 'event_timestamp'], inplace=True)
df_staged_events['event_id'] = df_staged_events['event_id'].astype(int)
print("✅ Preliminary data cleaning and FK enrichment complete.")

# --- B. THE LINKING CASCADE ---
df_t1_events = df_staged_events[df_staged_events['event_types_id_fk'] == 2].copy()
df_offers_timed = df_offers[df_offers['offer_timestamp'].notna()].copy()
df_offers_orphan = df_offers[df_offers['offer_timestamp'].isna()].copy()

df_t1_events['day'] = df_t1_events['event_timestamp'].dt.strftime('%Y-%m-%d')
df_offers_timed['day'] = df_offers_timed['offer_timestamp'].dt.strftime('%Y-%m-%d')

print("\nExecuting Tier 1 & 2: Matching against TIMED offers...")
perfect_matches = pd.merge(df_t1_events, df_offers_timed, on=['day', 'upfront_fare'], how='inner')
if not perfect_matches.empty: perfect_matches['is_fuzzy_match'] = False
print(f"✅ Found {len(perfect_matches)} perfect timed matches.")

perfectly_matched_ids = perfect_matches['event_id'].unique() if not perfect_matches.empty else []
events_for_fuzzy_match = df_t1_events[~df_t1_events['event_id'].isin(perfectly_matched_ids)].copy()
if not events_for_fuzzy_match.empty:
    events_for_fuzzy_match['rounded_fare'] = events_for_fuzzy_match['upfront_fare'].fillna(0).astype(int)
    df_offers_timed['rounded_fare'] = df_offers_timed['upfront_fare'].fillna(0).astype(int)
    fuzzy_matches = pd.merge(events_for_fuzzy_match, df_offers_timed, on=['day', 'rounded_fare'], how='inner')
    if not fuzzy_matches.empty: fuzzy_matches['is_fuzzy_match'] = True
    print(f"✅ Found {len(fuzzy_matches)} fuzzy timed matches.")
else:
    fuzzy_matches = pd.DataFrame()

# --- TIER 3: MATCHING AGAINST ORPHAN OFFERS ---
timed_matched_ids = pd.concat([perfect_matches, fuzzy_matches])['event_id'].unique()
events_for_orphan_match = df_t1_events[~df_t1_events['event_id'].isin(timed_matched_ids)]

print(f"\nExecuting Tier 3: Matching {len(events_for_orphan_match)} remaining events against ORPHAN offers...")
orphan_matches = pd.merge(events_for_orphan_match, df_offers_orphan, on='upfront_fare', how='inner')
if not orphan_matches.empty:
    orphan_matches.drop_duplicates(subset='event_id', keep='first', inplace=True)
    orphan_matches['is_imputed_link'] = True
    orphan_matches['is_fuzzy_match'] = False
print(f"✅ Found {len(orphan_matches)} imputed links (orphan matches).")

# --- C. FINAL RECONCILIATION & ANOMALY REPORT ---
all_matches = pd.concat([perfect_matches, fuzzy_matches, orphan_matches], ignore_index=True)
final_matched_ids = all_matches['event_id'].unique()
final_unmatched = df_t1_events[~df_t1_events['event_id'].isin(final_matched_ids)]

if not final_unmatched.empty:
    print(f"\n🛑 ANOMALY REPORT: Found {len(final_unmatched)} T1 events that are TRUE anomalies.")
    display(final_unmatched[['event_id', 'trip_id_legacy', 'event_timestamp', 'upfront_fare']])
else:
    print("\n✅ All T1 events were successfully matched.")

# ==============================================================================
# --- 3. FINAL MERGE & LOAD ---
# ==============================================================================
print("\n--- [L] Merging links and loading to database ---")
if not all_matches.empty:
    df_linked_final = pd.merge(df_staged_events, all_matches[['event_id', 'offer_id', 'session_fk', 'is_fuzzy_match', 'is_imputed_link']], on='event_id', how='left')
else:
    df_linked_final = df_staged_events.copy()
    df_linked_final['offer_id'], df_linked_final['session_fk'], df_linked_final['is_imputed_link'], df_linked_final['is_fuzzy_match'] = None, None, False, False
df_linked_final.rename(columns={'offer_id': 'offer_id_fk', 'session_fk': 'offers_session_fk'}, inplace=True)
df_linked_final['is_imputed_link'] = df_linked_final['is_imputed_link'].fillna(False).astype(bool)
df_linked_final['is_fuzzy_match'] = df_linked_final['is_fuzzy_match'].fillna(False).astype(bool)
final_columns = [
    'event_id', 'event_timestamp', 'event_lat', 'event_lon', 'event_address',
    'upfront_fare', 'realized_fare', 'source', 'is_imputed', 'comment',
    'trip_id_legacy', 'event_types_id_fk', 'offer_id_fk', 'offers_session_fk', 'is_fuzzy_match', 'is_imputed_link'
]
df_to_load = df_linked_final[[col for col in final_columns if col in df_linked_final.columns]]

try:
    with sqlite3.connect(db_file_path) as conn:
        df_to_load.to_sql('trip_events', conn, if_exists='replace', index=False)
    print(f"✅ SUCCESS: Loaded {len(df_to_load)} records into 'trip_events'.")
    print("\n--- PHASE 3 COMPLETE ---")
except Exception as e:
    print(f"❌ ERROR: Failed to load 'trip_events' table. Reason: {e}")


--- PHASE 3: ENRICHING & LOADING trip_events ---
Executing self-contained extraction of all sources...
✅ Extracted and prepared 1031 raw events from GSheet.
✅ Extracted 346 accepted offers and 5 event types from database.

--- [T] TRANSFORMING and enriching 'trip_events' data ---
✅ Preliminary data cleaning and FK enrichment complete.

Executing Tier 1 & 2: Matching against TIMED offers...
✅ Found 0 perfect timed matches.
✅ Found 0 fuzzy timed matches.

Executing Tier 3: Matching 259 remaining events against ORPHAN offers...
✅ Found 259 imputed links (orphan matches).

✅ All T1 events were successfully matched.

--- [L] Merging links and loading to database ---
✅ SUCCESS: Loaded 1031 records into 'trip_events'.

--- PHASE 3 COMPLETE ---


In [ ]:
# ==============================================================================
# CELL 11.1: Golden Propagation Patch (TD-002) - v2.0 Self-Contained
# ==============================================================================
# Author: Pienza Ex Machina
# Purpose: This cell reads the 'trip_events' table, applies the "Golden
#          Propagation" doctrine in memory, and overwrites the table with the
#          fully-linked version. This version is self-contained and establishes
#          its own database connection.
# ==============================================================================

import pandas as pd
from sqlalchemy import create_engine
import os # Import os to check for file existence

print("Initiating Golden Propagation Patch v2.0...")

# --- SETUP & CONNECTION ---
# This block makes the cell self-sufficient.
try:
    db_path = '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("✅ Database engine created successfully.")
except Exception as e:
    print(f"🔴 CRITICAL ERROR: Could not create database engine. Halting execution. Details: {e}")
    raise

# --- READ-MODIFY-WRITE PATTERN ---

# STEP 1: READ the current state of the trip_events table from the database
try:
    events_to_patch_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"✅ Successfully read {len(events_to_patch_df)} events to be patched.")
except Exception as e:
    print(f"🔴 ERROR: Could not read from trip_events table. Ensure CELL 11 has been run successfully. Details: {e}")
    raise

# STEP 2: MODIFY the DataFrame in memory to apply the propagation doctrine
print("⏳ Applying propagation logic...")
events_to_patch_df = events_to_patch_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
events_to_patch_df['offer_id_fk'] = events_to_patch_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("✅ Propagation logic applied in memory.")

# In-line Unit Test: Assert that propagation was successful for a known trip before writing.
known_trip_id = 'GTS-4-20251001-073003-B'
if known_trip_id in events_to_patch_df['trip_id_legacy'].values:
    propagated_ids = events_to_patch_df[events_to_patch_df['trip_id_legacy'] == known_trip_id]['offer_id_fk']
    assert propagated_ids.notna().all(), f"Validation failed: Nulls still exist for trip {known_trip_id}"
    assert propagated_ids.nunique() == 1, f"Validation failed: Multiple different offer_ids found for trip {known_trip_id}"
    print(f"✅ Validation successful for test case trip: {known_trip_id}")
else:
    print(f"⚠️ Warning: Test case trip ID '{known_trip_id}' not found. Skipping validation.")


# STEP 3: WRITE the patched DataFrame back to the database, overwriting the old version.
print("⏳ Writing patched data back to opus.db...")
events_to_patch_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
print("✅ Patched data successfully written to database.")

print("\n--- MISSION COMPLETE ---")
print("TD-002 Resolved. The 'trip_events' table has been successfully patched.")

Initiating Golden Propagation Patch v2.0...
✅ Database engine created successfully.
✅ Successfully read 1031 events to be patched.
⏳ Applying propagation logic...
✅ Propagation logic applied in memory.
⚠️ Warning: Test case trip ID 'GTS-4-20251001-073003-B' not found. Skipping validation.
⏳ Writing patched data back to opus.db...
✅ Patched data successfully written to database.

--- MISSION COMPLETE ---
TD-002 Resolved. The 'trip_events' table has been successfully patched.


In [ ]:
# ==============================================================================
# === CELL 12 (NEW): The Post-ETL "Golden Link" Patch ===
# ==============================================================================
print("--- Starting the Post-ETL 'Golden Link' Patch for trip_events ---")

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')

# --- The Override Rulebook ---
# Format: { event_id_to_fix: 'correct_offer_id' }
override_rules = {
    55: 'OF00318'
    # Add any other future corrections here.
}

print(f"Found {len(override_rules)} override rule(s) to apply.")

try:
    with sqlite3.connect(db_file_path) as conn:
        cursor = conn.cursor()

        # We need to get the session_fk for the corrected offer
        df_offers = pd.read_sql_query("SELECT offer_id, session_fk FROM offers", conn)

        for event_id, correct_offer_id in override_rules.items():
            # Find the correct session_fk from our offers data
            try:
                correct_session_fk = df_offers.loc[df_offers['offer_id'] == correct_offer_id, 'session_fk'].iloc[0]
            except IndexError:
                print(f"⚠️ WARNING: Could not find correct_offer_id '{correct_offer_id}' in the offers table. Skipping override for event {event_id}.")
                continue

            # The surgical UPDATE statement
            update_sql = """
            UPDATE trip_events
            SET
                offer_id_fk = ?,
                offers_session_fk = ?
            WHERE
                event_id = ?;
            """

            cursor.execute(update_sql, (correct_offer_id, correct_session_fk, event_id))
            print(f"  - Override Applied: Event ID {event_id} is now linked to Offer ID {correct_offer_id} (Session: {correct_session_fk}).")

    print("\n✅ Patching complete.")

    # --- Final Verification Step ---
    print("\n--- Verifying the fix for Event ID 55 ---")
    with sqlite3.connect(db_file_path) as conn:
        verify_query = "SELECT event_id, offer_id_fk, offers_session_fk FROM trip_events WHERE event_id = 55;"
        df_verify = pd.read_sql_query(verify_query, conn)
        display(df_verify)

except Exception as e:
    print(f"❌ ERROR during patching process: {e}")

--- Starting the Post-ETL 'Golden Link' Patch for trip_events ---
Found 1 override rule(s) to apply.
  - Override Applied: Event ID 55 is now linked to Offer ID OF00318 (Session: SID0006).

✅ Patching complete.

--- Verifying the fix for Event ID 55 ---


,event_id,offer_id_fk,offers_session_fk
0,55,OF00318,SID0006


In [ ]:
# ==============================================================================
# === DIAGNOSTIC CELL: The Pre-Join Audit for CELL 11 ===
# ==============================================================================
print("--- Performing a pre-join audit to diagnose the total join failure ---")

import pandas as pd
import sqlite3
import os

# --- [1] Replicate the Full Extraction Logic ---
print("Extracting all necessary source data...")
try:
    with sqlite3.connect(db_file_path) as conn:
        df_staged_events_raw = pd.read_sql_query("SELECT * FROM trip_events;", conn) # Assuming CELL 9&10 created this
        df_staged_events_raw['event_timestamp'] = pd.to_datetime(df_staged_events_raw['event_timestamp'])

        offers_query = "SELECT offer_id, session_fk, offer_timestamp, upfront_fare FROM offers WHERE offer_action_fk = 1;"
        df_offers_raw = pd.read_sql_query(offers_query, conn)
        df_offers_raw['offer_timestamp'] = pd.to_datetime(df_offers_raw['offer_timestamp'], errors='coerce')

except Exception as e:
    print(f"❌ ERROR: Failed to extract source data. Reason: {e}")
    assert False, "Halting execution."

# --- [2] Replicate the Preparation and Key Creation Logic ---
print("\n--- Preparing DataFrames for the join ---")
df_staged_events = df_staged_events_raw.copy()
df_offers = df_offers_raw.copy()

df_staged_events['upfront_fare'] = pd.to_numeric(df_staged_events['upfront_fare'].astype(str).str.replace(r'[MX$,]', '', regex=True), errors='coerce')
df_t1_events = df_staged_events[df_staged_events['event_types_id_fk'] == 2].copy()

df_t1_events['day'] = df_t1_events['event_timestamp'].dt.strftime('%Y-%m-%d')
df_offers['day'] = df_offers['offer_timestamp'].dt.strftime('%Y-%m-%d')

# --- [3] THE DEFINITIVE AUDIT ---
print("\n--- AUDIT: Data types and sample values of JOIN KEYS ---")

print("\n--- `df_t1_events` (Left Side of Join) ---")
print("Data Types:")
print(df_t1_events[['day', 'upfront_fare']].info())
print("\nSample Data:")
display(df_t1_events[['day', 'upfront_fare']].head())

print("\n--- `df_offers` (Right Side of Join) ---")
print("Data Types:")
print(df_offers[['day', 'upfront_fare']].info())
print("\nSample Data:")
display(df_offers[['day', 'upfront_fare']].head())

--- Performing a pre-join audit to diagnose the total join failure ---
Extracting all necessary source data...

--- Preparing DataFrames for the join ---

--- AUDIT: Data types and sample values of JOIN KEYS ---

--- `df_t1_events` (Left Side of Join) ---
Data Types:
<class 'pandas.core.frame.DataFrame'>
Index: 259 entries, 0 to 1027
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   day           259 non-null    object 
 1   upfront_fare  259 non-null    float64
dtypes: float64(1), object(1)
memory usage: 6.1+ KB
None

Sample Data:


,day,upfront_fare
0,2025-08-22,136.53
3,2025-08-22,162.00
6,2025-08-22,64.31
9,2025-08-22,146.00
12,2025-08-22,60.00



--- `df_offers` (Right Side of Join) ---
Data Types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 346 entries, 0 to 345
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   day           346 non-null    object 
 1   upfront_fare  346 non-null    float64
dtypes: float64(1), object(1)
memory usage: 5.5+ KB
None

Sample Data:


,day,upfront_fare
0,2025-08-22,136.53
1,2025-08-22,162.00
2,2025-08-22,64.31
3,2025-08-22,146.00
4,2025-08-22,60.00


In [ ]:
# ==============================================================================
# === CELL 12 (NEW): The Final "Side-by-Side" Reconciliation Audit ===
# ==============================================================================
print("--- Performing the final 'Side-by-Side' Reconciliation Audit ---")

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("INFO: 'db_file_path' not found. Using default path.")

# This query joins the final trip_events table back to the offers table
# to create a direct, side-by-side comparison for validation.
query_sql = """
SELECT
    te.trip_id_legacy,
    te.offer_id_fk,
    te.upfront_fare AS event_upfront_fare,
    o.upfront_fare AS linked_offer_upfront_fare,
    -- Calculate the difference to easily spot mismatches
    (te.upfront_fare - o.upfront_fare) AS fare_discrepancy,
    te.is_imputed_link
FROM
    trip_events te
-- Use a LEFT JOIN to ensure we see all trip events, even if a link failed
LEFT JOIN
    offers o ON te.offer_id_fk = o.offer_id
WHERE
    te.event_types_id_fk = 2 -- We only care about the T1 'Accepted' events that were linked
ORDER BY
    te.event_timestamp ASC;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_reconciliation = pd.read_sql_query(query_sql, conn)

    print("\n✅ Reconciliation query executed successfully.")
    print("Displaying a side-by-side comparison for all linked trips:")

    # Configure pandas to display all rows
    pd.set_option('display.max_rows', None)

    display(df_reconciliation)

    # --- Final Automated Check ---
    # Find any rows where the link was made but the fares don't match (within a tolerance)
    mismatch_count = df_reconciliation[df_reconciliation['fare_discrepancy'].abs() > 0.01].shape[0]

    print("\n--- FINAL VERDICT ---")
    if mismatch_count > 0:
        print(f"🛑 WARNING: Found {mismatch_count} linked records with a significant fare discrepancy. Manual review required.")
    else:
        print("✅ SUCCESS: All linked records have a perfect fare match. The linkage is 100% validated.")

except Exception as e:
    print(f"❌ ERROR during reconciliation audit: {e}")

--- Performing the final 'Side-by-Side' Reconciliation Audit ---

✅ Reconciliation query executed successfully.
Displaying a side-by-side comparison for all linked trips:


,trip_id_legacy,offer_id_fk,event_upfront_fare,linked_offer_upfront_fare,fare_discrepancy,is_imputed_link
0,250822-01,OF00003,136.53,136.53,0.0,1
1,250822-02,OF00012,162.00,162.00,0.0,1
2,250822-03,OF00032,64.31,64.31,0.0,1
3,250822-04,OF00034,146.00,146.00,0.0,1
4,250822-05,OF00035,60.00,60.00,0.0,1
5,250822-06,OF00101,108.67,108.67,0.0,1
6,250822-07,OF00113,170.00,170.00,0.0,1
7,250822-08,OF00130,173.44,173.44,0.0,1
8,250822-09,OF00167,89.00,89.00,0.0,1
9,250822-10,OF00174,202.00,202.00,0.0,1



--- FINAL VERDICT ---
✅ SUCCESS: All linked records have a perfect fare match. The linkage is 100% validated.


In [ ]:
# ==============================================================================
# === EMERGENCY AUDIT: Re-establishing Ground Truth for 'raw_offers_ocr' ===
# ==============================================================================
import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("WARNING: 'db_file_path' not found in global scope. Using default path.")

print("--- Auditing the Canonical Schema of the 'raw_offers_ocr' table ---")

query_sql = "PRAGMA table_info(raw_offers_ocr);"

try:
    with sqlite3.connect(f'file:{db_file_path}?mode=ro', uri=True) as conn:
        df_schema = pd.read_sql_query(query_sql, conn)

    print("\n✅ SUCCESS: Schema audit complete.")
    print("This is the definitive, canonical list of columns for 'raw_offers_ocr':")
    display(df_schema)

except Exception as e:
    print(f"❌ ERROR: Audit query failed. Reason: {e}")

--- Auditing the Canonical Schema of the 'raw_offers_ocr' table ---

✅ SUCCESS: Schema audit complete.
This is the definitive, canonical list of columns for 'raw_offers_ocr':


,cid,name,type,notnull,dflt_value,pk
0,0,ocr_id,TEXT,0,None,0
1,1,image_filename,TEXT,0,None,0
2,2,time_taken,TEXT,0,None,0
3,3,ride_type,TEXT,0,None,0
4,4,upfront_fare,TEXT,0,None,0
5,5,pickup_details,TEXT,0,None,0
6,6,pickup_address,TEXT,0,None,0
7,7,trip_details,TEXT,0,None,0
8,8,dropoff_address,TEXT,0,None,0
9,9,rider_rating,TEXT,0,None,0


In [ ]:
# =========================================================================================
# === TASK 3 (v2.0 - Final Validation): VERIFY LINK between 'raw_offers_ocr' and 'offers' ===
# =========================================================================================

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("INFO: 'db_file_path' not found in global scope. Using default path.")

print("\n--- Verifying the integrity of the ocr_fk link between 'offers' and 'raw_offers_ocr' ---")

# The query has been corrected to use the true PK of 'raw_offers_ocr', which is 'ocr_id'
query_sql = """
SELECT
    CASE
        WHEN ocr.ocr_id IS NULL THEN 'Orphaned (Link Failed)'
        ELSE 'Linked Successfully'
    END AS link_status,
    COUNT(o.offer_id) AS number_of_offers
FROM
    offers o
LEFT JOIN
    raw_offers_ocr ocr ON o.ocr_fk = ocr.ocr_id
GROUP BY
    link_status;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_link_status = pd.read_sql_query(query_sql, conn)

    print("\n✅ SUCCESS: Audit query executed.")
    print("Displaying link status count:")
    display(df_link_status)

    # Provide a clear, actionable summary
    if 'Orphaned (Link Failed)' in df_link_status['link_status'].values:
        print("\n🛑 FINDING: One or more offers could not be linked back to their OCR source. Investigation required.")
    else:
        print("\n✅ FINDING: 100% of offers are successfully linked to their raw OCR source. Data provenance is intact.")

except Exception as e:
    print(f"❌ ERROR during audit: {e}")


--- Verifying the integrity of the ocr_fk link between 'offers' and 'raw_offers_ocr' ---

✅ SUCCESS: Audit query executed.
Displaying link status count:


,link_status,number_of_offers
0,Orphaned (Link Failed),4765



🛑 FINDING: One or more offers could not be linked back to their OCR source. Investigation required.


In [ ]:
# ==============================================================================
# === EMERGENCY AUDIT: Re-establishing Ground Truth for the 'offers' table ===
# ==============================================================================
import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("INFO: 'db_file_path' not found in global scope. Using default path.")

print("\n--- Auditing the Canonical Schema of the 'offers' table ---")

query_sql = "PRAGMA table_info(offers);"

try:
    with sqlite3.connect(f'file:{db_file_path}?mode=ro', uri=True) as conn:
        df_schema = pd.read_sql_query(query_sql, conn)

    print("\n✅ SUCCESS: Schema audit complete.")
    print("This is the definitive, canonical list of columns for the 'offers' table:")
    display(df_schema)

except Exception as e:
    print(f"❌ ERROR: Audit query failed. Reason: {e}")


--- Auditing the Canonical Schema of the 'offers' table ---

✅ SUCCESS: Schema audit complete.
This is the definitive, canonical list of columns for the 'offers' table:


,cid,name,type,notnull,dflt_value,pk
0,0,offer_id,TEXT,0,None,0
1,1,session_fk,TEXT,0,None,0
2,2,ocr_fk,TEXT,0,None,0
3,3,image_content_hash,TEXT,0,None,0
4,4,offer_timestamp,TEXT,0,None,0
5,5,upfront_fare,REAL,0,None,0
6,6,time_to_pickup_sec,REAL,0,None,0
7,7,dist_to_pickup_km,REAL,0,None,0
8,8,est_trip_time_sec,REAL,0,None,0
9,9,est_trip_dist_km,REAL,0,None,0


In [ ]:
# ==============================================================================
# === EMERGENCY AUDIT: Finalizing Ground Truth for the MAIN 'offers' table ===
# ==============================================================================
import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("INFO: 'db_file_path' not found in global scope. Using default path.")

table_to_audit = 'offers'
print(f"\n--- Auditing the Canonical Schema of the '{table_to_audit}' table ---")

query_sql = f"PRAGMA table_info({table_to_audit});"

try:
    with sqlite3.connect(f'file:{db_file_path}?mode=ro', uri=True) as conn:
        df_schema = pd.read_sql_query(query_sql, conn)

    print("\n✅ SUCCESS: Schema audit complete.")
    print(f"This is the definitive, canonical list of columns for the '{table_to_audit}' table:")
    display(df_schema)

except Exception as e:
    print(f"❌ ERROR: Audit query failed. Reason: {e}")


--- Auditing the Canonical Schema of the 'offers' table ---

✅ SUCCESS: Schema audit complete.
This is the definitive, canonical list of columns for the 'offers' table:


,cid,name,type,notnull,dflt_value,pk
0,0,offer_id,TEXT,0,None,0
1,1,session_fk,TEXT,0,None,0
2,2,ocr_fk,TEXT,0,None,0
3,3,image_content_hash,TEXT,0,None,0
4,4,offer_timestamp,TEXT,0,None,0
5,5,upfront_fare,REAL,0,None,0
6,6,time_to_pickup_sec,REAL,0,None,0
7,7,dist_to_pickup_km,REAL,0,None,0
8,8,est_trip_time_sec,REAL,0,None,0
9,9,est_trip_dist_km,REAL,0,None,0


In [ ]:
 # ==============================================================================
# === DIAGNOSTIC: Investigating the Broken 'ocr_fk' Link ===
# ==============================================================================
print("--- Performing a two-sided audit of the 'ocr_fk' link ---")

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("INFO: 'db_file_path' not found in global scope. Using default path.")

try:
    with sqlite3.connect(db_file_path) as conn:
        # Query 1: What values are we trying to join FROM?
        print("\n--- [1] Sample of values in 'offers.ocr_fk' ---")
        # We also check the COUNT of NULLs to see if the column is just empty.
        query1 = "SELECT ocr_fk FROM offers LIMIT 10;"
        query1_nulls = "SELECT COUNT(*) FROM offers WHERE ocr_fk IS NULL;"

        df_from = pd.read_sql_query(query1, conn)
        null_count = conn.execute(query1_nulls).fetchone()[0]

        display(df_from)
        print(f"Number of NULL values in 'offers.ocr_fk': {null_count}")


        # Query 2: What values are we trying to join TO?
        print("\n--- [2] Sample of values in 'raw_offers_ocr.ocr_id' ---")
        query2 = "SELECT ocr_id FROM raw_offers_ocr LIMIT 10;"
        df_to = pd.read_sql_query(query2, conn)
        display(df_to)

except Exception as e:
    print(f"❌ ERROR: {e}")

--- Performing a two-sided audit of the 'ocr_fk' link ---

--- [1] Sample of values in 'offers.ocr_fk' ---


,ocr_fk
0,None
1,None
2,None
3,None
4,None
5,None
6,None
7,None
8,None
9,None


Number of NULL values in 'offers.ocr_fk': 4765

--- [2] Sample of values in 'raw_offers_ocr.ocr_id' ---


,ocr_id
0,OCR00001
1,OCR00002
2,OCR00003
3,OCR00004
4,OCR00005
5,OCR00006
6,OCR00007
7,OCR00008
8,OCR00009
9,OCR00010


In [ ]:
# =========================================================================================
# === TASK 3 (v2.1 - Final Validation): VERIFY LINK between 'raw_offers_ocr' and 'offers' ===
# =========================================================================================

import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("INFO: 'db_file_path' not found in global scope. Using default path.")

print("\n--- Verifying the integrity of the ocr_fk link between 'offers' and 'raw_offers_ocr' ---")

# This query is now expected to succeed, as the 'offers' table should be correctly structured.
query_sql = """
SELECT
    CASE
        WHEN ocr.ocr_id IS NULL THEN 'Orphaned (Link Failed)'
        ELSE 'Linked Successfully'
    END AS link_status,
    COUNT(o.offer_id) AS number_of_offers
FROM
    offers o
LEFT JOIN
    raw_offers_ocr ocr ON o.ocr_fk = ocr.ocr_id
GROUP BY
    link_status;
"""

try:
    with sqlite3.connect(db_file_path) as conn:
        df_link_status = pd.read_sql_query(query_sql, conn)

    print("\n✅ SUCCESS: Audit query executed.")
    print("Displaying link status count:")
    display(df_link_status)

    # Provide a clear, actionable summary
    if 'Orphaned (Link Failed)' in df_link_status['link_status'].values:
        print("\n🛑 FINDING: One or more offers could not be linked back to their OCR source. Investigation required.")
    else:
        print("\n✅ FINDING: 100% of offers are successfully linked to their raw OCR source. Data provenance is intact.")

except Exception as e:
    print(f"❌ ERROR during audit: {e}")


--- Verifying the integrity of the ocr_fk link between 'offers' and 'raw_offers_ocr' ---

✅ SUCCESS: Audit query executed.
Displaying link status count:


,link_status,number_of_offers
0,Orphaned (Link Failed),4765



🛑 FINDING: One or more offers could not be linked back to their OCR source. Investigation required.


In [ ]:
# =========================================================================================
# === CELL 7 (v2.9 - Resilient Header): PHASE 5 - ETL FOR 'activity_earnings' TABLE ===
# =========================================================================================
print("--- PHASE 5 STARTED: ETL for 'activity_earnings' table ---")

import pandas as pd
import sqlite3
import os
import gspread
from google.colab import auth
from google.auth import default

# --- Self-Contained Configuration ---
try: gc
except NameError:
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

PLATFORM_DATA_SHEET_NAME = "platform_data"
EARNINGS_TAB_NAME = "activity_earnings"
TARGET_TABLE = "activity_earnings"

# ==============================================================================
# --- 1. EXTRACT (WITH RESILIENT HEADER FIX) ---
# ==============================================================================
print("\n--- [E] EXTRACTING raw data from GSheet ---")
df_raw_earnings = None
try:
    workbook = gc.open(PLATFORM_DATA_SHEET_NAME)
    sheet = workbook.worksheet(EARNINGS_TAB_NAME)
    print(f"Successfully opened worksheet '{EARNINGS_TAB_NAME}'.")

    all_values = sheet.get_all_values()
    data_rows = all_values[1:]

    # Define our own canonical headers. This is the source of truth, ignoring the sheet's header.
    canonical_headers = [
        'activity_earnings_id', 'activity_earnings_timestamp', 'product_category',
        'net_earning', 'details_url'
    ]

    df_raw_earnings = pd.DataFrame(data_rows)
    df_raw_earnings = df_raw_earnings.iloc[:, 0:len(canonical_headers)]
    df_raw_earnings.columns = canonical_headers
    print(f"✅ Success: Extracted {len(df_raw_earnings)} rows using canonical headers.")

except Exception as e:
    print(f"❌ ERROR during extraction: {e}")
    df_raw_earnings = None

# ==============================================================================
# --- 2. TRANSFORM ---
# ==============================================================================
if df_raw_earnings is not None:
    print("\n--- [T] TRANSFORMING and cleaning 'activity_earnings' data ---")
    df_transformed = df_raw_earnings.copy()

    # --- A. Robust Data Cleaning & Parsing (Correct Order) ---
    df_transformed.replace({'': None, 'N/A': None, 'nan': None}, inplace=True)
    df_transformed['net_earning'] = pd.to_numeric(df_transformed['net_earning'].astype(str).str.replace(r'[MX$,]', '', regex=True).str.strip(), errors='coerce')

    df_transformed['activity_earnings_id'] = df_transformed['activity_earnings_id'].astype(str).str.replace('AE', '')
    df_transformed['activity_earnings_id'] = pd.to_numeric(df_transformed['activity_earnings_id'], errors='coerce')

    def intelligent_date_parser(date_string):
        if not isinstance(date_string, str) or date_string.strip() == '': return pd.NaT
        try: return pd.to_datetime(date_string, infer_datetime_format=True)
        except (ValueError, TypeError):
            try:
                s_cleaned = date_string.replace('st', '').replace('nd', '').replace('rd', '').replace('th', '')
                return pd.to_datetime(s_cleaned, format='%A, %B %d, %Y\n%H:%M')
            except (ValueError, TypeError): return pd.NaT
    df_transformed['activity_earnings_timestamp'] = df_transformed['activity_earnings_timestamp'].apply(intelligent_date_parser)
    print("Intelligent, inferential parsing of timestamps complete.")

    initial_rows = len(df_transformed)
    df_transformed.dropna(subset=['activity_earnings_timestamp', 'activity_earnings_id'], inplace=True)
    df_transformed['activity_earnings_id'] = df_transformed['activity_earnings_id'].astype(int)
    print(f"Dropped {initial_rows - len(df_transformed)} rows with missing critical data.")

    df_transformed.sort_values('activity_earnings_timestamp', inplace=True, ascending=True)
    df_transformed.reset_index(drop=True, inplace=True)
    print("DataFrame sorted chronologically and index reset.")

    # --- B. Final Schema Alignment ---
    final_columns = ['activity_earnings_id', 'activity_earnings_timestamp', 'product_category', 'net_earning', 'details_url']
    df_to_load = df_transformed[[col for col in final_columns if col in df_transformed.columns]]
    print("Final schema alignment complete.")

    # --- Enhanced Diagnostic Display ---
    print(f"\n--- Displaying diagnostics for the {len(df_to_load)} records to be loaded ---")
    if not df_to_load.empty:
        print("\nFirst 20 records (True chronological start):")
        display(df_to_load.head(20))
        print("\nLast 20 records (True chronological end):")
        display(df_to_load.tail(20))
    else:
        print("\nWARNING: DataFrame is empty after cleaning and filtering.")

    # ==============================================================================
    # --- 3. LOAD ---
    # ==============================================================================
    print(f"\n--- [L] LOADING data into database '{db_file_path}' ---")
    try:
        with sqlite3.connect(db_file_path) as conn:
            df_to_load.to_sql(TARGET_TABLE, conn, if_exists='replace', index=False)
        print(f"✅ Success: Loaded {len(df_to_load)} records into '{TARGET_TABLE}'.")
        print("\n--- PHASE 5 COMPLETE ---")
    except Exception as e:
        print(f"❌ ERROR during load operation: {e}")
else:
    print("\n--- Halting execution due to error in EXTRACT phase. ---")

--- PHASE 5 STARTED: ETL for 'activity_earnings' table ---

--- [E] EXTRACTING raw data from GSheet ---
Successfully opened worksheet 'activity_earnings'.
✅ Success: Extracted 3446 rows using canonical headers.

--- [T] TRANSFORMING and cleaning 'activity_earnings' data ---


/tmp/ipython-input-3675718752.py:68: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  try: return pd.to_datetime(date_string, infer_datetime_format=True)


Intelligent, inferential parsing of timestamps complete.
Dropped 0 rows with missing critical data.
DataFrame sorted chronologically and index reset.
Final schema alignment complete.

--- Displaying diagnostics for the 3446 records to be loaded ---

First 20 records (True chronological start):


,activity_earnings_id,activity_earnings_timestamp,product_category,net_earning,details_url
0,1,2023-05-29 12:18:00,UberX,36.98,None
1,2,2023-05-29 12:39:00,UberX,41.98,None
2,3,2023-05-29 13:05:00,UberX,75.89,None
3,4,2023-05-29 13:47:00,UberX,69.04,None
4,5,2023-05-29 19:49:00,UberX,49.11,None
5,6,2023-05-29 20:21:00,UberX,34.41,None
6,7,2023-05-29 20:32:00,UberX,53.77,None
7,8,2023-06-05 11:38:00,UberX,82.16,None
8,9,2023-06-05 17:00:00,UberX,61.14,None
9,10,2023-06-05 17:31:00,UberX,56.38,None



Last 20 records (True chronological end):


,activity_earnings_id,activity_earnings_timestamp,product_category,net_earning,details_url
3426,3427,2025-09-29 17:11:00,Uber Priority,194.62,View Details
3427,3428,2025-09-30 06:02:00,UberX,110.64,View Details
3428,3429,2025-09-30 06:37:00,Uber Priority,133.01,View Details
3429,3430,2025-09-30 07:20:00,Uber Priority,90.55,View Details
3430,3431,2025-09-30 07:47:00,UberX,0.00,View Details
3431,3432,2025-09-30 08:16:00,Comfort,12.14,View Details
3432,3433,2025-09-30 08:36:00,Uber Priority,129.04,View Details
3433,3434,2025-09-30 13:41:00,Comfort,62.16,View Details
3434,3435,2025-09-30 14:04:00,UberX,89.87,View Details
3435,3436,2025-09-30 14:44:00,Comfort,88.81,View Details



--- [L] LOADING data into database '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db' ---
✅ Success: Loaded 3446 records into 'activity_earnings'.

--- PHASE 5 COMPLETE ---


In [ ]:
# ====================================================================================
# === CELL 8 (v1.3 - Fully Self-Contained): PHASE 6 - ETL FOR 'lifetime_trips' TABLE ===
# ====================================================================================
print("--- PHASE 6 STARTED: ETL for 'lifetime_trips' table ---")

import pandas as pd
import sqlite3
import os
import gspread
from google.colab import auth
from google.auth import default

# === THE CRITICAL FIX: The full, robust, self-contained configuration block ===
try:
    gc
except NameError:
    print("INFO: Google client 'gc' not found. Re-authenticating...")
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    print("INFO: 'db_file_path' not found. Defining with default path...")
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
# ================================================================================

PLATFORM_DATA_SHEET_NAME = "platform_data"
LIFETIME_TRIPS_TAB_NAME = "lifetime_trips"
TARGET_TABLE = "lifetime_trips"

# ==============================================================================
# --- 1. EXTRACT (WITH CORRECTED HEADERS) ---
# ==============================================================================
print("\n--- [E] EXTRACTING raw data from GSheet ---")
df_raw_trips = None
try:
    workbook = gc.open(PLATFORM_DATA_SHEET_NAME)
    sheet = workbook.worksheet(LIFETIME_TRIPS_TAB_NAME)
    print(f"Successfully opened worksheet '{LIFETIME_TRIPS_TAB_NAME}'.")

    all_values = sheet.get_all_values()
    data_rows = all_values[1:]

    canonical_headers = [
        'lifetime_trips_id', 'global_product_name', 'status', 'request_timestamp',
        'pickup_timestamp', 'dropoff_timestamp', 'trip_distance_miles',
        'trip_duration_seconds', 'base_fare', 'original_fare',
        'cancellation_fee', 'currency_code', 'vehicle_uuid', 'license_plate'
    ]

    df_raw_trips = pd.DataFrame(data_rows)
    df_raw_trips = df_raw_trips.iloc[:, 0:len(canonical_headers)]
    df_raw_trips.columns = canonical_headers
    print(f"✅ Success: Extracted {len(df_raw_trips)} rows using CORRECTED canonical headers.")

except Exception as e:
    print(f"❌ ERROR during extraction: {e}")
    df_raw_trips = None

# ==============================================================================
# --- 2. TRANSFORM ---
# ==============================================================================
if df_raw_trips is not None:
    print("\n--- [T] TRANSFORMING and cleaning 'lifetime_trips' data ---")
    df_transformed = df_raw_trips.copy()

    df_transformed.replace({'': None, 'N/A': None, 'nan': None}, inplace=True)
    df_transformed['lifetime_trips_id'] = pd.to_numeric(df_transformed['lifetime_trips_id'].astype(str).str.replace('LT', ''), errors='coerce')
    numeric_cols = ['trip_distance_miles', 'trip_duration_seconds', 'base_fare', 'original_fare', 'cancellation_fee']
    for col in numeric_cols:
        if col in df_transformed.columns:
            df_transformed[col] = pd.to_numeric(df_transformed[col], errors='coerce')
    timestamp_cols = ['request_timestamp', 'pickup_timestamp', 'dropoff_timestamp']
    for col in timestamp_cols:
        if col in df_transformed.columns:
            df_transformed[col] = pd.to_datetime(df_transformed[col], errors='coerce')
    print("Data types cleaned and coerced.")

    df_transformed.dropna(subset=['lifetime_trips_id', 'request_timestamp'], inplace=True)
    df_transformed['lifetime_trips_id'] = df_transformed['lifetime_trips_id'].astype(int)
    print("Dropped rows with missing critical data and cast PK to integer.")

    final_columns = [
        'lifetime_trips_id', 'global_product_name', 'status',
        'request_timestamp', 'pickup_timestamp', 'dropoff_timestamp',
        'trip_distance_miles', 'trip_duration_seconds', 'base_fare',
        'original_fare', 'cancellation_fee', 'currency_code',
        'vehicle_uuid', 'license_plate'
    ]
    df_to_load = df_transformed[[col for col in final_columns if col in df_transformed.columns]].copy()
    for col in ['vehicle_uuid', 'license_plate']:
        if col not in df_to_load.columns:
            df_to_load[col] = None
    print("Final schema alignment complete.")

    # ==============================================================================
    # --- 3. LOAD ---
    # ==============================================================================
    print(f"\n--- [L] LOADING data into database '{db_file_path}' ---")
    try:
        with sqlite3.connect(db_file_path) as conn:
            df_to_load.to_sql(TARGET_TABLE, conn, if_exists='replace', index=False)
        print(f"✅ Success: Loaded {len(df_to_load)} records into '{TARGET_TABLE}'.")
        print("\n--- PHASE 6 COMPLETE ---")
    except Exception as e:
        print(f"❌ ERROR during load operation: {e}")
else:
    print("\n--- Halting execution due to error in EXTRACT phase. ---")

--- PHASE 6 STARTED: ETL for 'lifetime_trips' table ---

--- [E] EXTRACTING raw data from GSheet ---
Successfully opened worksheet 'lifetime_trips'.
✅ Success: Extracted 3446 rows using CORRECTED canonical headers.

--- [T] TRANSFORMING and cleaning 'lifetime_trips' data ---
Data types cleaned and coerced.
Dropped rows with missing critical data and cast PK to integer.
Final schema alignment complete.

--- [L] LOADING data into database '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db' ---
✅ Success: Loaded 3446 records into 'lifetime_trips'.

--- PHASE 6 COMPLETE ---


In [ ]:
# ==============================================================================
# === DIAGNOSTIC CELL: Inspect First & Last 20 Records of 'lifetime_trips' ===
# This verifies the chronological integrity of the newly loaded table.
# ==============================================================================

print("--- Running Diagnostic: Inspecting 'lifetime_trips' table ---")

# We will query the entire table and let pandas do the inspection.
query = "SELECT * FROM lifetime_trips ORDER BY request_timestamp ASC;"

try:
    with sqlite3.connect(db_file_path) as conn:
        df_lifetime_trips_audit = pd.read_sql_query(query, conn)

    print("\n✅ SUCCESS: Query executed.")
    print(f"Total records found in 'lifetime_trips': {len(df_lifetime_trips_audit)}")

    if not df_lifetime_trips_audit.empty:
        print("\n--- First 20 records (Chronological Start) ---")
        display(df_lifetime_trips_audit.head(20))

        print("\n--- Last 20 records (Chronological End) ---")
        display(df_lifetime_trips_audit.tail(20))

        print("\n--- Schema Verification ---")
        df_lifetime_trips_audit.info()
    else:
        print("\nWARNING: The 'lifetime_trips' table is empty.")

except Exception as e:
    print(f"❌ ERROR: Query failed. Reason: {e}")

--- Running Diagnostic: Inspecting 'lifetime_trips' table ---

✅ SUCCESS: Query executed.
Total records found in 'lifetime_trips': 3446

--- First 20 records (Chronological Start) ---


,lifetime_trips_id,global_product_name,status,request_timestamp,pickup_timestamp,dropoff_timestamp,trip_distance_miles,trip_duration_seconds,base_fare,original_fare,cancellation_fee,currency_code,vehicle_uuid,license_plate
0,1,UberX,completed,2023-05-29 12:18:42+00:00,2023-05-29 12:23:45+00:00,2023-05-29 12:38:02+00:00,2.005310,857.0,6.87,69.99,0.0,MXN,None,None
1,2,UberX,completed,2023-05-29 12:39:17+00:00,2023-05-29 12:47:04+00:00,2023-05-29 12:58:27+00:00,1.892561,683.0,6.87,54.49,0.0,MXN,None,None
2,3,UberX,completed,2023-05-29 13:05:37+00:00,2023-05-29 13:18:17+00:00,2023-05-29 13:47:01+00:00,4.338404,1717.0,6.87,97.86,0.0,MXN,None,None
3,4,UberX,completed,2023-05-29 13:47:56+00:00,2023-05-29 13:53:08+00:00,2023-05-29 14:13:29+00:00,2.944863,1221.0,6.87,79.90,0.0,MXN,None,None
4,5,UberX,completed,2023-05-29 19:49:52+00:00,2023-05-29 20:04:28+00:00,2023-05-29 20:15:42+00:00,2.301361,674.0,6.87,71.39,0.0,MXN,None,None
5,6,UberX,completed,2023-05-29 20:21:57+00:00,2023-05-29 20:27:00+00:00,2023-05-29 20:36:23+00:00,2.547121,563.0,6.87,59.91,0.0,MXN,None,None
6,7,UberX,completed,2023-05-29 20:32:55+00:00,2023-05-29 20:39:51+00:00,2023-05-29 20:50:30+00:00,1.561299,639.0,6.87,49.90,0.0,MXN,None,None
7,8,UberX,completed,2023-06-05 11:38:51+00:00,2023-06-05 11:41:21+00:00,2023-06-05 12:10:28+00:00,6.361610,1747.0,6.87,149.93,0.0,MXN,None,None
8,9,UberX,completed,2023-06-05 17:00:03+00:00,2023-06-05 17:08:11+00:00,2023-06-05 17:32:03+00:00,4.397000,1432.0,6.87,99.96,0.0,MXN,None,None
9,10,UberX,completed,2023-06-05 17:31:28+00:00,2023-06-05 17:43:27+00:00,2023-06-05 17:49:39+00:00,1.119857,372.0,6.87,69.91,0.0,MXN,None,None



--- Last 20 records (Chronological End) ---


,lifetime_trips_id,global_product_name,status,request_timestamp,pickup_timestamp,dropoff_timestamp,trip_distance_miles,trip_duration_seconds,base_fare,original_fare,cancellation_fee,currency_code,vehicle_uuid,license_plate
3426,3427,UberX,completed,2025-09-29 17:11:44+00:00,2025-09-29 17:28:26+00:00,2025-09-29 18:34:59+00:00,7.413063,3993.0,10.73,321.50,0.00,MXN,None,None
3427,3428,UberX,completed,2025-09-30 06:02:33+00:00,2025-09-30 06:08:46+00:00,2025-09-30 06:36:34+00:00,9.793376,1668.0,10.73,189.93,0.00,MXN,None,None
3428,3429,UberX,completed,2025-09-30 06:37:48+00:00,2025-09-30 06:52:52+00:00,2025-09-30 07:14:47+00:00,5.871442,1315.0,10.73,153.54,0.00,MXN,None,None
3429,3430,UberX,completed,2025-09-30 07:20:36+00:00,2025-09-30 07:29:12+00:00,2025-09-30 07:50:08+00:00,4.478963,1256.0,10.73,148.32,0.00,MXN,None,None
3430,3431,UberX,rider_canceled,2025-09-30 07:47:13+00:00,None,None,0.000000,0.0,NaN,0.00,0.00,MXN,None,None
3431,3432,Mid-Tier,rider_canceled,2025-09-30 08:16:56+00:00,None,None,0.000000,0.0,NaN,16.21,13.97,MXN,None,None
3432,3433,UberX,completed,2025-09-30 08:36:22+00:00,2025-09-30 08:47:05+00:00,2025-09-30 09:13:16+00:00,3.463705,1570.0,10.73,153.46,0.00,MXN,None,None
3433,3434,Mid-Tier,completed,2025-09-30 13:41:46+00:00,2025-09-30 13:47:51+00:00,2025-09-30 14:03:50+00:00,2.188194,959.0,13.30,121.51,0.00,MXN,None,None
3434,3435,UberX,completed,2025-09-30 14:04:33+00:00,2025-09-30 14:10:06+00:00,2025-09-30 14:41:07+00:00,6.237990,1862.0,10.73,202.07,0.00,MXN,None,None
3435,3436,Mid-Tier,completed,2025-09-30 14:44:42+00:00,2025-09-30 14:57:34+00:00,2025-09-30 15:10:19+00:00,3.370230,764.0,13.30,197.50,0.00,MXN,None,None



--- Schema Verification ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3446 entries, 0 to 3445
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   lifetime_trips_id      3446 non-null   int64  
 1   global_product_name    3446 non-null   object 
 2   status                 3446 non-null   object 
 3   request_timestamp      3446 non-null   object 
 4   pickup_timestamp       3402 non-null   object 
 5   dropoff_timestamp      3402 non-null   object 
 6   trip_distance_miles    3446 non-null   float64
 7   trip_duration_seconds  3444 non-null   float64
 8   base_fare              3402 non-null   float64
 9   original_fare          3444 non-null   float64
 10  cancellation_fee       3446 non-null   float64
 11  currency_code          3444 non-null   object 
 12  vehicle_uuid           0 non-null      object 
 13  license_plate          0 non-null      object 
dtypes: float64(5), int64(1), ob

In [ ]:
# =================================================================================================
# === CELL 9 (v1.5 - Comprehensive Fail-Safe): TASK A - FORGE THE "GOLDEN LINK" ===
# =================================================================================================
import pandas as pd
import sqlite3
import os

# --- Self-Contained Configuration ---
if 'db_file_path' not in locals() and 'db_file_path' not in globals():
    project_root = '/content/drive/My Drive/_Pienza'
    db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')
    print("WARNING: 'db_file_path' not found in global scope. Using default path.")

print("--- TASK A STARTED: Forging the 'Golden Link' using Direct ID Match (v1.5) ---")

try:
    with sqlite3.connect(db_file_path) as conn:
        print("Extracting source tables from opus.db...")
        df_earnings = pd.read_sql_query("SELECT * FROM activity_earnings;", conn)
        df_earnings['activity_earnings_timestamp'] = pd.to_datetime(df_earnings['activity_earnings_timestamp'])

        df_lifetime = pd.read_sql_query("SELECT lifetime_trips_id FROM lifetime_trips;", conn)
        df_lifetime['lifetime_trips_id'] = pd.to_numeric(df_lifetime['lifetime_trips_id'], errors='coerce') # Ensure ID is numeric
    print(f"✅ Extracted {len(df_earnings)} earnings records and {len(df_lifetime)} lifetime trips.")

# --- [2] The Direct, Perfect Inner Join ---
    print("\n--- [T] Performing Direct INNER JOIN on Primary Keys (1:1 Match) ---")

    df_linked = pd.merge(
        df_earnings,
        df_lifetime,
        left_on='activity_earnings_id',  # PK of left table
        right_on='lifetime_trips_id', # PK of right table
        how='inner'
    )

    print("✅ Direct ID Join complete. Only perfectly matched records proceed.")

# --- [3] Final Schema Alignment and Load ---
    df_linked.rename(columns={'lifetime_trips_id': 'lifetime_trips_fk'}, inplace=True)

    final_columns = [
        'activity_earnings_id',
        'lifetime_trips_fk', # The new, critical link
        'activity_earnings_timestamp',
        'product_category',
        'net_earning',
        'details_url'
    ]

    # Defensive selection to ensure only expected columns are carried forward
    df_to_load = df_linked[[col for col in final_columns if col in df_linked.columns]].copy()

    print("Schema columns finalized for loading.")

# --- [4] Load to Database ---
    print("\n--- [L] LOADING enriched data back into 'activity_earnings' table ---")
    with sqlite3.connect(db_file_path) as conn:
        df_to_load.to_sql('activity_earnings', conn, if_exists='replace', index=False)
    print(f"✅ Success: Loaded {len(df_to_load)} records into 'activity_earnings' with new FK.")
    print("\n--- TASK A COMPLETE ---")

except Exception as e:
    print(f"❌ ERROR during ETL: {e}")

# --- Final Verification ---
print("\n--- Verifying a sample of the newly linked data ---")
try:
    with sqlite3.connect(db_file_path) as conn:
        verify_query = "SELECT * FROM activity_earnings ORDER BY activity_earnings_id DESC LIMIT 5;"
        df_verify = pd.read_sql_query(verify_query, conn)
        display(df_verify)
except Exception as e:
    print(f"❌ ERROR during verification: {e}")

--- TASK A STARTED: Forging the 'Golden Link' using Direct ID Match (v1.5) ---
Extracting source tables from opus.db...
✅ Extracted 3446 earnings records and 3446 lifetime trips.

--- [T] Performing Direct INNER JOIN on Primary Keys (1:1 Match) ---
✅ Direct ID Join complete. Only perfectly matched records proceed.
Schema columns finalized for loading.

--- [L] LOADING enriched data back into 'activity_earnings' table ---
✅ Success: Loaded 3446 records into 'activity_earnings' with new FK.

--- TASK A COMPLETE ---

--- Verifying a sample of the newly linked data ---


,activity_earnings_id,lifetime_trips_fk,activity_earnings_timestamp,product_category,net_earning,details_url
0,3446,3446,2025-10-01 09:06:00,Comfort,142.17,View Details
1,3445,3445,2025-10-01 08:35:00,Business Comfort,148.14,View Details
2,3444,3444,2025-10-01 08:27:00,Comfort,0.00,View Details
3,3443,3443,2025-10-01 07:56:00,UberX,104.96,View Details
4,3442,3442,2025-10-01 07:33:00,Black,117.32,View Details


In [ ]:
# ==============================================================================
# === DIAGNOSTIC: Hunting for Whitespace-Only 'outcome_fk' Values ===
# ==============================================================================
print("--- Hunting for whitespace-only 'outcome_fk' values in the 'diamond_offers' GSheet ---")

import pandas as pd
import gspread
from google.colab import auth
from google.auth import default

# ... (Self-Contained Configuration) ...

try:
    # --- Get the raw data from the GSheet source ---
    workbook = gc.open(DIAMOND_SHEET_NAME)
    sheet = workbook.worksheet(DIAMOND_TAB_NAME)
    df_source = pd.DataFrame(sheet.get_all_records())
    df_source.columns = df_source.columns.str.lower().str.replace(' ', '_')
    print(f"\nExtracted {len(df_source)} rows from the GSheet source.")

    # --- THE "WHITESPACE HUNTER" LOGIC ---
    # We are looking for rows that are NOT empty, but BECOME empty after stripping whitespace.
    anomaly_mask = (df_source['outcome_fk'] != '') & (df_source['outcome_fk'].str.strip() == '')
    df_anomalies = df_source[anomaly_mask]

    print(f"\n✅ Anomaly hunt complete. Found {len(df_anomalies)} records containing only whitespace.")

    if not df_anomalies.empty:
        print("These are the specific offers that need to be fixed at the source (cells should be made truly empty):")
        display(df_anomalies[['offer_id', 'outcome_fk']])
    else:
        print("No whitespace-only discrepancies were found.")

except Exception as e:
    print(f"❌ ERROR during diagnostic: {e}")

--- Hunting for whitespace-only 'outcome_fk' values in the 'diamond_offers' GSheet ---

Extracted 4765 rows from the GSheet source.

✅ Anomaly hunt complete. Found 0 records containing only whitespace.
No whitespace-only discrepancies were found.


In [ ]:
# ==============================================================================
# === DIAGNOSTIC: "ID Set" Reconciliation for 'outcome_fk' ===
# ==============================================================================
print("--- Performing 'ID Set' Reconciliation for 'outcome_fk' ---")

import pandas as pd
import sqlite3
import os
import gspread
from google.colab import auth
from google.auth import default

# --- Self-Contained Configuration ---
try: gc
except NameError:
    auth.authenticate_user(); creds, _ = default(); gc = gspread.authorize(creds)
if 'db_file_path' not in locals():
    project_root = '/content/drive/My Drive/_Pienza'; db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')

DIAMOND_SHEET_NAME = "raw_Offers"
DIAMOND_TAB_NAME = "diamond_offers"

try:
    # --- [1] Get the ID set from the GSheet Source ---
    print("\nExtracting ID set from GSheet source...")
    workbook = gc.open(DIAMOND_SHEET_NAME)
    sheet = workbook.worksheet(DIAMOND_TAB_NAME)
    df_source = pd.DataFrame(sheet.get_all_records())
    df_source.columns = df_source.columns.str.lower().str.replace(' ', '_')

    # Filter for non-empty 'outcome_fk' and get the set of 'offer_id's
    gsheet_ids = set(df_source[df_source['outcome_fk'] != '']['offer_id'])
    print(f"✅ Found {len(gsheet_ids)} offer_ids with non-empty 'outcome_fk' in GSheet.")

    # --- [2] Get the ID set from the Database Destination ---
    print("\nExtracting ID set from opus.db destination...")
    with sqlite3.connect(db_file_path) as conn:
        db_query = "SELECT offer_id FROM offers WHERE outcome_fk IS NOT NULL;"
        df_db = pd.read_sql_query(db_query, conn)

    db_ids = set(df_db['offer_id'])
    print(f"✅ Found {len(db_ids)} offer_ids with non-NULL 'outcome_fk' in the database.")

    # --- [3] THE RECONCILIATION ---
    print("\n--- Performing Set Difference Analysis ---")

    # Find IDs that are in the GSheet set but NOT in the database set
    discrepancy_ids = gsheet_ids - db_ids

    if not discrepancy_ids:
        print("\n✅ SUCCESS: The sets are identical. Reconciliation is perfect.")
    else:
        print(f"\n🛑 ANOMALY DETECTED: Found {len(discrepancy_ids)} 'offer_id's present in the GSheet count but missing from the database count.")
        print("This is definitive proof that these are the records with whitespace-only 'outcome_fk' values.")
        print("\nAnomaly List (offer_id):")
        # Print the list of problematic IDs
        for offer_id in sorted(list(discrepancy_ids)):
            print(offer_id)

except Exception as e:
    print(f"❌ ERROR during diagnostic: {e}")

--- Performing 'ID Set' Reconciliation for 'outcome_fk' ---

Extracting ID set from GSheet source...
✅ Found 347 offer_ids with non-empty 'outcome_fk' in GSheet.

Extracting ID set from opus.db destination...
✅ Found 347 offer_ids with non-NULL 'outcome_fk' in the database.

--- Performing Set Difference Analysis ---

✅ SUCCESS: The sets are identical. Reconciliation is perfect.


In [ ]:
# ==============================================================================
# === DIAGNOSTIC: "ID Set" Reconciliation for 'pickup_address' ===
# ==============================================================================
print("--- Performing 'ID Set' Reconciliation for 'pickup_address' ---")

import pandas as pd
import sqlite3
import os
import gspread
from google.colab import auth
from google.auth import default

# --- Self-Contained Configuration ---
try: gc
except NameError:
    auth.authenticate_user(); creds, _ = default(); gc = gspread.authorize(creds)
if 'db_file_path' not in locals():
    project_root = '/content/drive/My Drive/_Pienza'; db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')

DIAMOND_SHEET_NAME = "raw_Offers"
DIAMOND_TAB_NAME = "diamond_offers"

try:
    # --- [1] Get the ID set from the GSheet Source ---
    print("\nExtracting ID set from GSheet source...")
    workbook = gc.open(DIAMOND_SHEET_NAME)
    sheet = workbook.worksheet(DIAMOND_TAB_NAME)
    df_source = pd.DataFrame(sheet.get_all_records())
    df_source.columns = df_source.columns.str.lower().str.replace(' ', '_')

    # Filter for non-empty 'pickup_address' and get the set of 'offer_id's
    gsheet_ids = set(df_source[df_source['pickup_address'] != '']['offer_id'])
    print(f"✅ Found {len(gsheet_ids)} offer_ids with non-empty 'pickup_address' in GSheet.")

    # --- [2] Get the ID set from the Database Destination ---
    print("\nExtracting ID set from opus.db destination...")
    with sqlite3.connect(db_file_path) as conn:
        db_query = "SELECT offer_id FROM offers WHERE pickup_address IS NOT NULL AND pickup_address != '';"
        df_db = pd.read_sql_query(db_query, conn)

    db_ids = set(df_db['offer_id'])
    print(f"✅ Found {len(db_ids)} offer_ids with non-NULL 'pickup_address' in the database.")

    # --- [3] THE RECONCILIATION ---
    print("\n--- Performing Set Difference Analysis ---")

    # Find IDs that are in the GSheet set but NOT in the database set
    discrepancy_ids = gsheet_ids - db_ids

    if not discrepancy_ids:
        print("\n✅ SUCCESS: The sets are identical. Reconciliation is perfect.")
    else:
        print(f"\n🛑 ANOMALY DETECTED: Found {len(discrepancy_ids)} 'offer_id'(s) present in the GSheet count but missing from the database count.")
        print("This is the record with the whitespace-only 'pickup_address' value.")
        print("\nAnomaly (offer_id):")
        for offer_id in sorted(list(discrepancy_ids)):
            print(offer_id)

except Exception as e:
    print(f"❌ ERROR during diagnostic: {e}")

--- Performing 'ID Set' Reconciliation for 'pickup_address' ---

Extracting ID set from GSheet source...
✅ Found 4757 offer_ids with non-empty 'pickup_address' in GSheet.

Extracting ID set from opus.db destination...
✅ Found 4757 offer_ids with non-NULL 'pickup_address' in the database.

--- Performing Set Difference Analysis ---

✅ SUCCESS: The sets are identical. Reconciliation is perfect.


In [ ]:
# ==============================================================================
# === DIAGNOSTIC: The "Boolean Interrogation" ===
# ==============================================================================
print("--- Interrogating the true format of the 'is_surge' column from the GSheet ---")

import pandas as pd
import gspread
from google.colab import auth
from google.auth import default

# --- Self-Contained Configuration ---
try: gc
except NameError:
    auth.authenticate_user(); creds, _ = default(); gc = gspread.authorize(creds)

DIAMOND_SHEET_NAME = "raw_Offers"
DIAMOND_TAB_NAME = "diamond_offers"

try:
    # --- [1] Extract the raw data ---
    workbook = gc.open(DIAMOND_SHEET_NAME)
    sheet = workbook.worksheet(DIAMOND_TAB_NAME)
    df_source = pd.DataFrame(sheet.get_all_records())
    df_source.columns = df_source.columns.str.lower().str.replace(' ', '_')
    print(f"\nExtracted {len(df_source)} rows from the GSheet source.")

    # --- [2] THE INTERROGATION ---
    # We will inspect the 'is_surge' column as a representative sample.
    if 'is_surge' in df_source.columns:
        unique_values = df_source['is_surge'].unique()
        print("\n✅ Interrogation complete.")
        print("These are the unique raw values found in the 'is_surge' column:")
        print(unique_values)
        print("\nData type of the column:")
        print(df_source['is_surge'].dtype)
    else:
        print("🛑 ERROR: 'is_surge' column not found in the GSheet data.")

except Exception as e:
    print(f"❌ ERROR during diagnostic: {e}")

--- Interrogating the true format of the 'is_surge' column from the GSheet ---

Extracted 4765 rows from the GSheet source.

✅ Interrogation complete.
These are the unique raw values found in the 'is_surge' column:
['TRUE' 'FALSE']

Data type of the column:
object


In [ ]:
# ==============================================================================
# === DIAGNOSTIC: "ID Set" Reconciliation for 'special_note_raw' ===
# ==============================================================================
print("--- Performing 'ID Set' Reconciliation for 'special_note_raw' ---")

import pandas as pd
import sqlite3
import os
import gspread
from google.colab import auth
from google.auth import default

# --- Self-Contained Configuration ---
try: gc
except NameError:
    auth.authenticate_user(); creds, _ = default(); gc = gspread.authorize(creds)
if 'db_file_path' not in locals():
    project_root = '/content/drive/My Drive/_Pienza'; db_file_path = os.path.join(project_root, 'Assets/Database/opus.db')

DIAMOND_SHEET_NAME = "raw_Offers"
DIAMOND_TAB_NAME = "diamond_offers"

try:
    # --- [1] Get the ID set from the GSheet Source ---
    print("\nExtracting ID set from GSheet source...")
    workbook = gc.open(DIAMOND_SHEET_NAME)
    sheet = workbook.worksheet(DIAMOND_TAB_NAME)
    df_source = pd.DataFrame(sheet.get_all_records())
    df_source.columns = df_source.columns.str.lower().str.replace(' ', '_')

    # Filter for what GSheet's COUNTA sees as "non-empty"
    gsheet_ids = set(df_source[df_source['special_note_raw'] != '']['offer_id'])
    print(f"✅ Found {len(gsheet_ids)} offer_ids with non-empty 'special_note_raw' in GSheet.")

    # --- [2] Get the ID set from the Database Destination ---
    print("\nExtracting ID set from opus.db destination...")
    with sqlite3.connect(db_file_path) as conn:
        # Our ETL converts whitespace-only to NULL, so we check for NOT NULL.
        db_query = "SELECT offer_id FROM offers WHERE special_note_raw IS NOT NULL;"
        df_db = pd.read_sql_query(db_query, conn)

    db_ids = set(df_db['offer_id'])
    print(f"✅ Found {len(db_ids)} offer_ids with non-NULL 'special_note_raw' in the database.")

    # --- [3] THE RECONCILIATION ---
    print("\n--- Performing Set Difference Analysis ---")

    discrepancy_ids = gsheet_ids - db_ids

    if not discrepancy_ids:
        print("\n✅ SUCCESS: The sets are identical. Reconciliation is perfect.")
    else:
        print(f"\n🛑 ANOMALY DETECTED: Found {len(discrepancy_ids)} 'offer_id'(s) present in the GSheet count but missing from the database count.")
        print("This is definitive proof that these are the records with whitespace-only 'special_note_raw' values.")
        print("\nAnomaly List (offer_id):")
        for offer_id in sorted(list(discrepancy_ids)):
            print(offer_id)

except Exception as e:
    print(f"❌ ERROR during diagnostic: {e}")

--- Performing 'ID Set' Reconciliation for 'special_note_raw' ---

Extracting ID set from GSheet source...
✅ Found 3466 offer_ids with non-empty 'special_note_raw' in GSheet.

Extracting ID set from opus.db destination...
✅ Found 3466 offer_ids with non-NULL 'special_note_raw' in the database.

--- Performing Set Difference Analysis ---

✅ SUCCESS: The sets are identical. Reconciliation is perfect.


In [ ]:
# ==============================================================================
# CELL 12 (v1.3): SURGICAL STRIKE PATCH (CORRECT COLUMN NAME)
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: This definitive version corrects the target column name in the
#          override step to the correct 'offer_id_fk', ensuring the patch
#          operates on the correct data.
# ==============================================================================

import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
import time

# ------------------------------------------------------------------------------
# 0. SETUP & CONSTANTS
# ------------------------------------------------------------------------------
print("--- Initiating Surgical Strike Patch v1.3 ---")
try:
    db_path = '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("✅ Database engine created successfully.")
except Exception as e:
    print(f"🔴 CRITICAL ERROR: Could not create database engine. Halting. Details: {e}")
    raise

TARGET_TRIP_ID = '250825-01'
CORRECT_OFFER_ID = 'OF00318'

# ------------------------------------------------------------------------------
# 1. READ & DISPLAY "BEFORE" STATE
# ------------------------------------------------------------------------------
print("\n⏳ Reading current (corrupted) state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"✅ Successfully read {len(trip_events_df)} records.")
    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])
except Exception as e:
    print(f"🔴 ERROR: Could not read from trip_events table. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 2. THE SURGICAL STRIKE (CORRECTED)
# ------------------------------------------------------------------------------
print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID
# --- THE FIX IS HERE ---
# First, NULL out the existing, corrupted values in the correct column.
trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan

# Now, set the CORRECT offer_id on the T1 event in the correct column.
t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
# --- END OF FIX ---
print(f"✅ Override applied in memory to column 'offer_id_fk'.")

# ------------------------------------------------------------------------------
# 3. RE-RUN "GOLDEN PROPAGATION" in Memory
# ------------------------------------------------------------------------------
print("\n⏳ Re-running Golden Propagation in memory...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("✅ Propagation complete in memory.")

# ------------------------------------------------------------------------------
# 4. WRITE the corrected data back to the database (FORCED SAVE)
# ------------------------------------------------------------------------------
print("\n⏳ Writing patched data back to opus.db (Forced Save Protocol)...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written and connection disposed. Pausing for 5 seconds to allow sync...")
    time.sleep(5)
    print("✅ Write operation complete.")
except Exception as e:
    print(f"🔴 ERROR: Failed to write patched data to the database. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 5. FINAL VERIFICATION (READ-BACK & DISPLAY "AFTER" STATE)
# ------------------------------------------------------------------------------
print("\n--- VERIFICATION (AFTER PATCH) ---")
print("Performing final verification by re-reading data from disk...")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()

    print(f"Displaying corrected data for trip '{TARGET_TRIP_ID}':")
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])

    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("✅ VERIFICATION SUCCESS: Changes have been successfully persisted to the opus.db file.")
except Exception as e:
    print(f"🔴 VERIFICATION FAILED: Could not re-read or verify the data. The sync issue persists. Details: {e}")

print("\n--- SURGICAL STRIKE COMPLETE ---")

--- Initiating Surgical Strike Patch v1.3 ---
✅ Database engine created successfully.

⏳ Reading current (corrupted) state of 'trip_events' table...
✅ Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---


,trip_id_legacy,event_types_id_fk,offer_id_fk
54,250825-01,2,OF00318
55,250825-01,4,OF00034
56,250825-01,5,OF00034



⏳ Applying manual override to trip '250825-01' in memory...
✅ Override applied in memory to column 'offer_id_fk'.

⏳ Re-running Golden Propagation in memory...
✅ Propagation complete in memory.

⏳ Writing patched data back to opus.db (Forced Save Protocol)...
   - Data written and connection disposed. Pausing for 5 seconds to allow sync...
✅ Write operation complete.

--- VERIFICATION (AFTER PATCH) ---
Performing final verification by re-reading data from disk...
Displaying corrected data for trip '250825-01':


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250825-01,2,OF00318
1,250825-01,4,OF00318
2,250825-01,5,OF00318


✅ VERIFICATION SUCCESS: Changes have been successfully persisted to the opus.db file.

--- SURGICAL STRIKE COMPLETE ---


In [ ]:
# ==============================================================================
# CELL [NEW]: THE "OF02649/OF02650" SURGICAL STRIKE PATCH
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: This is a one-time, manual override patch to correct a specific
#          "Stolen Identity" error where trip '250917-10' was incorrectly
#          matched to 'OF02649'. This script surgically remaps it to the
#          correct 'OF02650' and ensures the fix is propagated consistently.
# ==============================================================================

import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
import time

# ------------------------------------------------------------------------------
# 0. SETUP & CONSTANTS
# ------------------------------------------------------------------------------
print("--- Initiating Surgical Strike Patch: The 'OF02649/OF02650' Incident ---")
try:
    db_path = '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("✅ Database engine created successfully.")
except Exception as e:
    print(f"🔴 CRITICAL ERROR: Could not create database engine. Halting. Details: {e}")
    raise

# Define the ground truth based on the Architect's audit
TARGET_TRIP_ID = '250917-10'
INCORRECT_OFFER_ID = 'OF02649' # For verification
CORRECT_OFFER_ID = 'OF02650'

# ------------------------------------------------------------------------------
# 1. READ & DISPLAY "BEFORE" STATE
# ------------------------------------------------------------------------------
print("\n⏳ Reading current (corrupted) state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"✅ Successfully read {len(trip_events_df)} records.")

    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    print(f"Displaying corrupted data for trip '{TARGET_TRIP_ID}':")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])

except Exception as e:
    print(f"🔴 ERROR: Could not read from trip_events table. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 2. THE SURGICAL STRIKE: Manual Override in Memory
# ------------------------------------------------------------------------------
print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
# Create a boolean mask to identify all events for the target trip
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID

# Step 2.1: UNMATCH - Wipe the slate clean for this trip by NULLing out all existing links.
trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan
print(f"   - Unmatched all events for trip '{TARGET_TRIP_ID}'.")

# Step 2.2: REMATCH - Set the CORRECT offer_id, but ONLY on the T1 event for that trip.
t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"   - Rematched T1 event for '{TARGET_TRIP_ID}' to correct offer '{CORRECT_OFFER_ID}'.")

# ------------------------------------------------------------------------------
# 3. RE-RUN "GOLDEN PROPAGATION" in Memory
# ------------------------------------------------------------------------------
print("\n⏳ Re-running Golden Propagation to ensure consistency for the patched trip...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("✅ Propagation complete in memory.")

# ------------------------------------------------------------------------------
# 4. WRITE the corrected data back to the database (FORCED SAVE)
# ------------------------------------------------------------------------------
print("\n⏳ Writing patched data back to opus.db (Forced Save Protocol)...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written and connection disposed. Pausing for 5 seconds to allow sync...")
    time.sleep(5)
    print("✅ Write operation complete.")
except Exception as e:
    print(f"🔴 ERROR: Failed to write patched data to the database. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 5. FINAL VERIFICATION (READ-BACK & DISPLAY "AFTER" STATE)
# ------------------------------------------------------------------------------
print("\n--- VERIFICATION (AFTER PATCH) ---")
print("Performing final verification by re-reading data from disk...")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()

    print(f"Displaying corrected data for trip '{TARGET_TRIP_ID}':")
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])

    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("✅ VERIFICATION SUCCESS: Changes have been successfully persisted to the opus.db file.")
except Exception as e:
    print(f"🔴 VERIFICATION FAILED: Could not re-read or verify the data. The sync issue persists. Details: {e}")

print("\n--- SURGICAL STRIKE COMPLETE ---")

--- Initiating Surgical Strike Patch: The 'OF02649/OF02650' Incident ---
✅ Database engine created successfully.

⏳ Reading current (corrupted) state of 'trip_events' table...
✅ Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---
Displaying corrupted data for trip '250917-10':


,trip_id_legacy,event_types_id_fk,offer_id_fk
547,250917-10,1,OF02648
548,250917-10,2,OF02648
549,250917-10,3,OF02648
550,250917-10,4,OF02648
551,250917-10,5,OF02648



⏳ Applying manual override to trip '250917-10' in memory...
   - Unmatched all events for trip '250917-10'.
   - Rematched T1 event for '250917-10' to correct offer 'OF02650'.

⏳ Re-running Golden Propagation to ensure consistency for the patched trip...
✅ Propagation complete in memory.

⏳ Writing patched data back to opus.db (Forced Save Protocol)...
   - Data written and connection disposed. Pausing for 5 seconds to allow sync...
✅ Write operation complete.

--- VERIFICATION (AFTER PATCH) ---
Performing final verification by re-reading data from disk...
Displaying corrected data for trip '250917-10':


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250917-10,1,OF02650
1,250917-10,2,OF02650
2,250917-10,3,OF02650
3,250917-10,4,OF02650
4,250917-10,5,OF02650


✅ VERIFICATION SUCCESS: Changes have been successfully persisted to the opus.db file.

--- SURGICAL STRIKE COMPLETE ---


In [ ]:
# ==============================================================================
# CELL [NEW]: THE "OF03759/OF03780" SURGICAL STRIKE PATCH
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: This is a one-time, manual override patch to correct a specific
#          "Stolen Identity" error where trip '250924-04' was incorrectly
#          matched to 'OF03759'. This script surgically remaps it to the
#          correct 'OF03780'.
# ==============================================================================

import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
import time

# ------------------------------------------------------------------------------
# 0. SETUP & CONSTANTS
# ------------------------------------------------------------------------------
print("--- Initiating Surgical Strike Patch: The 'OF03759/OF03780' Incident ---")
try:
    db_path = '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("✅ Database engine created successfully.")
except Exception as e:
    print(f"🔴 CRITICAL ERROR: Could not create database engine. Halting. Details: {e}")
    raise

# Define the ground truth based on the Architect's audit
TARGET_TRIP_ID = '250924-04'
INCORRECT_OFFER_ID = 'OF03759' # For verification
CORRECT_OFFER_ID = 'OF03780'

# ------------------------------------------------------------------------------
# 1. READ & DISPLAY "BEFORE" STATE
# ------------------------------------------------------------------------------
print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"✅ Successfully read {len(trip_events_df)} records.")

    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    print(f"Displaying corrupted data for trip '{TARGET_TRIP_ID}':")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])

except Exception as e:
    print(f"🔴 ERROR: Could not read from trip_events table. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 2. THE SURGICAL STRIKE: Manual Override in Memory
# ------------------------------------------------------------------------------
print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID
trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan
t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"✅ Override applied in memory to column 'offer_id_fk'.")

# ------------------------------------------------------------------------------
# 3. RE-RUN "GOLDEN PROPAGATION" in Memory
# ------------------------------------------------------------------------------
print("\n⏳ Re-running Golden Propagation in memory...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("✅ Propagation complete in memory.")

# ------------------------------------------------------------------------------
# 4. WRITE the corrected data back to the database (FORCED SAVE)
# ------------------------------------------------------------------------------
print("\n⏳ Writing patched data back to opus.db (Forced Save Protocol)...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written and connection disposed. Pausing for 5 seconds to allow sync...")
    time.sleep(5)
    print("✅ Write operation complete.")
except Exception as e:
    print(f"🔴 ERROR: Failed to write patched data to the database. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 5. FINAL VERIFICATION (READ-BACK & DISPLAY "AFTER" STATE)
# ------------------------------------------------------------------------------
print("\n--- VERIFICATION (AFTER PATCH) ---")
print("Performing final verification by re-reading data from disk...")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()

    print(f"Displaying corrected data for trip '{TARGET_TRIP_ID}':")
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])

    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("✅ VERIFICATION SUCCESS: Changes have been successfully persisted to the opus.db file.")
except Exception as e:
    print(f"🔴 VERIFICATION FAILED: Could not re-read or verify the data. The sync issue persists. Details: {e}")

print("\n--- SURGICAL STRIKE COMPLETE ---")

--- Initiating Surgical Strike Patch: The 'OF03759/OF03780' Incident ---
✅ Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
✅ Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---
Displaying corrupted data for trip '250924-04':


,trip_id_legacy,event_types_id_fk,offer_id_fk
788,250924-04,2,OF03758
789,250924-04,3,OF03758
790,250924-04,4,OF03758
791,250924-04,5,OF03758



⏳ Applying manual override to trip '250924-04' in memory...
✅ Override applied in memory to column 'offer_id_fk'.

⏳ Re-running Golden Propagation in memory...
✅ Propagation complete in memory.

⏳ Writing patched data back to opus.db (Forced Save Protocol)...
   - Data written and connection disposed. Pausing for 5 seconds to allow sync...
✅ Write operation complete.

--- VERIFICATION (AFTER PATCH) ---
Performing final verification by re-reading data from disk...
Displaying corrected data for trip '250924-04':


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250924-04,2,OF03780
1,250924-04,3,OF03780
2,250924-04,4,OF03780
3,250924-04,5,OF03780


✅ VERIFICATION SUCCESS: Changes have been successfully persisted to the opus.db file.

--- SURGICAL STRIKE COMPLETE ---


In [ ]:
# ==============================================================================
# CELL [NEW]: THE "OF04729/OF04731" SURGICAL STRIKE PATCH
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: This is a one-time, manual override patch to correct a specific
#          "Stolen Identity" error where trip '251001-02' was incorrectly
#          matched to 'OF04729'. This script surgically remaps it to the
#          correct 'OF04731'.
# ==============================================================================

import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
import time

# ------------------------------------------------------------------------------
# 0. SETUP & CONSTANTS
# ------------------------------------------------------------------------------
print("--- Initiating Surgical Strike Patch: The 'OF04729/OF04731' Incident ---")
try:
    db_path = '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("✅ Database engine created successfully.")
except Exception as e:
    print(f"🔴 CRITICAL ERROR: Could not create database engine. Halting. Details: {e}")
    raise

# Define the ground truth based on the Architect's audit
TARGET_TRIP_ID = '251001-02'
INCORRECT_OFFER_ID = 'OF04731' # For verification
CORRECT_OFFER_ID = 'OF04733'

# ------------------------------------------------------------------------------
# 1. READ & DISPLAY "BEFORE" STATE
# ------------------------------------------------------------------------------
print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"✅ Successfully read {len(trip_events_df)} records.")

    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    print(f"Displaying corrupted data for trip '{TARGET_TRIP_ID}':")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])

except Exception as e:
    print(f"🔴 ERROR: Could not read from trip_events table. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 2. THE SURGICAL STRIKE: Manual Override in Memory
# ------------------------------------------------------------------------------
print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID
trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan
t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"✅ Override applied in memory to column 'offer_id_fk'.")

# ------------------------------------------------------------------------------
# 3. RE-RUN "GOLDEN PROPAGATION" in Memory
# ------------------------------------------------------------------------------
print("\n⏳ Re-running Golden Propagation in memory...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("✅ Propagation complete in memory.")

# ------------------------------------------------------------------------------
# 4. WRITE the corrected data back to the database (FORCED SAVE)
# ------------------------------------------------------------------------------
print("\n⏳ Writing patched data back to opus.db (Forced Save Protocol)...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written and connection disposed. Pausing for 5 seconds to allow sync...")
    time.sleep(5)
    print("✅ Write operation complete.")
except Exception as e:
    print(f"🔴 ERROR: Failed to write patched data to the database. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 5. FINAL VERIFICATION (READ-BACK & DISPLAY "AFTER" STATE)
# ------------------------------------------------------------------------------
print("\n--- VERIFICATION (AFTER PATCH) ---")
print("Performing final verification by re-reading data from disk...")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()

    print(f"Displaying corrected data for trip '{TARGET_TRIP_ID}':")
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])

    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("✅ VERIFICATION SUCCESS: Changes have been successfully persisted to the opus.db file.")
except Exception as e:
    print(f"🔴 VERIFICATION FAILED: Could not re-read or verify the data. The sync issue persists. Details: {e}")

print("\n--- SURGICAL STRIKE COMPLETE ---")

--- Initiating Surgical Strike Patch: The 'OF04729/OF04731' Incident ---
✅ Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
✅ Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---
Displaying corrupted data for trip '251001-02':


,trip_id_legacy,event_types_id_fk,offer_id_fk
1005,251001-02,1,OF04728
1006,251001-02,2,OF04728
1007,251001-02,3,OF04728
1008,251001-02,4,OF04728
1009,251001-02,5,OF04728



⏳ Applying manual override to trip '251001-02' in memory...
✅ Override applied in memory to column 'offer_id_fk'.

⏳ Re-running Golden Propagation in memory...
✅ Propagation complete in memory.

⏳ Writing patched data back to opus.db (Forced Save Protocol)...
   - Data written and connection disposed. Pausing for 5 seconds to allow sync...
✅ Write operation complete.

--- VERIFICATION (AFTER PATCH) ---
Performing final verification by re-reading data from disk...
Displaying corrected data for trip '251001-02':


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,251001-02,1,OF04733
1,251001-02,2,OF04733
2,251001-02,3,OF04733
3,251001-02,4,OF04733
4,251001-02,5,OF04733


✅ VERIFICATION SUCCESS: Changes have been successfully persisted to the opus.db file.

--- SURGICAL STRIKE COMPLETE ---


In [ ]:
# ==============================================================================
# CELL [NEW]: THE "OF04649/OF04565" SURGICAL STRIKE PATCH (CORRECTED)
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: This is a corrected manual override. It retargets trip '250930-04',
#          which was previously and incorrectly mapped, and now definitively
#          matches it to the Architect-certified correct offer: 'OF04565'.
# ==============================================================================

import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
import time

# ------------------------------------------------------------------------------
# 0. SETUP & CONSTANTS
# ------------------------------------------------------------------------------
print("--- Initiating CORRECTED Surgical Strike Patch: The 'OF04649/OF04565' Incident ---")
try:
    db_path = '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("✅ Database engine created successfully.")
except Exception as e:
    print(f"🔴 CRITICAL ERROR: Could not create database engine. Halting. Details: {e}")
    raise

# Define the new, definitive ground truth from the Architect's final audit
TARGET_TRIP_ID = '250930-04'
INCORRECT_OFFER_ID = 'OF04649' # The previously intended, but incorrect, target
CORRECT_OFFER_ID = 'OF04565'   # The new, final, correct target

# ------------------------------------------------------------------------------
# 1. READ & DISPLAY "BEFORE" STATE
# ------------------------------------------------------------------------------
print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"✅ Successfully read {len(trip_events_df)} records.")

    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    print(f"Displaying current data for trip '{TARGET_TRIP_ID}':")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])

except Exception as e:
    print(f"🔴 ERROR: Could not read from trip_events table. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 2. THE SURGICAL STRIKE: Manual Override in Memory
# ------------------------------------------------------------------------------
print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID
trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan
t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"✅ Override applied in memory to column 'offer_id_fk'.")

# ------------------------------------------------------------------------------
# 3. RE-RUN "GOLDEN PROPAGATION" in Memory
# ------------------------------------------------------------------------------
print("\n⏳ Re-running Golden Propagation in memory...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("✅ Propagation complete in memory.")

# ------------------------------------------------------------------------------
# 4. WRITE the corrected data back to the database (FORCED SAVE)
# ------------------------------------------------------------------------------
print("\n⏳ Writing patched data back to opus.db (Forced Save Protocol)...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written and connection disposed. Pausing for 5 seconds to allow sync...")
    time.sleep(5)
    print("✅ Write operation complete.")
except Exception as e:
    print(f"🔴 ERROR: Failed to write patched data to the database. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 5. FINAL VERIFICATION (READ-BACK & DISPLAY "AFTER" STATE)
# ------------------------------------------------------------------------------
print("\n--- VERIFICATION (AFTER PATCH) ---")
print("Performing final verification by re-reading data from disk...")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()

    print(f"Displaying corrected data for trip '{TARGET_TRIP_ID}':")
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])

    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("✅ VERIFICATION SUCCESS: Changes have been successfully persisted to the opus.db file.")
except Exception as e:
    print(f"🔴 VERIFICATION FAILED: Could not re-read or verify the data. The sync issue persists. Details: {e}")

print("\n--- SURGICAL STRIKE COMPLETE ---")

--- Initiating CORRECTED Surgical Strike Patch: The 'OF04649/OF04565' Incident ---
✅ Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
✅ Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---
Displaying current data for trip '250930-04':


,trip_id_legacy,event_types_id_fk,offer_id_fk
972,250930-04,1,OF04564
973,250930-04,2,OF04564



⏳ Applying manual override to trip '250930-04' in memory...
✅ Override applied in memory to column 'offer_id_fk'.

⏳ Re-running Golden Propagation in memory...
✅ Propagation complete in memory.

⏳ Writing patched data back to opus.db (Forced Save Protocol)...
   - Data written and connection disposed. Pausing for 5 seconds to allow sync...
✅ Write operation complete.

--- VERIFICATION (AFTER PATCH) ---
Performing final verification by re-reading data from disk...
Displaying corrected data for trip '250930-04':


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250930-04,1,OF04565
1,250930-04,2,OF04565


✅ VERIFICATION SUCCESS: Changes have been successfully persisted to the opus.db file.

--- SURGICAL STRIKE COMPLETE ---


In [ ]:
# ==============================================================================
# CELL [NEW]: THE "OF03904/OF04649" SURGICAL STRIKE PATCH
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: This is a one-time, manual override patch to correct a specific
#          "Stolen Identity" error where trip '250930-09' was incorrectly
#          matched to 'OF03904'. This script surgically remaps it to the
#          correct 'OF04649'.
# ==============================================================================

import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
import time

# ------------------------------------------------------------------------------
# 0. SETUP & CONSTANTS
# ------------------------------------------------------------------------------
print("--- Initiating Surgical Strike Patch: The 'OF03904/OF04649' Incident ---")
try:
    db_path = '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("✅ Database engine created successfully.")
except Exception as e:
    print(f"🔴 CRITICAL ERROR: Could not create database engine. Halting. Details: {e}")
    raise

# Define the ground truth based on the Architect's audit
TARGET_TRIP_ID = '250930-09'
INCORRECT_OFFER_ID = 'OF03904' # For verification
CORRECT_OFFER_ID = 'OF04649'

# ------------------------------------------------------------------------------
# 1. READ & DISPLAY "BEFORE" STATE
# ------------------------------------------------------------------------------
print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"✅ Successfully read {len(trip_events_df)} records.")

    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    print(f"Displaying corrupted data for trip '{TARGET_TRIP_ID}':")
    display(trip_events_df[trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])

except Exception as e:
    print(f"🔴 ERROR: Could not read from trip_events table. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 2. THE SURGICAL STRIKE: Manual Override in Memory
# ------------------------------------------------------------------------------
print(f"\n⏳ Applying manual override to trip '{TARGET_TRIP_ID}' in memory...")
target_mask = trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID
trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan
t1_mask = (trip_events_df['trip_id_legacy'] == TARGET_TRIP_ID) & (trip_events_df['event_types_id_fk'] == 2)
trip_events_df.loc[t1_mask, 'offer_id_fk'] = CORRECT_OFFER_ID
print(f"✅ Override applied in memory to column 'offer_id_fk'.")

# ------------------------------------------------------------------------------
# 3. RE-RUN "GOLDEN PROPAGATION" in Memory
# ------------------------------------------------------------------------------
print("\n⏳ Re-running Golden Propagation in memory...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("✅ Propagation complete in memory.")

# ------------------------------------------------------------------------------
# 4. WRITE the corrected data back to the database (FORCED SAVE)
# ------------------------------------------------------------------------------
print("\n⏳ Writing patched data back to opus.db (Forced Save Protocol)...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written and connection disposed. Pausing for 5 seconds to allow sync...")
    time.sleep(5)
    print("✅ Write operation complete.")
except Exception as e:
    print(f"🔴 ERROR: Failed to write patched data to the database. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 5. FINAL VERIFICATION (READ-BACK & DISPLAY "AFTER" STATE)
# ------------------------------------------------------------------------------
print("\n--- VERIFICATION (AFTER PATCH) ---")
print("Performing final verification by re-reading data from disk...")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy = '{TARGET_TRIP_ID}'", verify_engine)
    verify_engine.dispose()

    print(f"Displaying corrected data for trip '{TARGET_TRIP_ID}':")
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])

    assert (verify_df['offer_id_fk'] == CORRECT_OFFER_ID).all(), "Verification Failed: Mismatch after re-reading!"
    print("✅ VERIFICATION SUCCESS: Changes have been successfully persisted to the opus.db file.")
except Exception as e:
    print(f"🔴 VERIFICATION FAILED: Could not re-read or verify the data. The sync issue persists. Details: {e}")

print("\n--- SURGICAL STRIKE COMPLETE ---")

--- Initiating Surgical Strike Patch: The 'OF03904/OF04649' Incident ---
✅ Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
✅ Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---
Displaying corrupted data for trip '250930-09':


,trip_id_legacy,event_types_id_fk,offer_id_fk
993,250930-09,2,OF03903
994,250930-09,4,OF03903
995,250930-09,5,OF03903



⏳ Applying manual override to trip '250930-09' in memory...
✅ Override applied in memory to column 'offer_id_fk'.

⏳ Re-running Golden Propagation in memory...
✅ Propagation complete in memory.

⏳ Writing patched data back to opus.db (Forced Save Protocol)...
   - Data written and connection disposed. Pausing for 5 seconds to allow sync...
✅ Write operation complete.

--- VERIFICATION (AFTER PATCH) ---
Performing final verification by re-reading data from disk...
Displaying corrected data for trip '250930-09':


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250930-09,2,OF04649
1,250930-09,4,OF04649
2,250930-09,5,OF04649


✅ VERIFICATION SUCCESS: Changes have been successfully persisted to the opus.db file.

--- SURGICAL STRIKE COMPLETE ---


In [ ]:
# ==============================================================================
# CELL [FINAL PATCH]: THE "GOLDEN EDICT" BATCH OVERRIDE
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: This is the definitive, final patch for Operation Golden Link. It
#          takes a manually curated "Golden Edict" from the Architect and
#          executes a batch of surgical corrections to the trip_events table,
#          guaranteeing 100% data integrity for the known error cases.
# ==============================================================================

import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
import time

# ------------------------------------------------------------------------------
# 0. SETUP & THE "GOLDEN EDICT"
# ------------------------------------------------------------------------------
try:
    db_path = '/content/drive/MyDrive/_Pienza/Assets/Database/opus.db'
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file not found at path: {db_path}")
    db_engine = create_engine(f'sqlite:///{db_path}')
    print("✅ Database engine created successfully.")
except Exception as e:
    print(f"🔴 CRITICAL ERROR: Could not create database engine. Halting. Details: {e}")
    raise

# The Architect's "Golden Edict" - The definitive list of corrections
# Format: (TARGET_TRIP_ID, INCORRECT_OFFER_ID, CORRECT_OFFER_ID)
GOLDEN_EDICT = [
    ('250911-10', 'OF02170', 'OF02169'),
    ('250911-09', 'OF02160', 'OF02159'),
    ('250904-05', 'OF01667', 'OF01666'),
    ('250902-02', 'OF01290', 'OF01289'),
    ('250901-10', 'OF01280', 'OF01279')
]
target_trip_ids = [item[0] for item in GOLDEN_EDICT]

# ------------------------------------------------------------------------------
# 1. READ & DISPLAY "BEFORE" STATE
# ------------------------------------------------------------------------------
print("\n⏳ Reading current state of 'trip_events' table...")
try:
    trip_events_df = pd.read_sql('SELECT * FROM trip_events', db_engine)
    print(f"✅ Successfully read {len(trip_events_df)} records.")

    print("\n--- VERIFICATION (BEFORE PATCH) ---")
    print("Displaying corrupted data for target trips:")
    display(trip_events_df[trip_events_df['trip_id_legacy'].isin(target_trip_ids)][['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']])

except Exception as e:
    print(f"🔴 ERROR: Could not read from trip_events table. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 2. THE BATCH SURGICAL STRIKE: Manual Override in Memory
# ------------------------------------------------------------------------------
print(f"\n⏳ Applying {len(GOLDEN_EDICT)} manual overrides in memory...")
for trip_id, incorrect_id, correct_id in GOLDEN_EDICT:
    target_mask = trip_events_df['trip_id_legacy'] == trip_id
    trip_events_df.loc[target_mask, 'offer_id_fk'] = np.nan
    t1_mask = (trip_events_df['trip_id_legacy'] == trip_id) & (trip_events_df['event_types_id_fk'] == 2)
    trip_events_df.loc[t1_mask, 'offer_id_fk'] = correct_id
print("✅ All overrides applied in memory.")

# ------------------------------------------------------------------------------
# 3. RE-RUN "GOLDEN PROPAGATION" on the entire DataFrame
# ------------------------------------------------------------------------------
print("\n⏳ Re-running Golden Propagation to ensure consistency for all patched trips...")
trip_events_df = trip_events_df.sort_values(by=['trip_id_legacy', 'event_timestamp'])
trip_events_df['offer_id_fk'] = trip_events_df.groupby('trip_id_legacy')['offer_id_fk'].transform(lambda x: x.ffill().bfill())
print("✅ Propagation complete in memory.")

# ------------------------------------------------------------------------------
# 4. WRITE the corrected data back to the database (FORCED SAVE)
# ------------------------------------------------------------------------------
print("\n⏳ Writing final patched data back to opus.db...")
try:
    trip_events_df.to_sql('trip_events', db_engine, if_exists='replace', index=False)
    db_engine.dispose()
    print("   - Data written and connection disposed. Pausing for 5 seconds to allow sync...")
    time.sleep(5)
    print("✅ Write operation complete.")
except Exception as e:
    print(f"🔴 ERROR: Failed to write patched data to the database. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 5. FINAL VERIFICATION (READ-BACK & DISPLAY "AFTER" STATE)
# ------------------------------------------------------------------------------
print("\n--- VERIFICATION (AFTER PATCH) ---")
print("Performing final verification by re-reading data from disk...")
try:
    verify_engine = create_engine(f'sqlite:///{db_path}')
    verify_df = pd.read_sql_query(f"SELECT * FROM trip_events WHERE trip_id_legacy IN {tuple(target_trip_ids)}", verify_engine)
    verify_engine.dispose()

    print("Displaying corrected data for all target trips:")
    display(verify_df[['trip_id_legacy', 'event_types_id_fk', 'offer_id_fk']].sort_values(by='trip_id_legacy'))

    # Programmatic check for all corrections
    for trip_id, _, correct_id in GOLDEN_EDICT:
        assert (verify_df[verify_df['trip_id_legacy'] == trip_id]['offer_id_fk'] == correct_id).all(), f"Verification Failed for {trip_id}!"
    print("✅ VERIFICATION SUCCESS: All changes have been successfully persisted to the opus.db file.")

except Exception as e:
    print(f"🔴 VERIFICATION FAILED: Could not re-read or verify the data. Details: {e}")

print("\n--- BATCH SURGICAL STRIKE COMPLETE ---")

✅ Database engine created successfully.

⏳ Reading current state of 'trip_events' table...
✅ Successfully read 1031 records.

--- VERIFICATION (BEFORE PATCH) ---
Displaying corrupted data for target trips:


,trip_id_legacy,event_types_id_fk,offer_id_fk
252,250901-10,2,OF01279
253,250901-10,3,OF01279
254,250901-10,4,OF01279
255,250901-10,5,OF01279
261,250902-02,2,OF01289
262,250902-02,3,OF01289
263,250902-02,4,OF01289
264,250902-02,5,OF01289
346,250904-05,1,OF01666
347,250904-05,2,OF01666



⏳ Applying 5 manual overrides in memory...
✅ All overrides applied in memory.

⏳ Re-running Golden Propagation to ensure consistency for all patched trips...
✅ Propagation complete in memory.

⏳ Writing final patched data back to opus.db...
   - Data written and connection disposed. Pausing for 5 seconds to allow sync...
✅ Write operation complete.

--- VERIFICATION (AFTER PATCH) ---
Performing final verification by re-reading data from disk...
Displaying corrected data for all target trips:


,trip_id_legacy,event_types_id_fk,offer_id_fk
0,250901-10,2,OF01279
1,250901-10,3,OF01279
2,250901-10,4,OF01279
3,250901-10,5,OF01279
4,250902-02,2,OF01289
5,250902-02,3,OF01289
6,250902-02,4,OF01289
7,250902-02,5,OF01289
12,250904-05,5,OF01666
11,250904-05,4,OF01666


✅ VERIFICATION SUCCESS: All changes have been successfully persisted to the opus.db file.

--- BATCH SURGICAL STRIKE COMPLETE ---


In [71]:
# ==============================================================================
# CELL [NEW] (v1.3): THE "CONSOMASTER" BRIDGE (HARDENED PERSISTENCE)
# ==============================================================================
# Author v2: _Pienza, ex_machina
# Purpose: Connects trip_events to lifetime_trips via the Consomaster bridge.
#          Includes ID normalization, Forced Save Protocol, and Read-Back
#          Verification to guarantee data is persisted to disk.
# ==============================================================================

import pandas as pd
import numpy as np
import time
from sqlalchemy import create_engine

print("--- Initiating Consomaster Bridge Protocol (v1.3 - Hardened) ---")

# ------------------------------------------------------------------------------
# 1. SETUP & INGESTION
# ------------------------------------------------------------------------------
CONSO_GSHEET_NAME = 'ConsoMaster_with_MasterID_FINAL'
CONSO_WORKSHEET_NAME = 'Sheet1'

print(f"⏳ Ingesting Consomaster Bridge from: {CONSO_GSHEET_NAME}...")
try:
    conso_ws = gc.open(CONSO_GSHEET_NAME).worksheet(CONSO_WORKSHEET_NAME)
    conso_df = pd.DataFrame(conso_ws.get_all_records()).astype(str)

    bridge_df = conso_df[
        (conso_df['trip_id_legacy'] != '') &
        (conso_df['lifetime_trips_id'] != '')
    ][['trip_id_legacy', 'lifetime_trips_id']].copy()

    # NORMALIZE IDs (Strip 'LT' and leading zeros)
    bridge_df['lifetime_trips_id'] = bridge_df['lifetime_trips_id'].str.replace('LT', '').astype(float).astype(int).astype(str)

    print(f"✅ Successfully ingested bridge. Found {len(bridge_df)} valid link pairs.")

except Exception as e:
    print(f"🔴 ERROR: Failed to ingest/normalize Consomaster. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 2. FETCH SOURCE DATA
# ------------------------------------------------------------------------------
print("\n⏳ Fetching 'offer_id_fk' source data from 'trip_events'...")
try:
    source_links_df = pd.read_sql(
        "SELECT DISTINCT trip_id_legacy, offer_id_fk FROM trip_events WHERE offer_id_fk IS NOT NULL",
        db_engine
    )
    source_links_df['trip_id_legacy'] = source_links_df['trip_id_legacy'].astype(str)
    print(f"✅ Loaded {len(source_links_df)} source links from trip_events.")
except Exception as e:
    print(f"🔴 ERROR: Failed to read from trip_events. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 3. BUILD THE BRIDGE
# ------------------------------------------------------------------------------
print("\n⏳ Constructing the Golden Link map...")
merged_map = pd.merge(bridge_df, source_links_df, on='trip_id_legacy', how='inner')

lifetime_to_offer_map = pd.Series(
    merged_map.offer_id_fk.values,
    index=merged_map.lifetime_trips_id
).to_dict()

print(f"✅ Bridge constructed. Mapped {len(lifetime_to_offer_map)} lifetime trips to offers.")

# ------------------------------------------------------------------------------
# 4. UPDATE THE TARGET (WITH FORCED SAVE)
# ------------------------------------------------------------------------------
print("\n⏳ Updating 'lifetime_trips' table in opus.db...")
try:
    lifetime_trips_df = pd.read_sql("SELECT * FROM lifetime_trips", db_engine)

    # Apply Map
    lifetime_trips_df['offer_id_fk'] = lifetime_trips_df['lifetime_trips_id'].astype(str).map(lifetime_to_offer_map)

    updated_count = lifetime_trips_df['offer_id_fk'].notna().sum()
    print(f"   - Rows updated with Offer IDs: {updated_count}")

    if updated_count == 0:
        print("🔴 WARNING: Update count is 0. Check IDs.")
    else:
        # WRITE TO DB
        lifetime_trips_df.to_sql('lifetime_trips', db_engine, if_exists='replace', index=False)

        # FORCED SAVE PROTOCOL
        db_engine.dispose() # Close connection to flush
        print("   - Data written. Connection closed.")
        print("   - Pausing 5 seconds for Drive Sync...")
        time.sleep(5)
        print("✅ SUCCESS: 'lifetime_trips' table updated and saved.")

except Exception as e:
    print(f"🔴 ERROR: Failed to update lifetime_trips. Details: {e}")
    raise

# ------------------------------------------------------------------------------
# 5. READ-BACK VERIFICATION
# ------------------------------------------------------------------------------
print("\n⏳ Performing Read-Back Verification...")
try:
    # Re-open connection to verify disk state
    verify_engine = create_engine(f'sqlite:///{db_path}')

    # Check for non-null keys
    verify_count = pd.read_sql("SELECT COUNT(*) FROM lifetime_trips WHERE offer_id_fk IS NOT NULL", verify_engine).iloc[0,0]
    verify_engine.dispose()

    print(f"🔎 Verified on Disk: {verify_count} rows in 'lifetime_trips' now have an offer_id_fk.")

    if verify_count > 0:
        print("✅ VERIFICATION PASSED: The database is definitively updated.")
    else:
        print("🔴 VERIFICATION FAILED: The database file on disk still shows 0 links.")

except Exception as e:
    print(f"🔴 ERROR during verification: {e}")

print("\n--- BRIDGE PROTOCOL COMPLETE ---")

--- Initiating Consomaster Bridge Protocol (v1.3 - Hardened) ---
⏳ Ingesting Consomaster Bridge from: ConsoMaster_with_MasterID_FINAL...
✅ Successfully ingested bridge. Found 254 valid link pairs.

⏳ Fetching 'offer_id_fk' source data from 'trip_events'...
✅ Loaded 259 source links from trip_events.

⏳ Constructing the Golden Link map...
✅ Bridge constructed. Mapped 254 lifetime trips to offers.

⏳ Updating 'lifetime_trips' table in opus.db...
   - Rows updated with Offer IDs: 254
   - Data written. Connection closed.
   - Pausing 5 seconds for Drive Sync...
✅ SUCCESS: 'lifetime_trips' table updated and saved.

⏳ Performing Read-Back Verification...
🔎 Verified on Disk: 254 rows in 'lifetime_trips' now have an offer_id_fk.
✅ VERIFICATION PASSED: The database is definitively updated.

--- BRIDGE PROTOCOL COMPLETE ---
